# Valparaíso 2017 — consolidación auditable de celdas históricas

Este notebook fue **derivado del notebook histórico original** `Extraer_Vs30_USGS_Validacion_2024.ipynb` (Drive ID `1sePPNPQnz2g3rC1phL8uKl5m951Bn8Mq`) sin modificar el código ni los outputs de las celdas seleccionadas.

SHA-256 del notebook histórico fuente: `397967c5c283caf8142fb99c063c9ede18a09e8ea306195eabc76bd334e91fd7`.

Se preservan las celdas históricas correspondientes a elegibilidad, falla finita, descarga/auditoría CSN, Rrup/Vs30, sincronización E/N, RotD50 T=1 s, auditoría CESMD/escala, congelamiento formal y ejecución única de V5.1.

**Naturaleza del archivo:** consolidación para reproducibilidad; no es un notebook histórico independiente. La fuente histórica original permanece identificada por ID y hash.


In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — AUDITORÍA PREVIA DE ELEGIBILIDAD
#
# Evento:
# USGS us10008kce
# 24 abril 2017
#
# OBJETIVO:
# 1. confirmar independencia temporal;
# 2. archivar metadata USGS;
# 3. inspeccionar rupture.json de USGS;
# 4. descargar fuente científica de ruptura finita;
# 5. acceder al evento oficial CSN;
# 6. descubrir tablas y enlaces de acelerogramas.
#
# NO calcula Rrup.
# NO calcula RotD50.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, timezone

# ----------------------------------------------------------------------
# 1. CONFIGURACIÓN
# ----------------------------------------------------------------------

EVENT_ID = "us10008kce"
EVENT_YEAR = 2017

# Identificador del evento en la base de movimientos fuertes CSN
CSN_EVENT_HASH = "0565d07e6e3c0e823e8e32c18687eba8"

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

os.makedirs(
    CARPETA,
    exist_ok=True
)

print("=" * 80)
print("AUDITORÍA PREVIA — VALPARAÍSO 2017")
print("=" * 80)


# ----------------------------------------------------------------------
# 2. USGS DETAIL
# ----------------------------------------------------------------------

URL_DETAIL = (
    "https://earthquake.usgs.gov/"
    f"earthquakes/feed/v1.0/detail/{EVENT_ID}.geojson"
)

r = requests.get(
    URL_DETAIL,
    timeout=60
)

r.raise_for_status()

detail = r.json()

with open(
    os.path.join(
        CARPETA,
        f"{EVENT_ID}_USGS_detail.geojson"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        detail,
        f,
        indent=2,
        ensure_ascii=False
    )

prop = detail["properties"]

lon, lat, depth = (
    detail["geometry"]["coordinates"]
)

fecha = datetime.fromtimestamp(
    prop["time"] / 1000,
    tz=timezone.utc
)

print("\n" + "=" * 80)
print("CATÁLOGO USGS")
print("=" * 80)

print("ID          :", EVENT_ID)
print("Fecha UTC   :", fecha)
print("Magnitud    :", prop.get("mag"))
print("Tipo mag.   :", prop.get("magType"))
print("Latitud     :", lat)
print("Longitud    :", lon)
print("Profundidad :", depth, "km")
print("Lugar       :", prop.get("place"))
print("Status      :", prop.get("status"))


# ----------------------------------------------------------------------
# 3. INDEPENDENCIA TEMPORAL
# ----------------------------------------------------------------------

RUTA_ORIGINAL = os.path.join(
    BASE,
    "extracto_sudamerica_interfaz_sin_filtro_zhyp.xlsx"
)

assert os.path.exists(RUTA_ORIGINAL)

df_year = pd.read_excel(
    RUTA_ORIGINAL,
    usecols=lambda c:
        c in {
            "YEAR",
            "NGAsubEQID",
            "Earthquake_Name"
        }
)

años = pd.to_numeric(
    df_year["YEAR"],
    errors="coerce"
)

MAX_YEAR = int(años.max())

TEMPORAL_OK = (
    MAX_YEAR < EVENT_YEAR
)

print("\n" + "=" * 80)
print("INDEPENDENCIA TEMPORAL")
print("=" * 80)

print("Último año desarrollo:", MAX_YEAR)
print("Evento candidato     :", EVENT_YEAR)

print(
    "PASS"
    if TEMPORAL_OK
    else "FAIL"
)


# ----------------------------------------------------------------------
# 4. AUSENCIA EN DATASET ANALÍTICO
# ----------------------------------------------------------------------

RUTA_ANALITICO = os.path.join(
    BASE,
    "Resultados_V5",
    "01_dataset_analitico_v5_1987.csv"
)

analitico = pd.read_csv(
    RUTA_ANALITICO
)

mask_id = (
    analitico
    .astype(str)
    .apply(
        lambda c:
        c.str.contains(
            EVENT_ID,
            case=False,
            na=False
        )
    )
    .any(axis=1)
)

print(
    "\nCoincidencias explícitas "
    "del ID USGS en V5.1:",
    int(mask_id.sum())
)


# ----------------------------------------------------------------------
# 5. PRODUCTOS USGS
# ----------------------------------------------------------------------

products = prop.get(
    "products",
    {}
)

print("\n" + "=" * 80)
print("PRODUCTOS USGS")
print("=" * 80)

for key in sorted(products):

    print(
        f"{key:30s}: "
        f"{len(products[key])}"
    )


# ----------------------------------------------------------------------
# 6. INSPECCIONAR RUPTURE.JSON USGS
# ----------------------------------------------------------------------

if "shakemap" in products:

    sm = max(
        products["shakemap"],
        key=lambda x: (
            x.get("preferredWeight", 0),
            x.get("updateTime", 0)
        )
    )

    contents = sm.get(
        "contents",
        {}
    )

    rupture_keys = [
        k for k in contents
        if "rupture" in k.lower()
    ]

    print("\n" + "=" * 80)
    print("RUPTURE.JSON USGS")
    print("=" * 80)

    for key in rupture_keys:

        url = contents[key].get(
            "url"
        )

        print("\n", key)
        print(url)

        rr = requests.get(
            url,
            timeout=60
        )

        rr.raise_for_status()

        ruptura = rr.json()

        ruta_rup = os.path.join(
            CARPETA,
            "USGS_rupture.json"
        )

        with open(
            ruta_rup,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                ruptura,
                f,
                indent=2
            )

        tipos = []

        def buscar_tipos(obj):

            if isinstance(obj, dict):

                if obj.get("type") in {
                    "Point",
                    "Polygon",
                    "MultiPolygon",
                    "LineString",
                    "MultiLineString"
                }:

                    tipos.append(
                        obj["type"]
                    )

                for v in obj.values():
                    buscar_tipos(v)

            elif isinstance(obj, list):

                for v in obj:
                    buscar_tipos(v)

        buscar_tipos(
            ruptura
        )

        print(
            "Geometrías USGS:",
            sorted(set(tipos))
        )


# ----------------------------------------------------------------------
# 7. DESCARGAR ARTÍCULO CIENTÍFICO DE RUPTURA
# ----------------------------------------------------------------------

URL_PAPER = (
    "https://www.geologie.ens.fr/"
    "~madariag/Papers/"
    "Ruiz_etal_GRL_Valpo_2017.pdf"
)

RUTA_PAPER = os.path.join(
    CARPETA,
    "Ruiz_et_al_2017_Valparaiso.pdf"
)

rp = requests.get(
    URL_PAPER,
    timeout=90
)

rp.raise_for_status()

with open(
    RUTA_PAPER,
    "wb"
) as f:

    f.write(
        rp.content
    )

print("\n" + "=" * 80)
print("MODELO PUBLICADO DE RUPTURA")
print("=" * 80)

print(
    "PDF descargado:",
    RUTA_PAPER
)

print(
    "Tamaño:",
    os.path.getsize(
        RUTA_PAPER
    ),
    "bytes"
)


# ----------------------------------------------------------------------
# 8. DESCARGAR INFORME OFICIAL CSN
# ----------------------------------------------------------------------

URL_REPORTE_CSN = (
    "https://www.csn.uchile.cl/"
    "wp-content/uploads/2017/05/"
    "Reporte-Sismicidad-Abril-2017-Final.pdf"
)

RUTA_REPORTE = os.path.join(
    CARPETA,
    "CSN_Reporte_Valparaiso2017.pdf"
)

rcsn = requests.get(
    URL_REPORTE_CSN,
    timeout=90
)

rcsn.raise_for_status()

with open(
    RUTA_REPORTE,
    "wb"
) as f:

    f.write(
        rcsn.content
    )

print(
    "\nReporte CSN:",
    RUTA_REPORTE
)

print(
    "Tamaño:",
    os.path.getsize(
        RUTA_REPORTE
    ),
    "bytes"
)


# ----------------------------------------------------------------------
# 9. EVENTO EN BASE DE MOVIMIENTOS FUERTES CSN
# ----------------------------------------------------------------------

URL_CSN_EVENT = (
    "https://evtdb.csn.uchile.cl/"
    f"event/{CSN_EVENT_HASH}"
)

print("\n" + "=" * 80)
print("BASE DE MOVIMIENTOS FUERTES CSN")
print("=" * 80)

print(
    "URL:",
    URL_CSN_EVENT
)

headers = {
    "User-Agent":
        "Mozilla/5.0"
}

res = requests.get(
    URL_CSN_EVENT,
    headers=headers,
    timeout=90
)

print(
    "HTTP:",
    res.status_code
)

print(
    "Tamaño HTML:",
    len(res.text)
)


RUTA_HTML = os.path.join(
    CARPETA,
    "Valparaiso2017_CSN_event.html"
)

with open(
    RUTA_HTML,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        res.text
    )


# ----------------------------------------------------------------------
# 10. EXTRAER TABLAS HTML
# ----------------------------------------------------------------------

tablas = []

if res.status_code == 200:

    try:

        tablas = pd.read_html(
            res.text
        )

    except Exception as e:

        print(
            "No se detectaron tablas HTML:",
            e
        )


print(
    "\nTablas detectadas:",
    len(tablas)
)


for i, tabla in enumerate(
    tablas
):

    print(
        f"\nTabla {i}: "
        f"{tabla.shape}"
    )

    display(
        tabla.head(20)
    )

    tabla.to_csv(
        os.path.join(
            CARPETA,
            f"CSN_tabla_{i}.csv"
        ),
        index=False
    )


# ----------------------------------------------------------------------
# 11. EXTRAER TODOS LOS ENLACES
# ----------------------------------------------------------------------

soup = BeautifulSoup(
    res.text,
    "html.parser"
)

links = []

for a in soup.find_all(
    "a",
    href=True
):

    texto = (
        a.get_text(
            " ",
            strip=True
        )
    )

    href = a["href"]

    links.append(
        {
            "texto": texto,
            "href": href
        }
    )


df_links = pd.DataFrame(
    links
).drop_duplicates()


RUTA_LINKS = os.path.join(
    CARPETA,
    "Valparaiso2017_CSN_links.csv"
)

df_links.to_csv(
    RUTA_LINKS,
    index=False
)


print(
    "\nNúmero de enlaces:",
    len(df_links)
)


# Mostrar enlaces potencialmente útiles
if len(df_links):

    mask = (
        df_links["texto"]
        .str.contains(
            "VA|MT|PE|BO|download|descarga|registro",
            case=False,
            na=False
        )
        |
        df_links["href"]
        .str.contains(
            "download|record|file|wave|txt|zip",
            case=False,
            na=False
        )
    )

    print(
        "\nEnlaces potencialmente útiles:"
    )

    display(
        df_links[
            mask
        ].head(100)
    )


# ----------------------------------------------------------------------
# 12. BUSCAR CÓDIGOS DE ESTACIÓN EN EL HTML
# ----------------------------------------------------------------------

texto_html = soup.get_text(
    " ",
    strip=True
)


# Códigos conocidos típicos de red chilena
patron = r"\b(?:VA|MT|BO|PE|FA|ST)[A-Z0-9]{1,4}\b"

codigos = sorted(
    set(
        re.findall(
            patron,
            texto_html,
            flags=re.I
        )
    )
)


print("\n" + "=" * 80)
print("CÓDIGOS DE ESTACIÓN DETECTADOS")
print("=" * 80)

print(
    "N =",
    len(codigos)
)

print(
    codigos
)


# ----------------------------------------------------------------------
# 13. RESUMEN
# ----------------------------------------------------------------------

resumen = {
    "event_id":
        EVENT_ID,

    "fecha":
        fecha.isoformat(),

    "magnitud_usgs":
        prop.get("mag"),

    "tipo_magnitud":
        prop.get("magType"),

    "ultimo_año_desarrollo":
        MAX_YEAR,

    "independencia_temporal":
        bool(TEMPORAL_OK),

    "coincidencias_dataset_V5_1":
        int(mask_id.sum()),

    "csn_http":
        int(res.status_code),

    "csn_estaciones_detectadas":
        len(codigos),

    "paper_ruptura_descargado":
        os.path.exists(
            RUTA_PAPER
        )
}


RUTA_RESUMEN = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_previa.json"
)


with open(
    RUTA_RESUMEN,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        resumen,
        f,
        indent=2,
        ensure_ascii=False
    )


print("\n" + "=" * 80)
print("DECISIÓN AÚN NO TOMADA")
print("=" * 80)

print(
    "Todavía debemos confirmar:"
)

print(
    "1. >=10 estaciones con dos "
    "componentes horizontales utilizables."
)

print(
    "2. geometría tridimensional suficiente "
    "del modelo Ruiz et al. (2017) "
    "para calcular Rrup."
)

print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — AUDITORÍA DEL PRODUCTO USGS FINITE-FAULT
#
# Evento: us10008kce
#
# OBJETIVO:
# 1. recuperar el producto finite-fault oficial USGS;
# 2. listar todos sus archivos;
# 3. descargar FFM / FSP / PARAM / GeoJSON disponibles;
# 4. auditar si contiene geometría 3D utilizable para Rrup;
#
# NO calcula Rrup.
# NO calcula Sa.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import json
import requests
import numpy as np
import pandas as pd

# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

EVENT_ID = "us10008kce"

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

os.makedirs(
    CARPETA,
    exist_ok=True
)

# ======================================================================
# 2. RECUPERAR DETAIL USGS
# ======================================================================

URL_DETAIL = (
    "https://earthquake.usgs.gov/"
    f"earthquakes/feed/v1.0/detail/{EVENT_ID}.geojson"
)

r = requests.get(
    URL_DETAIL,
    timeout=60
)

r.raise_for_status()

detail = r.json()

products = detail[
    "properties"
].get(
    "products",
    {}
)

assert "finite-fault" in products, (
    "No aparece producto finite-fault."
)

print("=" * 80)
print("PRODUCTO FINITE-FAULT — VALPARAÍSO 2017")
print("=" * 80)

print(
    "Número de productos finite-fault:",
    len(products["finite-fault"])
)

# ======================================================================
# 3. SELECCIONAR PRODUCTO PREFERIDO
# ======================================================================

ff_products = products[
    "finite-fault"
]

ff_pref = max(
    ff_products,
    key=lambda x: (
        x.get(
            "preferredWeight",
            0
        ),
        x.get(
            "updateTime",
            0
        )
    )
)

print("\nProducto seleccionado:")
print("Source          :", ff_pref.get("source"))
print("PreferredWeight :", ff_pref.get("preferredWeight"))
print("UpdateTime      :", ff_pref.get("updateTime"))
print("Status          :", ff_pref.get("status"))
print("Code            :", ff_pref.get("code"))

# ======================================================================
# 4. PROPIEDADES DEL MODELO
# ======================================================================

print("\n" + "=" * 80)
print("PROPIEDADES FINITE-FAULT")
print("=" * 80)

ff_prop = ff_pref.get(
    "properties",
    {}
)

for k in sorted(
    ff_prop.keys()
):
    print(
        f"{k}: {ff_prop[k]}"
    )

# ======================================================================
# 5. LISTAR ARCHIVOS DEL PRODUCTO
# ======================================================================

contents = ff_pref.get(
    "contents",
    {}
)

print("\n" + "=" * 80)
print("ARCHIVOS DISPONIBLES")
print("=" * 80)

print(
    "Número total:",
    len(contents)
)

tabla_archivos = []

for key, info in contents.items():

    tabla_archivos.append(
        {
            "key": key,
            "url": info.get("url"),
            "contentType": info.get("contentType"),
            "length": info.get("length")
        }
    )

df_files = pd.DataFrame(
    tabla_archivos
)

display(
    df_files
)

df_files.to_csv(
    os.path.join(
        CARPETA,
        "Valparaiso2017_USGS_finite_fault_archivos.csv"
    ),
    index=False
)

# ======================================================================
# 6. IDENTIFICAR ARCHIVOS IMPORTANTES
# ======================================================================

claves_interes = []

for key in contents.keys():

    kl = key.lower()

    if any(
        palabra in kl
        for palabra in [
            "ffm",
            "fsp",
            "param",
            "geojson",
            "fault",
            "rupture"
        ]
    ):

        claves_interes.append(
            key
        )

print("\n" + "=" * 80)
print("ARCHIVOS DE INTERÉS")
print("=" * 80)

for key in claves_interes:

    print(
        "\n",
        key
    )

    print(
        contents[key].get(
            "url"
        )
    )

# ======================================================================
# 7. DESCARGAR ARCHIVOS DE INTERÉS
# ======================================================================

descargados = []

for i, key in enumerate(
    claves_interes
):

    info = contents[key]

    url = info.get(
        "url"
    )

    if not url:
        continue

    rr = requests.get(
        url,
        timeout=90
    )

    rr.raise_for_status()

    # nombre seguro
    nombre = os.path.basename(
        key
    )

    if not nombre:
        nombre = (
            f"finite_fault_{i}"
        )

    ruta = os.path.join(
        CARPETA,
        nombre
    )

    with open(
        ruta,
        "wb"
    ) as f:

        f.write(
            rr.content
        )

    descargados.append(
        ruta
    )

    print(
        "Descargado:",
        ruta,
        "|",
        len(rr.content),
        "bytes"
    )

# ======================================================================
# 8. BUSCAR GEOJSON ENTRE LOS DESCARGADOS
# ======================================================================

geojson_files = []

for ruta in descargados:

    if ruta.lower().endswith(
        (
            ".json",
            ".geojson"
        )
    ):

        try:

            with open(
                ruta,
                "r",
                encoding="utf-8"
            ) as f:

                obj = json.load(
                    f
                )

            geojson_files.append(
                (
                    ruta,
                    obj
                )
            )

        except Exception:

            pass

print("\nGeoJSON/JSON válidos encontrados:")
print(
    len(geojson_files)
)

# ======================================================================
# 9. FUNCIONES DE AUDITORÍA GEOMÉTRICA
# ======================================================================

def buscar_tipos(obj):

    tipos = []

    if isinstance(
        obj,
        dict
    ):

        tipo = obj.get(
            "type"
        )

        if tipo in {
            "Point",
            "MultiPoint",
            "LineString",
            "MultiLineString",
            "Polygon",
            "MultiPolygon"
        }:

            tipos.append(
                tipo
            )

        for v in obj.values():

            tipos.extend(
                buscar_tipos(v)
            )

    elif isinstance(
        obj,
        list
    ):

        for v in obj:

            tipos.extend(
                buscar_tipos(v)
            )

    return tipos


def buscar_xyz(obj):

    puntos = []

    if isinstance(
        obj,
        list
    ):

        if (
            len(obj) >= 3
            and
            all(
                isinstance(
                    x,
                    (int, float)
                )
                for x in obj[:3]
            )
        ):

            lon, lat, z = obj[:3]

            if (
                -180 <= lon <= 180
                and
                -90 <= lat <= 90
            ):

                puntos.append(
                    [
                        lon,
                        lat,
                        z
                    ]
                )

        else:

            for v in obj:

                puntos.extend(
                    buscar_xyz(v)
                )

    elif isinstance(
        obj,
        dict
    ):

        for v in obj.values():

            puntos.extend(
                buscar_xyz(v)
            )

    return puntos

# ======================================================================
# 10. AUDITAR GEOJSON
# ======================================================================

resultado_geo = []

SUPERFICIE_FINITA = False

for ruta, obj in geojson_files:

    tipos = sorted(
        set(
            buscar_tipos(
                obj
            )
        )
    )

    xyz = buscar_xyz(
        obj
    )

    es_superficie = any(
        t in {
            "Polygon",
            "MultiPolygon"
        }
        for t in tipos
    )

    if es_superficie:

        SUPERFICIE_FINITA = True

    print("\n" + "-" * 80)

    print(
        "Archivo:",
        os.path.basename(
            ruta
        )
    )

    print(
        "Geometrías:",
        tipos
    )

    print(
        "N.º coordenadas XYZ:",
        len(xyz)
    )

    if len(xyz):

        a = np.asarray(
            xyz,
            dtype=float
        )

        print(
            "Lon:",
            a[:, 0].min(),
            "a",
            a[:, 0].max()
        )

        print(
            "Lat:",
            a[:, 1].min(),
            "a",
            a[:, 1].max()
        )

        print(
            "Z:",
            a[:, 2].min(),
            "a",
            a[:, 2].max()
        )

    resultado_geo.append(
        {
            "archivo":
                os.path.basename(
                    ruta
                ),

            "geometrias":
                ",".join(
                    tipos
                ),

            "n_xyz":
                len(xyz),

            "superficie_finita":
                es_superficie
        }
    )

# ======================================================================
# 11. BUSCAR FSP / PARAM AUNQUE NO SEAN JSON
# ======================================================================

print("\n" + "=" * 80)
print("ARCHIVOS TEXTUALES FSP / PARAM")
print("=" * 80)

for ruta in descargados:

    nombre = os.path.basename(
        ruta
    ).lower()

    if (
        "fsp" in nombre
        or
        "param" in nombre
    ):

        print(
            "\nArchivo:",
            ruta
        )

        try:

            with open(
                ruta,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                lineas = [
                    next(f)
                    for _ in range(20)
                ]

            print(
                "".join(lineas)
            )

        except StopIteration:

            pass

        except Exception as e:

            print(
                "No pudo leerse:",
                e
            )

# ======================================================================
# 12. DECISIÓN DEL GATE
# ======================================================================

print("\n" + "=" * 80)
print("GATE FINITE-FAULT PARA Rrup")
print("=" * 80)

if SUPERFICIE_FINITA:

    print(
        "✅ PASS"
    )

    print(
        "El producto USGS finite-fault "
        "contiene Polygon/MultiPolygon."
    )

    print(
        "Valparaíso puede continuar "
        "a la auditoría de Rrup."
    )

else:

    print(
        "⚠ No apareció Polygon/MultiPolygon "
        "en los JSON descargados."
    )

    print(
        "NO descartamos todavía el evento:"
    )

    print(
        "FSP/PARAM pueden contener "
        "la malla de subfallas necesaria."
    )

# ======================================================================
# 13. GUARDAR RESULTADO
# ======================================================================

if len(
    resultado_geo
):

    pd.DataFrame(
        resultado_geo
    ).to_csv(
        os.path.join(
            CARPETA,
            "Valparaiso2017_auditoria_geometria_finite_fault.csv"
        ),
        index=False
    )

resumen = {
    "event_id":
        EVENT_ID,

    "finite_fault_present":
        True,

    "n_archivos_producto":
        len(contents),

    "n_archivos_interes":
        len(claves_interes),

    "superficie_polygon_detectada":
        bool(
            SUPERFICIE_FINITA
        )
}

with open(
    os.path.join(
        CARPETA,
        "Valparaiso2017_gate_finite_fault.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        resumen,
        f,
        indent=2
    )

print("\n" + "=" * 80)
print("FIN")
print("=" * 80)

print(
    "V5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — AUDITORÍA Y DESCARGA DE REGISTROS CSN
#
# Evento objetivo:
# 24/04/2017 ~21:38 UTC
# Mw predictor congelada = 6.9
#
# OBJETIVOS:
# 1. localizar automáticamente el evento en evtdb.csn.uchile.cl;
# 2. obtener todas las estaciones;
# 3. recuperar latitud/longitud oficiales;
# 4. descargar registros de texto plano disponibles;
# 5. verificar componentes E + N;
# 6. congelar inventario candidato.
#
# IMPORTANTE:
# - NO calcula Sa.
# - NO calcula Rrup.
# - NO extrae todavía Vs30 principal.
# - NO ejecuta V5.1.
#
# Vs30 que aparezca en CSN será SOLO auditoría secundaria.
# El predictor principal seguirá siendo global_vs30.grd + nearest.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import io
import json
import time
import zipfile
import requests
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from datetime import datetime

# ======================================================================
# 1. CONFIGURACIÓN CONGELADA
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

CARPETA_REG = os.path.join(
    CARPETA,
    "Registros_CSN"
)

os.makedirs(
    CARPETA_REG,
    exist_ok=True
)

BASE_CSN = "https://evtdb.csn.uchile.cl"

TARGET = pd.Timestamp(
    "2017-04-24 21:38:30"
)

MW_CONGELADA = 6.9

session = requests.Session()

session.headers.update(
    {
        "User-Agent":
            "Mozilla/5.0 "
            "(Tesis academica; auditoria reproducible)"
    }
)

print("=" * 80)
print("VALPARAÍSO 2017 — AUDITORÍA DE REGISTROS CSN")
print("=" * 80)

print("Fecha objetivo :", TARGET)
print("Mw congelada   :", MW_CONGELADA)
print("Directorio     :", CARPETA_REG)


# ======================================================================
# 2. FUNCIONES PARA CATÁLOGO CSN
# ======================================================================

def descargar_html(url):

    r = session.get(
        url,
        timeout=60
    )

    r.raise_for_status()

    return r.text


def eventos_de_pagina(page):

    url = (
        BASE_CSN
        if page == 1
        else f"{BASE_CSN}/?page={page}"
    )

    html = descargar_html(
        url
    )

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    eventos = []

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = a.get("href", "")

        if "/event/" not in href:
            continue

        texto_fecha = a.get_text(
            " ",
            strip=True
        )

        try:

            fecha = pd.to_datetime(
                texto_fecha
            )

        except Exception:

            continue

        tr = a.find_parent(
            "tr"
        )

        celdas = []

        if tr is not None:

            celdas = [
                td.get_text(
                    " ",
                    strip=True
                )
                for td in tr.find_all(
                    "td"
                )
            ]

        eventos.append(
            {
                "fecha": fecha,
                "href": href,
                "url": urljoin(
                    BASE_CSN,
                    href
                ),
                "celdas": celdas
            }
        )

    return eventos, html


# ======================================================================
# 3. NÚMERO TOTAL DE PÁGINAS
# ======================================================================

html_inicio = descargar_html(
    BASE_CSN
)

m = re.search(
    r"Página\s+\d+\s+de\s+(\d+)",
    html_inicio,
    flags=re.I
)

if not m:

    raise RuntimeError(
        "No se pudo determinar "
        "el número de páginas del catálogo CSN."
    )

TOTAL_PAGES = int(
    m.group(1)
)

print(
    "\nPáginas catálogo CSN:",
    TOTAL_PAGES
)


# ======================================================================
# 4. BÚSQUEDA BINARIA DEL 24/04/2017
# ======================================================================

lo = 1
hi = TOTAL_PAGES

pagina_objetivo = None

while lo <= hi:

    mid = (
        lo + hi
    ) // 2

    eventos, _ = eventos_de_pagina(
        mid
    )

    if not eventos:

        raise RuntimeError(
            f"No se obtuvieron eventos "
            f"de la página {mid}."
        )

    fechas = [
        x["fecha"]
        for x in eventos
    ]

    fecha_mas_nueva = max(
        fechas
    )

    fecha_mas_antigua = min(
        fechas
    )

    print(
        f"Página {mid:3d}: "
        f"{fecha_mas_nueva} -> "
        f"{fecha_mas_antigua}"
    )

    # Catálogo ordenado de más nuevo a más antiguo
    if TARGET > fecha_mas_nueva:

        hi = mid - 1

    elif TARGET < fecha_mas_antigua:

        lo = mid + 1

    else:

        pagina_objetivo = mid
        break


if pagina_objetivo is None:

    raise RuntimeError(
        "No se localizó automáticamente "
        "la fecha objetivo en el catálogo."
    )


print(
    "\nPágina candidata:",
    pagina_objetivo
)


# ======================================================================
# 5. BUSCAR EVENTO EN PÁGINAS VECINAS
# ======================================================================

candidatos = []

for page in range(
    max(
        1,
        pagina_objetivo - 2
    ),
    min(
        TOTAL_PAGES,
        pagina_objetivo + 2
    ) + 1
):

    eventos, _ = eventos_de_pagina(
        page
    )

    for ev in eventos:

        # mismo día
        if (
            ev["fecha"].date()
            ==
            TARGET.date()
        ):

            diferencia_s = abs(
                (
                    ev["fecha"]
                    - TARGET
                ).total_seconds()
            )

            ev["pagina"] = page
            ev["diferencia_s"] = diferencia_s

            candidatos.append(
                ev
            )


if len(candidatos) == 0:

    raise RuntimeError(
        "No se encontraron eventos "
        "del 24/04/2017."
    )


candidatos = sorted(
    candidatos,
    key=lambda x:
        x["diferencia_s"]
)


print("\nEventos CSN del día objetivo:")

for x in candidatos:

    print(
        x["fecha"],
        "| diferencia:",
        x["diferencia_s"],
        "s |",
        x["url"],
        "|",
        x["celdas"]
    )


# ======================================================================
# 6. SELECCIONAR EL MÁS PRÓXIMO A 21:38:30 UTC
# ======================================================================

evento = candidatos[0]

# Guardrail: debe estar dentro de 2 minutos
if evento["diferencia_s"] > 120:

    raise RuntimeError(
        "El evento CSN más cercano "
        "está a más de 120 s del objetivo."
    )


URL_EVENTO = evento[
    "url"
]

print("\n" + "=" * 80)
print("EVENTO CSN SELECCIONADO")
print("=" * 80)

print("Fecha :", evento["fecha"])
print("URL   :", URL_EVENTO)
print("Fila  :", evento["celdas"])


# ======================================================================
# 7. DESCARGAR PÁGINA DEL EVENTO
# ======================================================================

html_evento = descargar_html(
    URL_EVENTO
)

RUTA_HTML = os.path.join(
    CARPETA,
    "Valparaiso2017_CSN_event.html"
)

with open(
    RUTA_HTML,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        html_evento
    )


soup = BeautifulSoup(
    html_evento,
    "html.parser"
)


texto_evento = soup.get_text(
    " ",
    strip=True
)

print("\nCabecera de evento:")

for patron in [
    r"Evento del\s+[^L]+",
    r"Latitud:\s*-?\d+(?:\.\d+)?",
    r"Longitud:\s*-?\d+(?:\.\d+)?",
    r"Profundidad:\s*\d+(?:\.\d+)?",
    r"Magnitud:\s*\d+(?:\.\d+)?"
]:

    m = re.search(
        patron,
        texto_evento,
        re.I
    )

    if m:

        print(
            m.group(0)
        )


# ======================================================================
# 8. EXTRAER FILAS DE ESTACIONES
# ======================================================================

filas = []

for tr in soup.find_all(
    "tr"
):

    station_anchor = None

    for a in tr.find_all(
        "a",
        href=True
    ):

        if "/station/" in a["href"]:

            station_anchor = a
            break

    if station_anchor is None:
        continue


    code = station_anchor.get_text(
        " ",
        strip=True
    ).upper()


    tds = tr.find_all(
        "td"
    )


    # Todos los enlaces de la fila
    todos_links = []

    for a in tr.find_all(
        "a",
        href=True
    ):

        todos_links.append(
            {
                "texto":
                    a.get_text(
                        " ",
                        strip=True
                    ),

                "url":
                    urljoin(
                        BASE_CSN,
                        a["href"]
                    )
            }
        )


    # Tercera columna = datos en texto plano
    links_texto = []

    if len(tds) >= 3:

        for a in tds[2].find_all(
            "a",
            href=True
        ):

            links_texto.append(
                urljoin(
                    BASE_CSN,
                    a["href"]
                )
            )


    filas.append(
        {
            "Estacion":
                code,

            "Station_URL":
                urljoin(
                    BASE_CSN,
                    station_anchor["href"]
                ),

            "N_links_fila":
                len(todos_links),

            "Links_fila":
                json.dumps(
                    todos_links,
                    ensure_ascii=False
                ),

            "N_links_texto":
                len(links_texto),

            "Links_texto":
                json.dumps(
                    links_texto,
                    ensure_ascii=False
                )
        }
    )


estaciones = pd.DataFrame(
    filas
).drop_duplicates(
    subset=["Estacion"]
).reset_index(
    drop=True
)


print("\n" + "=" * 80)
print("ESTACIONES EN PÁGINA DEL EVENTO")
print("=" * 80)

print(
    "Número:",
    len(estaciones)
)

display(
    estaciones[
        [
            "Estacion",
            "N_links_texto",
            "Station_URL"
        ]
    ]
)


# ======================================================================
# 9. METADATA DE CADA ESTACIÓN
# ======================================================================

def extraer_numero(
    texto,
    patron
):

    m = re.search(
        patron,
        texto,
        flags=re.I
    )

    if not m:
        return np.nan

    try:

        return float(
            m.group(1)
        )

    except Exception:

        return np.nan


latitudes = []
longitudes = []
elevaciones = []
vs30_csn = []
nombres_estacion = []


for i, row in estaciones.iterrows():

    url = row[
        "Station_URL"
    ]

    print(
        f"Metadata {i+1}/{len(estaciones)}: "
        f"{row['Estacion']}"
    )

    html = descargar_html(
        url
    )

    ss = BeautifulSoup(
        html,
        "html.parser"
    )

    txt = ss.get_text(
        " ",
        strip=True
    )


    latitudes.append(
        extraer_numero(
            txt,
            r"Latitud:\s*(-?\d+(?:\.\d+)?)"
        )
    )

    longitudes.append(
        extraer_numero(
            txt,
            r"Longitud:\s*(-?\d+(?:\.\d+)?)"
        )
    )

    elevaciones.append(
        extraer_numero(
            txt,
            r"Elevación:\s*(-?\d+(?:\.\d+)?)"
        )
    )

    vs30_csn.append(
        extraer_numero(
            txt,
            r"Vs30\s+([0-9]+(?:\.[0-9]+)?)"
        )
    )


    h = ss.find(
        ["h1", "h2"]
    )

    nombres_estacion.append(
        h.get_text(
            " ",
            strip=True
        )
        if h
        else ""
    )


    time.sleep(
        0.05
    )


estaciones[
    "Latitud_CSN"
] = latitudes

estaciones[
    "Longitud_CSN"
] = longitudes

estaciones[
    "Elevacion_m_CSN"
] = elevaciones

# SOLO PARA AUDITORÍA SECUNDARIA.
# NO será predictor principal.
estaciones[
    "Vs30_CSN_m_s_auditoria"
] = vs30_csn

estaciones[
    "Nombre_estacion"
] = nombres_estacion


# ======================================================================
# 10. DESCARGAR DATOS DE TEXTO PLANO
# ======================================================================

def nombre_descarga(
    response,
    url,
    fallback
):

    cd = response.headers.get(
        "Content-Disposition",
        ""
    )

    m = re.search(
        r'filename="?([^";]+)"?',
        cd,
        re.I
    )

    if m:

        return os.path.basename(
            m.group(1)
        )


    nombre = os.path.basename(
        urlparse(url).path
    )

    if nombre:

        return nombre


    return fallback


archivos_por_estacion = {}


for i, row in estaciones.iterrows():

    codigo = row[
        "Estacion"
    ]

    carpeta_est = os.path.join(
        CARPETA_REG,
        codigo
    )

    os.makedirs(
        carpeta_est,
        exist_ok=True
    )


    try:

        links = json.loads(
            row["Links_texto"]
        )

    except Exception:

        links = []


    archivos_extraidos = []


    for j, url in enumerate(
        links
    ):

        try:

            rr = session.get(
                url,
                timeout=90
            )

            rr.raise_for_status()


            nombre = nombre_descarga(
                rr,
                url,
                f"{codigo}_texto_{j+1}"
            )


            ruta = os.path.join(
                carpeta_est,
                nombre
            )


            with open(
                ruta,
                "wb"
            ) as f:

                f.write(
                    rr.content
                )


            # ¿ZIP?
            bio = io.BytesIO(
                rr.content
            )

            if zipfile.is_zipfile(
                bio
            ):

                with zipfile.ZipFile(
                    bio
                ) as z:

                    z.extractall(
                        carpeta_est
                    )

                    for n in z.namelist():

                        if not n.endswith(
                            "/"
                        ):

                            archivos_extraidos.append(
                                os.path.basename(
                                    n
                                )
                            )

            else:

                archivos_extraidos.append(
                    nombre
                )


        except Exception as e:

            print(
                f"⚠ {codigo}: "
                f"falló descarga {url}"
            )

            print(
                "  ",
                e
            )


    # Además registrar todo lo físicamente presente
    presentes = []

    for raiz, dirs, files in os.walk(
        carpeta_est
    ):

        for file in files:

            presentes.append(
                os.path.relpath(
                    os.path.join(
                        raiz,
                        file
                    ),
                    carpeta_est
                )
            )


    archivos_por_estacion[
        codigo
    ] = sorted(
        set(
            presentes
        )
    )


# ======================================================================
# 11. IDENTIFICAR COMPONENTES SEGÚN NOMBRE DE ARCHIVO
# ======================================================================

def componente_archivo(
    nombre
):

    base = os.path.basename(
        nombre
    ).upper()

    # quitar extensiones sucesivas
    stem = base

    while "." in stem:

        nuevo = os.path.splitext(
            stem
        )[0]

        if nuevo == stem:
            break

        stem = nuevo


    # CSN:
    # última letra del código de canal = E, N o Z.
    # La mayoría de archivos termina directamente
    # en dicha componente.
    if len(stem) > 0:

        if stem[-1] in {
            "E",
            "N",
            "Z"
        }:

            return stem[-1]


    # fallback: canales típicos HNE/HNN/HNZ,
    # HLE/HLN/HLZ, etc.
    m = re.search(
        r"[A-Z0-9]{2}([ENZ])(?:[^A-Z0-9]|$)",
        base
    )

    if m:

        return m.group(1)


    return None


lista_archivos = []
lista_componentes = []
tiene_E = []
tiene_N = []
tiene_Z = []


for codigo in estaciones[
    "Estacion"
]:

    files = archivos_por_estacion.get(
        codigo,
        []
    )

    comps = sorted(
        {
            componente_archivo(f)
            for f in files
            if componente_archivo(f)
            is not None
        }
    )


    lista_archivos.append(
        " | ".join(
            files
        )
    )

    lista_componentes.append(
        ",".join(
            comps
        )
    )

    tiene_E.append(
        "E" in comps
    )

    tiene_N.append(
        "N" in comps
    )

    tiene_Z.append(
        "Z" in comps
    )


estaciones[
    "Archivos_descargados"
] = lista_archivos

estaciones[
    "Componentes_detectadas"
] = lista_componentes

estaciones[
    "Tiene_E"
] = tiene_E

estaciones[
    "Tiene_N"
] = tiene_N

estaciones[
    "Tiene_Z"
] = tiene_Z

estaciones[
    "Dos_horizontales_EN"
] = (
    estaciones["Tiene_E"]
    &
    estaciones["Tiene_N"]
)


# ======================================================================
# 12. CONTROLES GEOGRÁFICOS
# ======================================================================

estaciones[
    "Coordenadas_validas"
] = (
    estaciones[
        "Latitud_CSN"
    ].between(
        -40,
        -25
    )
    &
    estaciones[
        "Longitud_CSN"
    ].between(
        -75,
        -65
    )
)


# ======================================================================
# 13. RESULTADO
# ======================================================================

print("\n" + "=" * 80)
print("AUDITORÍA FINAL DE ESTACIONES CSN")
print("=" * 80)

display(
    estaciones[
        [
            "Estacion",
            "Latitud_CSN",
            "Longitud_CSN",
            "Elevacion_m_CSN",
            "Vs30_CSN_m_s_auditoria",
            "N_links_texto",
            "Componentes_detectadas",
            "Tiene_E",
            "Tiene_N",
            "Tiene_Z",
            "Dos_horizontales_EN",
            "Coordenadas_validas"
        ]
    ].round(
        {
            "Latitud_CSN": 6,
            "Longitud_CSN": 6,
            "Elevacion_m_CSN": 2,
            "Vs30_CSN_m_s_auditoria": 2
        }
    )
)


N_TOTAL = len(
    estaciones
)

N_COORD = int(
    estaciones[
        "Coordenadas_validas"
    ].sum()
)

N_EN = int(
    estaciones[
        "Dos_horizontales_EN"
    ].sum()
)


print("\nResumen:")

print(
    "Estaciones totales            :",
    N_TOTAL
)

print(
    "Con coordenadas válidas       :",
    N_COORD
)

print(
    "Con componentes E + N         :",
    N_EN
)


# ======================================================================
# 14. INVENTARIO CANDIDATO CONGELABLE
# ======================================================================

candidatas = estaciones[
    estaciones[
        "Dos_horizontales_EN"
    ]
    &
    estaciones[
        "Coordenadas_validas"
    ]
].copy()


candidatas[
    "Mw_congelada"
] = MW_CONGELADA


print("\n" + "=" * 80)
print("INVENTARIO CANDIDATO")
print("=" * 80)

display(
    candidatas[
        [
            "Estacion",
            "Latitud_CSN",
            "Longitud_CSN",
            "Mw_congelada",
            "Componentes_detectadas"
        ]
    ]
)


# ======================================================================
# 15. GUARDAR TODO
# ======================================================================

RUTA_AUDITORIA = os.path.join(
    CARPETA,
    "Valparaiso2017_estaciones_CSN_auditoria.csv"
)

RUTA_CANDIDATAS = os.path.join(
    CARPETA,
    "Valparaiso2017_estaciones_candidatas_PRE_Sa_PRE_Rrup.csv"
)

estaciones.to_csv(
    RUTA_AUDITORIA,
    index=False
)

candidatas.to_csv(
    RUTA_CANDIDATAS,
    index=False
)


resumen = {
    "evento_CSN_url":
        URL_EVENTO,

    "fecha_objetivo_USGS":
        str(TARGET),

    "Mw_predictora_congelada":
        MW_CONGELADA,

    "n_estaciones_total":
        int(N_TOTAL),

    "n_coordenadas_validas":
        int(N_COORD),

    "n_dos_horizontales_EN":
        int(N_EN),

    "gate_minimo_10_estaciones":
        bool(
            N_EN >= 10
        ),

    "V5_1_ejecutado":
        False
}


with open(
    os.path.join(
        CARPETA,
        "Valparaiso2017_gate_registros_CSN.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        resumen,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 16. DECISIÓN
# ======================================================================

print("\n" + "=" * 80)
print("GATE DE REGISTROS")
print("=" * 80)


if N_EN >= 10:

    print(
        "✅ PASS"
    )

    print(
        f"{N_EN} estaciones poseen "
        "las dos componentes horizontales."
    )

    print(
        "\nValparaíso puede continuar a:"
    )

    print(
        "- cálculo auditado de Rrup;"
    )

    print(
        "- procesamiento espectral RotD50;"
    )

    print(
        "- Vs30 USGS nearest."
    )

else:

    print(
        "❌ FAIL"
    )

    print(
        f"Solo {N_EN} estaciones con "
        "E + N."
    )

    print(
        "No ejecutar V5.1."
    )


print("\nArchivos guardados:")
print(RUTA_AUDITORIA)
print(RUTA_CANDIDATAS)

print("\nV5.1 NO FUE EJECUTADO.")

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — CSN ROBUSTO / REANUDABLE
#
# Continúa la auditoría después del HTTP 503.
#
# OBJETIVOS:
# - usar directamente el evento CSN ya identificado;
# - recuperar las 42 estaciones;
# - consultar metadata con retry + backoff;
# - guardar progreso después de cada estación;
# - descargar registros disponibles sin golpear el servidor;
# - identificar componentes E/N/Z;
# - determinar cuántas estaciones tienen E+N.
#
# NO calcula Rrup.
# NO calcula Sa.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import io
import json
import time
import random
import zipfile
import requests
import numpy as np
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

# ======================================================================
# 1. CONFIGURACIÓN CONGELADA
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

CARPETA_REG = os.path.join(
    CARPETA,
    "Registros_CSN"
)

CARPETA_META = os.path.join(
    CARPETA,
    "Metadata_estaciones_CSN"
)

os.makedirs(
    CARPETA_REG,
    exist_ok=True
)

os.makedirs(
    CARPETA_META,
    exist_ok=True
)

BASE_CSN = "https://evtdb.csn.uchile.cl"

# Evento YA identificado inequívocamente:
EVENT_HASH = "6c5752b76db0f46280949a798662a224"

URL_EVENTO = (
    f"{BASE_CSN}/event/{EVENT_HASH}"
)

MW_CONGELADA = 6.9

print("=" * 80)
print("VALPARAÍSO 2017 — CSN ROBUSTO")
print("=" * 80)

print("Evento CSN :", URL_EVENTO)
print("Mw         :", MW_CONGELADA)


# ======================================================================
# 2. SESIÓN HTTP
# ======================================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent":
            "Mozilla/5.0 "
            "(Tesis academica; "
            "auditoria reproducible; "
            "uso no automatizado intensivo)"
    }
)


# ======================================================================
# 3. GET ROBUSTO CON BACKOFF
# ======================================================================

def get_robusto(
    url,
    intentos=7,
    timeout=90
):
    """
    GET respetuoso con el servidor.

    Reintenta:
    429, 500, 502, 503 y 504.

    Nunca usa verify=False.
    """

    for intento in range(
        1,
        intentos + 1
    ):

        try:

            r = session.get(
                url,
                timeout=timeout
            )

            if r.status_code == 200:

                return r


            if r.status_code in {
                429,
                500,
                502,
                503,
                504
            }:

                retry_after = (
                    r.headers.get(
                        "Retry-After"
                    )
                )

                if retry_after:

                    try:
                        espera = float(
                            retry_after
                        )
                    except:
                        espera = None

                else:
                    espera = None


                if espera is None:

                    # backoff:
                    # 5, 10, 20, 40, 60...
                    espera = min(
                        60,
                        5 * (
                            2 ** (
                                intento - 1
                            )
                        )
                    )

                    # pequeño jitter
                    espera += random.uniform(
                        0.5,
                        2.0
                    )


                print(
                    f"⚠ HTTP {r.status_code} "
                    f"| intento {intento}/{intentos}"
                )

                print(
                    f"  Esperando "
                    f"{espera:.1f} s..."
                )

                time.sleep(
                    espera
                )

                continue


            # Otro código HTTP:
            print(
                f"❌ HTTP {r.status_code}: "
                f"{url}"
            )

            return None


        except requests.RequestException as e:

            espera = min(
                60,
                5 * (
                    2 ** (
                        intento - 1
                    )
                )
            )

            print(
                f"⚠ Error de red "
                f"| intento "
                f"{intento}/{intentos}"
            )

            print(
                " ",
                repr(e)
            )

            print(
                f"  Esperando "
                f"{espera:.1f} s..."
            )

            time.sleep(
                espera
            )


    print(
        "❌ No fue posible recuperar:"
    )

    print(
        url
    )

    return None


# ======================================================================
# 4. PÁGINA DEL EVENTO
# ======================================================================

RUTA_EVENTO_HTML = os.path.join(
    CARPETA,
    "Valparaiso2017_CSN_event.html"
)


# reutilizar copia previa si existe
if os.path.exists(
    RUTA_EVENTO_HTML
):

    print(
        "\nUsando HTML del evento "
        "ya guardado."
    )

    with open(
        RUTA_EVENTO_HTML,
        "r",
        encoding="utf-8"
    ) as f:

        html_evento = f.read()

else:

    rr = get_robusto(
        URL_EVENTO
    )

    if rr is None:

        raise RuntimeError(
            "No pudo recuperarse "
            "la página del evento."
        )

    html_evento = rr.text

    with open(
        RUTA_EVENTO_HTML,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            html_evento
        )


soup = BeautifulSoup(
    html_evento,
    "html.parser"
)


# ======================================================================
# 5. RECONSTRUIR INVENTARIO DE LAS 42 ESTACIONES
# ======================================================================

filas = []

for tr in soup.find_all(
    "tr"
):

    station_anchor = None

    for a in tr.find_all(
        "a",
        href=True
    ):

        if "/station/" in a["href"]:

            station_anchor = a
            break


    if station_anchor is None:

        continue


    codigo = (
        station_anchor
        .get_text(
            " ",
            strip=True
        )
        .upper()
    )


    tds = tr.find_all(
        "td"
    )


    links_texto = []

    # En la tabla CSN,
    # tercera columna = datos texto plano
    if len(tds) >= 3:

        for a in tds[2].find_all(
            "a",
            href=True
        ):

            links_texto.append(
                urljoin(
                    BASE_CSN,
                    a["href"]
                )
            )


    filas.append(
        {
            "Estacion":
                codigo,

            "Station_URL":
                urljoin(
                    BASE_CSN,
                    station_anchor["href"]
                ),

            "N_links_texto":
                len(
                    links_texto
                ),

            "Links_texto":
                json.dumps(
                    links_texto,
                    ensure_ascii=False
                )
        }
    )


estaciones = (
    pd.DataFrame(
        filas
    )
    .drop_duplicates(
        subset=[
            "Estacion"
        ]
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 80)
print("INVENTARIO RECONSTRUIDO")
print("=" * 80)

print(
    "Estaciones:",
    len(estaciones)
)

assert len(estaciones) == 42, (
    "Se esperaban las 42 estaciones "
    "ya observadas."
)

display(
    estaciones[
        [
            "Estacion",
            "N_links_texto",
            "Station_URL"
        ]
    ]
)


# Guardar inmediatamente
RUTA_INVENTARIO = os.path.join(
    CARPETA,
    "Valparaiso2017_inventario_42_estaciones_CSN.csv"
)

estaciones.to_csv(
    RUTA_INVENTARIO,
    index=False
)


# ======================================================================
# 6. PARSER DE METADATA
# ======================================================================

def extraer_numero(
    texto,
    patrones
):

    if isinstance(
        patrones,
        str
    ):

        patrones = [
            patrones
        ]


    for patron in patrones:

        m = re.search(
            patron,
            texto,
            flags=re.I
        )

        if m:

            try:

                return float(
                    m.group(1)
                )

            except:

                pass


    return np.nan


# ======================================================================
# 7. DESCARGAR METADATA CON CACHE
# ======================================================================

columnas_nuevas = [
    "Latitud_CSN",
    "Longitud_CSN",
    "Elevacion_m_CSN",
    "Vs30_CSN_m_s_auditoria",
    "Metadata_HTTP_OK"
]

for c in columnas_nuevas:

    if c not in estaciones.columns:

        estaciones[c] = np.nan


RUTA_PARCIAL = os.path.join(
    CARPETA,
    "Valparaiso2017_estaciones_CSN_METADATA_PARCIAL.csv"
)


for i, row in estaciones.iterrows():

    codigo = row[
        "Estacion"
    ]

    url = row[
        "Station_URL"
    ]

    archivo_html = os.path.join(
        CARPETA_META,
        f"{codigo}.html"
    )


    print(
        f"\n[{i+1:02d}/"
        f"{len(estaciones):02d}] "
        f"{codigo}"
    )


    # ----------------------------------------------------------
    # CACHE
    # ----------------------------------------------------------

    if os.path.exists(
        archivo_html
    ):

        print(
            "  ✓ metadata en cache"
        )

        with open(
            archivo_html,
            "r",
            encoding="utf-8"
        ) as f:

            html = f.read()

        ok = True


    else:

        r = get_robusto(
            url
        )

        if r is None:

            print(
                "  ⚠ metadata no disponible"
            )

            estaciones.loc[
                i,
                "Metadata_HTTP_OK"
            ] = False

            estaciones.to_csv(
                RUTA_PARCIAL,
                index=False
            )

            # Pausa antes de seguir
            time.sleep(
                5
            )

            continue


        html = r.text

        with open(
            archivo_html,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                html
            )

        ok = True


    ss = BeautifulSoup(
        html,
        "html.parser"
    )

    txt = ss.get_text(
        " ",
        strip=True
    )


    lat = extraer_numero(
        txt,
        [
            r"Latitud\s*:?\s*(-?\d+(?:\.\d+)?)",
            r"Latitude\s*:?\s*(-?\d+(?:\.\d+)?)"
        ]
    )

    lon = extraer_numero(
        txt,
        [
            r"Longitud\s*:?\s*(-?\d+(?:\.\d+)?)",
            r"Longitude\s*:?\s*(-?\d+(?:\.\d+)?)"
        ]
    )

    elev = extraer_numero(
        txt,
        [
            r"Elevaci[oó]n\s*:?\s*(-?\d+(?:\.\d+)?)",
            r"Elevation\s*:?\s*(-?\d+(?:\.\d+)?)"
        ]
    )

    vs30 = extraer_numero(
        txt,
        [
            r"Vs30\s*:?\s*([0-9]+(?:\.\d+)?)",
            r"V[sS]30\s*:?\s*([0-9]+(?:\.\d+)?)"
        ]
    )


    estaciones.loc[
        i,
        "Latitud_CSN"
    ] = lat

    estaciones.loc[
        i,
        "Longitud_CSN"
    ] = lon

    estaciones.loc[
        i,
        "Elevacion_m_CSN"
    ] = elev

    estaciones.loc[
        i,
        "Vs30_CSN_m_s_auditoria"
    ] = vs30

    estaciones.loc[
        i,
        "Metadata_HTTP_OK"
    ] = ok


    print(
        f"  Lat/Lon: "
        f"{lat}, {lon}"
    )


    # Guardar después de CADA estación
    estaciones.to_csv(
        RUTA_PARCIAL,
        index=False
    )


    # Pausa respetuosa:
    # no martillar el servidor.
    time.sleep(
        3.0
        +
        random.uniform(
            0.2,
            1.0
        )
    )


# ======================================================================
# 8. VALIDAR COORDENADAS
# ======================================================================

estaciones[
    "Coordenadas_validas"
] = (
    estaciones[
        "Latitud_CSN"
    ].between(
        -40,
        -25
    )
    &
    estaciones[
        "Longitud_CSN"
    ].between(
        -75,
        -65
    )
)


print("\n" + "=" * 80)
print("METADATA COMPLETADA")
print("=" * 80)

print(
    "HTTP OK:",
    int(
        (
            estaciones[
                "Metadata_HTTP_OK"
            ]
            == True
        ).sum()
    )
)

print(
    "Coordenadas válidas:",
    int(
        estaciones[
            "Coordenadas_validas"
        ].sum()
    )
)


# ======================================================================
# 9. DESCARGAR LOS REGISTROS
# ======================================================================

def nombre_descarga(
    response,
    url,
    fallback
):

    cd = response.headers.get(
        "Content-Disposition",
        ""
    )

    m = re.search(
        r'filename="?([^";]+)"?',
        cd,
        re.I
    )

    if m:

        return os.path.basename(
            m.group(1)
        )


    path = urlparse(
        url
    ).path

    nombre = os.path.basename(
        path
    )

    if nombre:

        return nombre


    return fallback


print("\n" + "=" * 80)
print("DESCARGA DE REGISTROS")
print("=" * 80)


for i, row in estaciones.iterrows():

    codigo = row[
        "Estacion"
    ]

    carpeta_est = os.path.join(
        CARPETA_REG,
        codigo
    )

    os.makedirs(
        carpeta_est,
        exist_ok=True
    )


    try:

        links = json.loads(
            row[
                "Links_texto"
            ]
        )

    except:

        links = []


    print(
        f"\n[{i+1:02d}/"
        f"{len(estaciones):02d}] "
        f"{codigo} "
        f"| links={len(links)}"
    )


    for j, url in enumerate(
        links
    ):

        # ----------------------------------------------------------
        # Evitar redescarga si ya existen archivos
        # ----------------------------------------------------------

        existentes = [
            f for f in os.listdir(
                carpeta_est
            )
            if not f.startswith(
                "."
            )
        ]


        if len(existentes) > 0:

            print(
                "  ✓ ya descargado"
            )

            break


        r = get_robusto(
            url
        )

        if r is None:

            print(
                "  ⚠ descarga fallida"
            )

            continue


        nombre = nombre_descarga(
            r,
            url,
            f"{codigo}_datos_{j+1}"
        )


        # ¿es zip?
        contenido = r.content

        bio = io.BytesIO(
            contenido
        )


        if zipfile.is_zipfile(
            bio
        ):

            print(
                "  ZIP detectado"
            )

            with zipfile.ZipFile(
                bio
            ) as z:

                z.extractall(
                    carpeta_est
                )


        else:

            ruta = os.path.join(
                carpeta_est,
                nombre
            )

            with open(
                ruta,
                "wb"
            ) as f:

                f.write(
                    contenido
                )


        time.sleep(
            3.0
            +
            random.uniform(
                0.2,
                1.0
            )
        )


# ======================================================================
# 10. IDENTIFICAR COMPONENTES
# ======================================================================

def componente_archivo(
    nombre
):

    base = os.path.basename(
        nombre
    ).upper()


    stem = base

    # retirar extensiones
    for _ in range(4):

        nuevo = os.path.splitext(
            stem
        )[0]

        if nuevo == stem:
            break

        stem = nuevo


    # Caso más común:
    # canal termina en E/N/Z
    if (
        len(stem)
        and
        stem[-1] in {
            "E",
            "N",
            "Z"
        }
    ):

        return stem[-1]


    # Buscar canales típicos:
    # HNE, HNN, HNZ,
    # HLE, HLN, HLZ,
    # ENE, ENN, ENZ, etc.
    patrones = [
        r"(?:HN|HL|EN|BN|BH)([ENZ])",
        r"([ENZ])(?:_|-|$)"
    ]


    for patron in patrones:

        m = re.search(
            patron,
            stem
        )

        if m:

            return m.group(1)


    return None


archivos_txt = []
componentes = []
tiene_E = []
tiene_N = []
tiene_Z = []


for codigo in estaciones[
    "Estacion"
]:

    carpeta_est = os.path.join(
        CARPETA_REG,
        codigo
    )

    files = []


    if os.path.exists(
        carpeta_est
    ):

        for raiz, dirs, fs in os.walk(
            carpeta_est
        ):

            for f in fs:

                files.append(
                    os.path.relpath(
                        os.path.join(
                            raiz,
                            f
                        ),
                        carpeta_est
                    )
                )


    comps = sorted(
        {
            c
            for c in [
                componente_archivo(
                    f
                )
                for f in files
            ]
            if c is not None
        }
    )


    archivos_txt.append(
        " | ".join(
            sorted(files)
        )
    )

    componentes.append(
        ",".join(
            comps
        )
    )

    tiene_E.append(
        "E" in comps
    )

    tiene_N.append(
        "N" in comps
    )

    tiene_Z.append(
        "Z" in comps
    )


estaciones[
    "Archivos_descargados"
] = archivos_txt

estaciones[
    "Componentes_detectadas"
] = componentes

estaciones[
    "Tiene_E"
] = tiene_E

estaciones[
    "Tiene_N"
] = tiene_N

estaciones[
    "Tiene_Z"
] = tiene_Z

estaciones[
    "Dos_horizontales_EN"
] = (
    estaciones[
        "Tiene_E"
    ]
    &
    estaciones[
        "Tiene_N"
    ]
)


# ======================================================================
# 11. RESUMEN
# ======================================================================

print("\n" + "=" * 80)
print("AUDITORÍA FINAL")
print("=" * 80)


columnas_mostrar = [
    "Estacion",
    "Latitud_CSN",
    "Longitud_CSN",
    "Vs30_CSN_m_s_auditoria",
    "Componentes_detectadas",
    "Tiene_E",
    "Tiene_N",
    "Tiene_Z",
    "Dos_horizontales_EN",
    "Coordenadas_validas"
]


display(
    estaciones[
        columnas_mostrar
    ].round(
        {
            "Latitud_CSN": 6,
            "Longitud_CSN": 6,
            "Vs30_CSN_m_s_auditoria": 2
        }
    )
)


N_TOTAL = len(
    estaciones
)

N_COORD = int(
    estaciones[
        "Coordenadas_validas"
    ].sum()
)

N_EN = int(
    estaciones[
        "Dos_horizontales_EN"
    ].sum()
)


print("\nResumen:")

print(
    "Estaciones totales       :",
    N_TOTAL
)

print(
    "Coordenadas válidas      :",
    N_COORD
)

print(
    "Dos horizontales E + N   :",
    N_EN
)


# ======================================================================
# 12. INVENTARIO CANDIDATO
# ======================================================================

candidatas = estaciones[
    estaciones[
        "Coordenadas_validas"
    ]
    &
    estaciones[
        "Dos_horizontales_EN"
    ]
].copy()


candidatas[
    "Mw_congelada"
] = MW_CONGELADA


print("\n" + "=" * 80)
print("INVENTARIO CANDIDATO FINAL")
print("=" * 80)

display(
    candidatas[
        [
            "Estacion",
            "Latitud_CSN",
            "Longitud_CSN",
            "Mw_congelada",
            "Componentes_detectadas"
        ]
    ]
)


# ======================================================================
# 13. GUARDAR
# ======================================================================

RUTA_AUDITORIA = os.path.join(
    CARPETA,
    "Valparaiso2017_estaciones_CSN_auditoria.csv"
)

RUTA_CANDIDATAS = os.path.join(
    CARPETA,
    "Valparaiso2017_estaciones_candidatas_PRE_Sa_PRE_Rrup.csv"
)

estaciones.to_csv(
    RUTA_AUDITORIA,
    index=False
)

candidatas.to_csv(
    RUTA_CANDIDATAS,
    index=False
)


resumen = {
    "event_hash_CSN":
        EVENT_HASH,

    "evento_CSN_url":
        URL_EVENTO,

    "Mw_predictora_congelada":
        MW_CONGELADA,

    "n_estaciones_total":
        int(
            N_TOTAL
        ),

    "n_coordenadas_validas":
        int(
            N_COORD
        ),

    "n_dos_horizontales_EN":
        int(
            N_EN
        ),

    "gate_minimo_10":
        bool(
            N_EN >= 10
        ),

    "V5_1_ejecutado":
        False
}


RUTA_GATE = os.path.join(
    CARPETA,
    "Valparaiso2017_gate_registros_CSN.json"
)


with open(
    RUTA_GATE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        resumen,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 14. GATE
# ======================================================================

print("\n" + "=" * 80)
print("GATE DE REGISTROS")
print("=" * 80)


if N_EN >= 10:

    print(
        "✅ PASS"
    )

    print(
        f"{N_EN} estaciones tienen "
        "coordenadas válidas y E+N."
    )

    print(
        "\nValparaíso queda autorizado "
        "para la siguiente etapa:"
    )

    print(
        "Rrup + RotD50 + Vs30."
    )

else:

    print(
        "❌ FAIL"
    )

    print(
        f"Solo {N_EN} estaciones "
        "cumplen E+N."
    )


print("\nArchivos guardados:")

print(
    RUTA_AUDITORIA
)

print(
    RUTA_CANDIDATAS
)

print(
    RUTA_GATE
)

print("\nV5.1 NO FUE EJECUTADO.")

In [ ]:
# ======================================================================
# VALPARAÍSO 2017
# AUDITORÍA FINAL DE INPUTS: COORDENADAS + Rrup + Vs30
#
# Evento: USGS us10008kce
# Mw predictora congelada = 6.9
#
# FUENTES:
# - Coordenadas: encabezados de acelerogramas CSN E/N
# - Rrup: USGS FFM.geojson finite-fault
# - Vs30: global_vs30.grd, nearest (método histórico congelado)
#
# NO calcula Sa.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import hashlib
import numpy as np
import pandas as pd
import xarray as xr

from pyproj import CRS, Transformer


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

REG_DIR = os.path.join(
    CARPETA,
    "Registros_CSN"
)

FFM_PATH = os.path.join(
    CARPETA,
    "FFM.geojson"
)

VS30_PATH = os.path.join(
    BASE,
    "global_vs30.grd"
)

MW_CONGELADA = 6.9


for ruta in [
    REG_DIR,
    FFM_PATH,
    VS30_PATH
]:
    assert os.path.exists(ruta), ruta


print("=" * 80)
print("VALPARAÍSO 2017 — AUDITORÍA DE INPUTS")
print("=" * 80)

print("Mw congelada :", MW_CONGELADA)
print("FFM          :", FFM_PATH)
print("Vs30 raster  :", VS30_PATH)


# ======================================================================
# 2. HASH DEL MODELO DE RUPTURA
# ======================================================================

def sha256_file(
    path,
    chunk=1024*1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            bloque = f.read(
                chunk
            )

            if not bloque:
                break

            h.update(
                bloque
            )

    return h.hexdigest()


print(
    "\nSHA256 FFM:",
    sha256_file(
        FFM_PATH
    )
)


# ======================================================================
# 3. LEER ENCABEZADOS DE ACELEROGRAMAS
# ======================================================================

def leer_header_csn(
    path
):

    lineas = []

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        for _ in range(30):

            linea = f.readline()

            if not linea:
                break

            if linea.startswith("#"):
                lineas.append(
                    linea.strip()
                )

            else:
                break


    texto = "\n".join(
        lineas
    )


    def buscar(
        patron,
        tipo=str
    ):

        m = re.search(
            patron,
            texto,
            re.I
        )

        if not m:
            return None

        return tipo(
            m.group(1)
        )


    resultado = {

        "Tiempo_origen":
            buscar(
                r"Tiempo de Origen:\s*(.+)"
            ),

        "Fs_Hz":
            buscar(
                r"Tasa de muestreo:\s*"
                r"([0-9.]+)",
                float
            ),

        "N_header":
            buscar(
                r"Numero total de muestras:\s*"
                r"([0-9]+)",
                int
            ),

        "Estacion_header":
            buscar(
                r"Estacion:\s*([A-Za-z0-9]+)"
            ),

        "Componente":
            buscar(
                r"Componente:\s*([A-Za-z0-9]+)"
            ),

        "Latitud":
            buscar(
                r"Latitud:\s*"
                r"(-?[0-9.]+)",
                float
            ),

        "Longitud":
            buscar(
                r"Longitud:\s*"
                r"(-?[0-9.]+)",
                float
            ),

        "Unidades":
            buscar(
                r"Unidades:\s*(.+)"
            )
    }


    # Verificar número real de muestras
    datos = np.loadtxt(
        path,
        comments="#"
    )

    datos = np.asarray(
        datos
    ).reshape(-1)


    resultado[
        "N_real"
    ] = len(
        datos
    )


    resultado[
        "Finite"
    ] = bool(
        np.isfinite(
            datos
        ).all()
    )


    resultado[
        "PGA_abs_m_s2"
    ] = float(
        np.max(
            np.abs(
                datos
            )
        )
    )


    return resultado


# ======================================================================
# 4. INVENTARIAR E + N
# ======================================================================

filas = []


estaciones_carpetas = sorted(
    [
        x for x in os.listdir(
            REG_DIR
        )
        if os.path.isdir(
            os.path.join(
                REG_DIR,
                x
            )
        )
    ]
)


print(
    "\nCarpetas de estaciones:",
    len(estaciones_carpetas)
)


for codigo in estaciones_carpetas:

    carpeta = os.path.join(
        REG_DIR,
        codigo
    )

    archivos = [
        os.path.join(
            carpeta,
            f
        )
        for f in os.listdir(
            carpeta
        )
        if f.lower().endswith(
            ".txt"
        )
    ]


    componentes = {}


    for path in archivos:

        h = leer_header_csn(
            path
        )

        comp = str(
            h["Componente"]
        ).upper()

        componentes[
            comp
        ] = (
            path,
            h
        )


    # Detectar horizontal E
    e_candidates = [
        k for k in componentes
        if k.endswith("E")
    ]

    n_candidates = [
        k for k in componentes
        if k.endswith("N")
    ]


    if (
        len(e_candidates) != 1
        or
        len(n_candidates) != 1
    ):

        raise RuntimeError(
            f"{codigo}: componentes ambiguas. "
            f"E={e_candidates}, "
            f"N={n_candidates}"
        )


    compE = e_candidates[0]
    compN = n_candidates[0]

    pathE, E = componentes[
        compE
    ]

    pathN, N = componentes[
        compN
    ]


    # --------------------------------------------------------------
    # CONSISTENCIA E vs N
    # --------------------------------------------------------------

    assert (
        E["Estacion_header"].upper()
        ==
        codigo.upper()
    )

    assert (
        N["Estacion_header"].upper()
        ==
        codigo.upper()
    )


    assert np.isclose(
        E["Latitud"],
        N["Latitud"],
        atol=1e-8
    ), f"{codigo}: lat E/N distinta"


    assert np.isclose(
        E["Longitud"],
        N["Longitud"],
        atol=1e-8
    ), f"{codigo}: lon E/N distinta"


    assert np.isclose(
        E["Fs_Hz"],
        N["Fs_Hz"],
        atol=1e-12
    ), f"{codigo}: Fs E/N distinta"


    assert (
        E["N_header"]
        ==
        E["N_real"]
    ), f"{codigo}: N E inconsistente"


    assert (
        N["N_header"]
        ==
        N["N_real"]
    ), f"{codigo}: N N inconsistente"


    assert (
        E["N_real"]
        ==
        N["N_real"]
    ), f"{codigo}: longitudes E/N distintas"


    assert (
        E["Finite"]
        and
        N["Finite"]
    ), f"{codigo}: NaN/Inf encontrado"


    filas.append(
        {
            "Estacion":
                codigo,

            "Latitud":
                E["Latitud"],

            "Longitud":
                E["Longitud"],

            "Fs_Hz":
                E["Fs_Hz"],

            "N_muestras":
                E["N_real"],

            "Duracion_s":
                E["N_real"]
                /
                E["Fs_Hz"],

            "Unidad_E":
                E["Unidades"],

            "Unidad_N":
                N["Unidades"],

            "PGA_E_m_s2":
                E["PGA_abs_m_s2"],

            "PGA_N_m_s2":
                N["PGA_abs_m_s2"],

            "Archivo_E":
                pathE,

            "Archivo_N":
                pathN,

            "Componente_E":
                compE,

            "Componente_N":
                compN,

            "Inicio_E":
                E["Tiempo_origen"],

            "Inicio_N":
                N["Tiempo_origen"]
        }
    )


est = pd.DataFrame(
    filas
)


print("\n" + "=" * 80)
print("AUDITORÍA DE HEADERS E/N")
print("=" * 80)

print(
    "Estaciones auditadas:",
    len(est)
)

display(
    est[
        [
            "Estacion",
            "Latitud",
            "Longitud",
            "Fs_Hz",
            "N_muestras",
            "Duracion_s",
            "Unidad_E",
            "Unidad_N"
        ]
    ]
)


assert len(est) == 42


# ======================================================================
# 5. CONTROL DE PRECISIÓN DE COORDENADAS
# ======================================================================

print("\n" + "=" * 80)
print("COORDENADAS PRECISAS DE ACELEROGRAMAS")
print("=" * 80)

display(
    est[
        [
            "Estacion",
            "Latitud",
            "Longitud"
        ]
    ]
)


# No aceptar las coordenadas redondeadas a 0.1°
# provenientes de la página de estación.
#
# Las coordenadas congeladas desde aquí son
# las del HEADER de los registros.


# ======================================================================
# 6. CARGAR FFM.GEOJSON
# ======================================================================

with open(
    FFM_PATH,
    "r",
    encoding="utf-8"
) as f:

    ffm = json.load(
        f
    )


# ======================================================================
# 7. EXTRAER POLÍGONOS DE SUBFALLA
# ======================================================================

poligonos = []


def recorrer_geojson(
    obj
):

    if isinstance(
        obj,
        dict
    ):

        tipo = obj.get(
            "type"
        )

        if tipo == "Polygon":

            coords = obj[
                "coordinates"
            ]

            if len(coords):

                poligonos.append(
                    coords[0]
                )


        elif tipo == "MultiPolygon":

            for poly in obj[
                "coordinates"
            ]:

                if len(poly):

                    poligonos.append(
                        poly[0]
                    )


        for valor in obj.values():

            recorrer_geojson(
                valor
            )


    elif isinstance(
        obj,
        list
    ):

        for valor in obj:

            recorrer_geojson(
                valor
            )


recorrer_geojson(
    ffm
)


print("\n" + "=" * 80)
print("MALLA FINITE-FAULT")
print("=" * 80)

print(
    "Polígonos encontrados:",
    len(poligonos)
)


assert len(poligonos) > 0


# ======================================================================
# 8. INSPECCIONAR PROFUNDIDAD Y UNIDADES
# ======================================================================

todos_xyz = []


for ring in poligonos:

    for p in ring:

        if len(p) >= 3:

            todos_xyz.append(
                p[:3]
            )


todos_xyz = np.asarray(
    todos_xyz,
    dtype=float
)


z_raw = todos_xyz[
    :,
    2
]


print(
    "Profundidad raw min/max:",
    z_raw.min(),
    z_raw.max()
)


# USGS FFM.geojson de este evento:
# profundidad viene en METROS.
if np.nanmedian(
    np.abs(
        z_raw
    )
) > 1000:

    FACTOR_Z = 1 / 1000.0
    unidad_z = "m -> km"

else:

    FACTOR_Z = 1.0
    unidad_z = "km"


print(
    "Conversión profundidad:",
    unidad_z
)


# ======================================================================
# 9. PROYECCIÓN LOCAL
# ======================================================================

lon0 = float(
    np.mean(
        todos_xyz[
            :,
            0
        ]
    )
)

lat0 = float(
    np.mean(
        todos_xyz[
            :,
            1
        ]
    )
)


crs_local = CRS.from_proj4(
    f"+proj=aeqd "
    f"+lat_0={lat0} "
    f"+lon_0={lon0} "
    f"+datum=WGS84 "
    f"+units=m "
    f"+no_defs"
)


transformer = Transformer.from_crs(
    "EPSG:4326",
    crs_local,
    always_xy=True
)


print(
    "Centro proyección:",
    lat0,
    lon0
)


# ======================================================================
# 10. TRIANGULAR SUBFALLAS
# ======================================================================

triangulos = []


for ring in poligonos:

    arr = np.asarray(
        ring,
        dtype=float
    )


    # Eliminar último vértice si cierra el polígono
    if (
        len(arr) >= 2
        and
        np.allclose(
            arr[0],
            arr[-1]
        )
    ):

        arr = arr[:-1]


    if len(arr) < 3:

        continue


    x_m, y_m = transformer.transform(
        arr[:, 0],
        arr[:, 1]
    )


    xyz = np.column_stack(
        [
            np.asarray(
                x_m
            ) / 1000.0,

            np.asarray(
                y_m
            ) / 1000.0,

            arr[:, 2]
            * FACTOR_Z
        ]
    )


    # FFM está compuesto por cuadriláteros.
    # Fan triangulation desde vértice 0.
    for j in range(
        1,
        len(xyz) - 1
    ):

        triangulos.append(
            (
                xyz[0],
                xyz[j],
                xyz[j+1]
            )
        )


print(
    "Triángulos creados:",
    len(triangulos)
)


assert len(
    triangulos
) > 0


# ======================================================================
# 11. DISTANCIA PUNTO-TRIÁNGULO 3D
# ======================================================================
#
# Implementación del algoritmo geométrico estándar
# de mínima distancia euclídea a un triángulo.
# ======================================================================

def distancia_punto_triangulo(
    p,
    a,
    b,
    c
):

    ab = b - a
    ac = c - a
    ap = p - a

    d1 = np.dot(
        ab,
        ap
    )

    d2 = np.dot(
        ac,
        ap
    )

    if (
        d1 <= 0
        and
        d2 <= 0
    ):

        return np.linalg.norm(
            p - a
        )


    bp = p - b

    d3 = np.dot(
        ab,
        bp
    )

    d4 = np.dot(
        ac,
        bp
    )

    if (
        d3 >= 0
        and
        d4 <= d3
    ):

        return np.linalg.norm(
            p - b
        )


    vc = (
        d1 * d4
        -
        d3 * d2
    )

    if (
        vc <= 0
        and
        d1 >= 0
        and
        d3 <= 0
    ):

        v = (
            d1
            /
            (d1 - d3)
        )

        proj = (
            a
            +
            v * ab
        )

        return np.linalg.norm(
            p - proj
        )


    cp = p - c

    d5 = np.dot(
        ab,
        cp
    )

    d6 = np.dot(
        ac,
        cp
    )

    if (
        d6 >= 0
        and
        d5 <= d6
    ):

        return np.linalg.norm(
            p - c
        )


    vb = (
        d5 * d2
        -
        d1 * d6
    )

    if (
        vb <= 0
        and
        d2 >= 0
        and
        d6 <= 0
    ):

        w = (
            d2
            /
            (d2 - d6)
        )

        proj = (
            a
            +
            w * ac
        )

        return np.linalg.norm(
            p - proj
        )


    va = (
        d3 * d6
        -
        d5 * d4
    )

    if (
        va <= 0
        and
        (d4 - d3) >= 0
        and
        (d5 - d6) >= 0
    ):

        bc = c - b

        w = (
            (d4 - d3)
            /
            (
                (d4 - d3)
                +
                (d5 - d6)
            )
        )

        proj = (
            b
            +
            w * bc
        )

        return np.linalg.norm(
            p - proj
        )


    denom = 1.0 / (
        va
        +
        vb
        +
        vc
    )

    v = vb * denom
    w = vc * denom

    proj = (
        a
        +
        ab * v
        +
        ac * w
    )

    return np.linalg.norm(
        p - proj
    )


# ======================================================================
# 12. CALCULAR Rrup
# ======================================================================

rrups = []


for _, row in est.iterrows():

    x_m, y_m = transformer.transform(
        row["Longitud"],
        row["Latitud"]
    )


    # Estación en superficie z = 0 km.
    p = np.array(
        [
            x_m / 1000.0,
            y_m / 1000.0,
            0.0
        ],
        dtype=float
    )


    dmin = np.inf


    for a, b, c in triangulos:

        d = distancia_punto_triangulo(
            p,
            a,
            b,
            c
        )

        if d < dmin:

            dmin = d


    rrups.append(
        dmin
    )


est[
    "Rrup_km"
] = rrups


assert np.isfinite(
    est["Rrup_km"]
).all()

assert (
    est["Rrup_km"] > 0
).all()


print("\n" + "=" * 80)
print("Rrup — RESULTADOS")
print("=" * 80)

display(
    est[
        [
            "Estacion",
            "Latitud",
            "Longitud",
            "Rrup_km"
        ]
    ].sort_values(
        "Rrup_km"
    ).round(4)
)


print(
    "\nRrup mínimo:",
    est["Rrup_km"].min()
)

print(
    "Rrup máximo:",
    est["Rrup_km"].max()
)


# Control físico:
# la falla comienza aproximadamente a 14.4 km
# de profundidad.
assert (
    est["Rrup_km"].min()
    >
    10
), (
    "Rrup mínimo sospechosamente pequeño."
)


# ======================================================================
# 13. Vs30 HISTÓRICO — NEAREST
# ======================================================================

print("\n" + "=" * 80)
print("Vs30 — global_vs30.grd / nearest")
print("=" * 80)


ds = xr.open_dataset(
    VS30_PATH
)


print(
    ds
)


# detectar variable
if "z" in ds.data_vars:

    da = ds[
        "z"
    ]

else:

    if len(
        ds.data_vars
    ) != 1:

        raise RuntimeError(
            "No se pudo identificar "
            "variable Vs30."
        )

    da = ds[
        list(
            ds.data_vars
        )[0]
    ]


vs30 = []
raster_lat = []
raster_lon = []


for _, row in est.iterrows():

    sel = da.sel(
        lat=row["Latitud"],
        lon=row["Longitud"],
        method="nearest"
    )


    valor = float(
        np.asarray(
            sel.values
        ).squeeze()
    )


    lat_sel = float(
        np.asarray(
            sel["lat"].values
        ).squeeze()
    )

    lon_sel = float(
        np.asarray(
            sel["lon"].values
        ).squeeze()
    )


    vs30.append(
        valor
    )

    raster_lat.append(
        lat_sel
    )

    raster_lon.append(
        lon_sel
    )


est[
    "Vs30_m_s"
] = vs30

est[
    "Vs30_raster_lat"
] = raster_lat

est[
    "Vs30_raster_lon"
] = raster_lon


assert np.isfinite(
    est["Vs30_m_s"]
).all()

assert (
    est["Vs30_m_s"] > 0
).all()


print(
    "Vs30 mínimo:",
    est["Vs30_m_s"].min()
)

print(
    "Vs30 máximo:",
    est["Vs30_m_s"].max()
)

print(
    "Vs30 medio:",
    est["Vs30_m_s"].mean()
)


# ======================================================================
# 14. MATRIZ CONGELADA PRE-SA
# ======================================================================

matriz = est[
    [
        "Estacion",
        "Latitud",
        "Longitud",
        "Rrup_km",
        "Vs30_m_s",
        "Fs_Hz",
        "N_muestras",
        "Duracion_s",
        "Archivo_E",
        "Archivo_N"
    ]
].copy()


matriz.insert(
    3,
    "Mw",
    MW_CONGELADA
)


matriz = matriz.sort_values(
    "Rrup_km"
).reset_index(
    drop=True
)


print("\n" + "=" * 80)
print("MATRIZ DE INPUTS CONGELADA — PRE Sa")
print("=" * 80)

display(
    matriz[
        [
            "Estacion",
            "Latitud",
            "Longitud",
            "Mw",
            "Rrup_km",
            "Vs30_m_s",
            "Fs_Hz",
            "N_muestras"
        ]
    ].round(
        {
            "Latitud": 5,
            "Longitud": 5,
            "Rrup_km": 3,
            "Vs30_m_s": 3
        }
    )
)


# ======================================================================
# 15. GUARDAR AUDITORÍAS
# ======================================================================

RUTA_HEADERS = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_headers_EN.csv"
)

RUTA_PRE_SA = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_PRE_Sa.csv"
)


est.to_csv(
    RUTA_HEADERS,
    index=False
)

matriz.to_csv(
    RUTA_PRE_SA,
    index=False
)


# Hash de matriz final
hash_matriz = sha256_file(
    RUTA_PRE_SA
)


print("\n" + "=" * 80)
print("ARCHIVOS GUARDADOS")
print("=" * 80)

print(
    RUTA_HEADERS
)

print(
    RUTA_PRE_SA
)

print(
    "\nSHA256 matriz PRE-Sa:",
    hash_matriz
)


# ======================================================================
# 16. GATE
# ======================================================================

gate = (
    len(matriz) == 42
    and
    np.isfinite(
        matriz[
            [
                "Latitud",
                "Longitud",
                "Rrup_km",
                "Vs30_m_s"
            ]
        ].to_numpy()
    ).all()
    and
    (
        matriz["Rrup_km"]
        >
        0
    ).all()
    and
    (
        matriz["Vs30_m_s"]
        >
        0
    ).all()
)


print("\n" + "=" * 80)
print("GATE PRE-SA")
print("=" * 80)


if gate:

    print(
        "✅ PASS"
    )

    print(
        "42 estaciones con:"
    )

    print(
        "- coordenadas precisas desde "
        "headers CSN;"
    )

    print(
        "- Rrup a superficie finite-fault USGS;"
    )

    print(
        "- Vs30 histórico nearest."
    )

    print(
        "\nSiguiente etapa:"
    )

    print(
        "auditoría y cálculo "
        "RotD50(T=1.0 s, 5%)."
    )

else:

    print(
        "❌ FAIL"
    )


print("\nV5.1 NO FUE EJECUTADO.")

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — AUDITORÍA DE SINCRONIZACIÓN E/N
#
# Objetivo:
# - detectar todas las estaciones con diferencias E/N;
# - comparar fecha/hora inicial;
# - comparar Fs;
# - comparar número de muestras;
# - calcular intervalo temporal común;
#
# NO recorta todavía las señales.
# NO calcula Sa.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

REG_DIR = os.path.join(
    CARPETA,
    "Registros_CSN"
)

assert os.path.exists(REG_DIR)


# ----------------------------------------------------------------------
# 1. LECTOR DE HEADER + DATOS
# ----------------------------------------------------------------------

def leer_registro(path):

    header = []

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        for _ in range(30):

            linea = f.readline()

            if not linea:
                break

            if linea.startswith("#"):
                header.append(
                    linea.strip()
                )
            else:
                break


    texto = "\n".join(header)


    def buscar(
        patron,
        tipo=str
    ):

        m = re.search(
            patron,
            texto,
            re.I
        )

        if not m:
            return None

        return tipo(
            m.group(1)
        )


    tiempo_txt = buscar(
        r"Tiempo de Origen:\s*(.+)"
    )

    tiempo = pd.to_datetime(
        tiempo_txt,
        utc=True
    )


    fs = buscar(
        r"Tasa de muestreo:\s*([0-9.]+)",
        float
    )


    n_header = buscar(
        r"Numero total de muestras:\s*([0-9]+)",
        int
    )


    estacion = buscar(
        r"Estacion:\s*([A-Za-z0-9]+)"
    )


    componente = buscar(
        r"Componente:\s*([A-Za-z0-9]+)"
    )


    lat = buscar(
        r"Latitud:\s*(-?[0-9.]+)",
        float
    )


    lon = buscar(
        r"Longitud:\s*(-?[0-9.]+)",
        float
    )


    unidades = buscar(
        r"Unidades:\s*(.+)"
    )


    datos = np.loadtxt(
        path,
        comments="#"
    )

    datos = np.asarray(
        datos,
        dtype=float
    ).reshape(-1)


    return {
        "path":
            path,

        "inicio":
            tiempo,

        "fs":
            fs,

        "n_header":
            n_header,

        "n_real":
            len(datos),

        "estacion":
            estacion,

        "componente":
            componente,

        "lat":
            lat,

        "lon":
            lon,

        "unidades":
            unidades,

        "finite":
            np.isfinite(
                datos
            ).all()
    }


# ----------------------------------------------------------------------
# 2. RECORRER LAS 42 ESTACIONES
# ----------------------------------------------------------------------

resultados = []


carpetas = sorted(
    [
        x
        for x in os.listdir(REG_DIR)
        if os.path.isdir(
            os.path.join(
                REG_DIR,
                x
            )
        )
    ]
)


for codigo in carpetas:

    carpeta = os.path.join(
        REG_DIR,
        codigo
    )


    archivos = [
        os.path.join(
            carpeta,
            f
        )
        for f in os.listdir(
            carpeta
        )
        if f.lower().endswith(".txt")
    ]


    regs = []

    for path in archivos:

        r = leer_registro(
            path
        )

        regs.append(
            r
        )


    E = [
        r for r in regs
        if str(
            r["componente"]
        ).upper().endswith("E")
    ]

    N = [
        r for r in regs
        if str(
            r["componente"]
        ).upper().endswith("N")
    ]


    if (
        len(E) != 1
        or
        len(N) != 1
    ):

        resultados.append(
            {
                "Estacion":
                    codigo,

                "Estado":
                    "COMPONENTES_AMBIGUAS",

                "n_E_encontradas":
                    len(E),

                "n_N_encontradas":
                    len(N)
            }
        )

        continue


    E = E[0]
    N = N[0]


    fs_igual = np.isclose(
        E["fs"],
        N["fs"],
        atol=1e-12
    )


    lat_igual = np.isclose(
        E["lat"],
        N["lat"],
        atol=1e-8
    )


    lon_igual = np.isclose(
        E["lon"],
        N["lon"],
        atol=1e-8
    )


    inicio_diff_s = (
        N["inicio"]
        -
        E["inicio"]
    ).total_seconds()


    if fs_igual:

        dt = 1.0 / E["fs"]


        fin_E = (
            E["inicio"]
            +
            pd.to_timedelta(
                (E["n_real"] - 1)
                /
                E["fs"],
                unit="s"
            )
        )


        fin_N = (
            N["inicio"]
            +
            pd.to_timedelta(
                (N["n_real"] - 1)
                /
                N["fs"],
                unit="s"
            )
        )


        inicio_comun = max(
            E["inicio"],
            N["inicio"]
        )


        fin_comun = min(
            fin_E,
            fin_N
        )


        if fin_comun >= inicio_comun:

            duracion_comun_s = (
                fin_comun
                -
                inicio_comun
            ).total_seconds()


            n_comun_teorico = int(
                np.floor(
                    duracion_comun_s
                    *
                    E["fs"]
                    +
                    1e-9
                )
            ) + 1


            offset_E = (
                inicio_comun
                -
                E["inicio"]
            ).total_seconds() * E["fs"]


            offset_N = (
                inicio_comun
                -
                N["inicio"]
            ).total_seconds() * N["fs"]


            offsets_enteros = (
                np.isclose(
                    offset_E,
                    round(offset_E),
                    atol=1e-6
                )
                and
                np.isclose(
                    offset_N,
                    round(offset_N),
                    atol=1e-6
                )
            )

        else:

            duracion_comun_s = np.nan
            n_comun_teorico = 0
            offset_E = np.nan
            offset_N = np.nan
            offsets_enteros = False


    else:

        fin_E = pd.NaT
        fin_N = pd.NaT
        inicio_comun = pd.NaT
        fin_comun = pd.NaT

        duracion_comun_s = np.nan
        n_comun_teorico = 0

        offset_E = np.nan
        offset_N = np.nan

        offsets_enteros = False


    if not fs_igual:

        estado = "FS_DISTINTO"

    elif not (
        lat_igual
        and
        lon_igual
    ):

        estado = "COORDENADAS_DISTINTAS"

    elif not offsets_enteros:

        estado = "DESFASE_NO_ENTERO"

    elif n_comun_teorico <= 0:

        estado = "SIN_SOLAPE"

    elif (
        E["n_real"]
        ==
        N["n_real"]
        and
        np.isclose(
            inicio_diff_s,
            0,
            atol=1e-9
        )
    ):

        estado = "OK_IDENTICAS"

    else:

        estado = "ALINEABLE_POR_SOLAPE"


    resultados.append(
        {
            "Estacion":
                codigo,

            "Estado":
                estado,

            "Comp_E":
                E["componente"],

            "Comp_N":
                N["componente"],

            "Fs_E":
                E["fs"],

            "Fs_N":
                N["fs"],

            "Inicio_E":
                E["inicio"],

            "Inicio_N":
                N["inicio"],

            "Delta_inicio_N_menos_E_s":
                inicio_diff_s,

            "N_E":
                E["n_real"],

            "N_N":
                N["n_real"],

            "Delta_N_N_menos_E":
                N["n_real"]
                -
                E["n_real"],

            "Fin_E":
                fin_E,

            "Fin_N":
                fin_N,

            "N_intervalo_comun":
                n_comun_teorico,

            "Offset_E_muestras":
                offset_E,

            "Offset_N_muestras":
                offset_N,

            "Lat_E":
                E["lat"],

            "Lat_N":
                N["lat"],

            "Lon_E":
                E["lon"],

            "Lon_N":
                N["lon"],

            "Header_N_OK_E":
                E["n_header"]
                ==
                E["n_real"],

            "Header_N_OK_N":
                N["n_header"]
                ==
                N["n_real"],

            "Finite_E":
                E["finite"],

            "Finite_N":
                N["finite"]
        }
    )


audit = pd.DataFrame(
    resultados
)


# ----------------------------------------------------------------------
# 3. RESULTADOS
# ----------------------------------------------------------------------

print("=" * 80)
print("AUDITORÍA DE SINCRONIZACIÓN E/N")
print("=" * 80)

print(
    "Estaciones:",
    len(audit)
)


print("\nEstados:")

print(
    audit[
        "Estado"
    ].value_counts(
        dropna=False
    )
)


print("\nEstaciones NO idénticas:")

display(
    audit[
        audit[
            "Estado"
        ]
        !=
        "OK_IDENTICAS"
    ][
        [
            "Estacion",
            "Estado",
            "Fs_E",
            "Fs_N",
            "Delta_inicio_N_menos_E_s",
            "N_E",
            "N_N",
            "Delta_N_N_menos_E",
            "N_intervalo_comun",
            "Offset_E_muestras",
            "Offset_N_muestras"
        ]
    ]
)


print("\nAuditoría completa:")

display(
    audit
)


# ----------------------------------------------------------------------
# 4. GUARDAR
# ----------------------------------------------------------------------

RUTA = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_sincronizacion_EN.csv"
)

audit.to_csv(
    RUTA,
    index=False
)

print("\nGuardado:")
print(RUTA)


# ----------------------------------------------------------------------
# 5. GATE PRELIMINAR
# ----------------------------------------------------------------------

estados_aceptables = {
    "OK_IDENTICAS",
    "ALINEABLE_POR_SOLAPE"
}


gate = audit[
    "Estado"
].isin(
    estados_aceptables
).all()


print("\n" + "=" * 80)
print("GATE DE SINCRONIZACIÓN")
print("=" * 80)


if gate:

    print("✅ PASS")

    print(
        "Todas las estaciones son "
        "idénticas o pueden alinearse "
        "objetivamente por intervalo temporal común."
    )

else:

    print("❌ FAIL")

    print(
        "Existe al menos una estación "
        "que requiere revisión adicional."
    )


print("\nV5.1 NO FUE EJECUTADO.")


In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — CIERRE DEFINITIVO PRE-SA
#
# 1. Alineación E/N exclusivamente por índice entero de muestra.
# 2. Tolerancia de timestamp = 0.00011 s.
# 3. NO interpolación.
# 4. NO resampling.
# 5. Coordenadas precisas de headers CSN.
# 6. Rrup 3D a USGS FFM.geojson mediante ECEF.
# 7. Vs30 histórico global_vs30.grd mediante nearest.
#
# NO calcula Sa.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import hashlib
import numpy as np
import pandas as pd
import xarray as xr

from pyproj import Transformer


# ======================================================================
# 1. CONFIGURACIÓN CONGELADA
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

REG_DIR = os.path.join(
    CARPETA,
    "Registros_CSN"
)

FFM_PATH = os.path.join(
    CARPETA,
    "FFM.geojson"
)

VS30_PATH = os.path.join(
    BASE,
    "global_vs30.grd"
)

MW = 6.9

# Timestamps CSN presentan precisión de 0.0001 s.
# Al restar dos timestamps redondeados, la discrepancia
# máxima compatible con esa cuantización es ~0.0001 s.
# Se añade únicamente margen numérico:
TOL_TIEMPO_S = 0.00011


assert os.path.isdir(REG_DIR)
assert os.path.exists(FFM_PATH)
assert os.path.exists(VS30_PATH)


print("=" * 80)
print("VALPARAÍSO 2017 — CIERRE DEFINITIVO PRE-SA")
print("=" * 80)

print("Mw congelada          :", MW)
print("Tolerancia timestamp  :", TOL_TIEMPO_S, "s")
print("Interpolación          : NO")
print("Resampling             : NO")
print("Alineación             : índices enteros de muestra")


# ======================================================================
# 2. HASH
# ======================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            bloque = f.read(
                1024 * 1024
            )

            if not bloque:
                break

            h.update(
                bloque
            )

    return h.hexdigest()


print(
    "\nSHA256 FFM:",
    sha256_file(FFM_PATH)
)


# ======================================================================
# 3. LECTOR DE REGISTRO CSN
# ======================================================================

def leer_registro(path):

    lineas = []

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        for _ in range(30):

            linea = f.readline()

            if not linea:
                break

            if linea.startswith("#"):

                lineas.append(
                    linea.strip()
                )

            else:
                break


    texto = "\n".join(
        lineas
    )


    def buscar(
        patron,
        tipo=str
    ):

        m = re.search(
            patron,
            texto,
            flags=re.I
        )

        if not m:
            return None

        return tipo(
            m.group(1)
        )


    inicio_txt = buscar(
        r"Tiempo de Origen:\s*(.+)"
    )

    inicio = pd.to_datetime(
        inicio_txt,
        utc=True
    )


    fs = buscar(
        r"Tasa de muestreo:\s*"
        r"([0-9.]+)",
        float
    )


    n_header = buscar(
        r"Numero total de muestras:\s*"
        r"([0-9]+)",
        int
    )


    estacion = buscar(
        r"Estacion:\s*([A-Za-z0-9]+)"
    )


    componente = buscar(
        r"Componente:\s*([A-Za-z0-9]+)"
    )


    lat = buscar(
        r"Latitud:\s*(-?[0-9.]+)",
        float
    )


    lon = buscar(
        r"Longitud:\s*(-?[0-9.]+)",
        float
    )


    unidades = buscar(
        r"Unidades:\s*(.+)"
    )


    datos = np.loadtxt(
        path,
        comments="#"
    )

    datos = np.asarray(
        datos,
        dtype=float
    ).reshape(-1)


    assert np.isfinite(
        datos
    ).all(), path


    assert (
        len(datos)
        ==
        n_header
    ), (
        f"{path}: número de muestras "
        "header != archivo"
    )


    return {

        "path":
            path,

        "inicio":
            inicio,

        "fs":
            fs,

        "n":
            len(datos),

        "estacion":
            estacion,

        "componente":
            componente,

        "lat":
            lat,

        "lon":
            lon,

        "unidades":
            unidades
    }


# ======================================================================
# 4. AUDITAR Y FIJAR ALINEACIÓN E/N
# ======================================================================

filas = []


carpetas = sorted(
    [
        x
        for x in os.listdir(REG_DIR)
        if os.path.isdir(
            os.path.join(
                REG_DIR,
                x
            )
        )
    ]
)


assert len(carpetas) == 42


for codigo in carpetas:

    carpeta = os.path.join(
        REG_DIR,
        codigo
    )


    archivos = [
        os.path.join(
            carpeta,
            f
        )
        for f in os.listdir(
            carpeta
        )
        if f.lower().endswith(
            ".txt"
        )
    ]


    regs = [
        leer_registro(path)
        for path in archivos
    ]


    E = [
        r for r in regs
        if str(
            r["componente"]
        ).upper().endswith("E")
    ]

    N = [
        r for r in regs
        if str(
            r["componente"]
        ).upper().endswith("N")
    ]


    assert len(E) == 1, (
        f"{codigo}: E ambiguo"
    )

    assert len(N) == 1, (
        f"{codigo}: N ambiguo"
    )


    E = E[0]
    N = N[0]


    # --------------------------------------------------------------
    # Consistencia básica
    # --------------------------------------------------------------

    assert np.isclose(
        E["fs"],
        N["fs"],
        atol=1e-12
    ), f"{codigo}: Fs distinto"


    assert np.isclose(
        E["lat"],
        N["lat"],
        atol=1e-8
    ), f"{codigo}: latitud distinta"


    assert np.isclose(
        E["lon"],
        N["lon"],
        atol=1e-8
    ), f"{codigo}: longitud distinta"


    fs = E[
        "fs"
    ]


    # --------------------------------------------------------------
    # Desfase original N-E
    # --------------------------------------------------------------

    delta_s = (
        N["inicio"]
        -
        E["inicio"]
    ).total_seconds()


    delta_muestras = (
        delta_s
        *
        fs
    )


    delta_muestras_entero = int(
        np.round(
            delta_muestras
        )
    )


    # Residuo temporal al imponer índice entero
    residual_muestra = (
        delta_muestras
        -
        delta_muestras_entero
    )


    residual_tiempo_s = (
        residual_muestra
        /
        fs
    )


    # --------------------------------------------------------------
    # Clasificación
    # --------------------------------------------------------------

    if (
        E["inicio"] == N["inicio"]
        and
        E["n"] == N["n"]
    ):

        clase = "IDENTICAS"


    elif np.isclose(
        residual_tiempo_s,
        0.0,
        atol=1e-9
    ):

        clase = "OFFSET_ENTERO"


    elif (
        abs(
            residual_tiempo_s
        )
        <=
        TOL_TIEMPO_S
    ):

        clase = (
            "OFFSET_CUANTIZACION_HEADER"
        )


    else:

        clase = (
            "NO_ALINEABLE_SIN_INTERPOLAR"
        )


    # --------------------------------------------------------------
    # Gate crítico
    # --------------------------------------------------------------

    assert (
        clase
        !=
        "NO_ALINEABLE_SIN_INTERPOLAR"
    ), (
        f"{codigo}: residual temporal "
        f"{residual_tiempo_s:.9f} s"
    )


    # --------------------------------------------------------------
    # Inicio común mediante ÍNDICES ENTEROS
    # --------------------------------------------------------------

    inicio_comun = max(
        E["inicio"],
        N["inicio"]
    )


    offset_E_float = (
        (
            inicio_comun
            -
            E["inicio"]
        ).total_seconds()
        *
        fs
    )


    offset_N_float = (
        (
            inicio_comun
            -
            N["inicio"]
        ).total_seconds()
        *
        fs
    )


    iE = int(
        np.round(
            offset_E_float
        )
    )


    iN = int(
        np.round(
            offset_N_float
        )
    )


    # Error introducido al hacer snap
    error_E_s = (
        offset_E_float
        -
        iE
    ) / fs


    error_N_s = (
        offset_N_float
        -
        iN
    ) / fs


    assert (
        abs(error_E_s)
        <=
        TOL_TIEMPO_S
    )


    assert (
        abs(error_N_s)
        <=
        TOL_TIEMPO_S
    )


    # --------------------------------------------------------------
    # Número común de muestras
    # --------------------------------------------------------------

    n_restante_E = (
        E["n"]
        -
        iE
    )

    n_restante_N = (
        N["n"]
        -
        iN
    )


    n_comun = min(
        n_restante_E,
        n_restante_N
    )


    assert n_comun > 0


    duracion_comun_s = (
        n_comun
        /
        fs
    )


    filas.append(
        {

            "Estacion":
                codigo,

            "Latitud":
                E["lat"],

            "Longitud":
                E["lon"],

            "Fs_Hz":
                fs,

            "N_E_original":
                E["n"],

            "N_N_original":
                N["n"],

            "Delta_inicio_N_menos_E_s":
                delta_s,

            "Delta_inicio_muestras":
                delta_muestras,

            "Delta_muestras_entero":
                delta_muestras_entero,

            "Residual_timing_s":
                residual_tiempo_s,

            "Clasificacion_sync":
                clase,

            "Indice_inicio_E":
                iE,

            "Indice_inicio_N":
                iN,

            "N_comun":
                n_comun,

            "Duracion_comun_s":
                duracion_comun_s,

            "Archivo_E":
                E["path"],

            "Archivo_N":
                N["path"],

            "SHA256_E":
                sha256_file(
                    E["path"]
                ),

            "SHA256_N":
                sha256_file(
                    N["path"]
                ),

            "Unidad_E":
                E["unidades"],

            "Unidad_N":
                N["unidades"]
        }
    )


sync = pd.DataFrame(
    filas
)


# ======================================================================
# 5. RESULTADO SINCRONIZACIÓN
# ======================================================================

print("\n" + "=" * 80)
print("SINCRONIZACIÓN FINAL")
print("=" * 80)


print(
    sync[
        "Clasificacion_sync"
    ].value_counts()
)


print(
    "\nMáximo |residual temporal|:"
)

print(
    sync[
        "Residual_timing_s"
    ].abs().max(),
    "s"
)


display(
    sync[
        [
            "Estacion",
            "Fs_Hz",
            "Clasificacion_sync",
            "Delta_inicio_muestras",
            "Delta_muestras_entero",
            "Residual_timing_s",
            "Indice_inicio_E",
            "Indice_inicio_N",
            "N_comun",
            "Duracion_comun_s"
        ]
    ]
)


assert len(sync) == 42


assert (
    sync[
        "Residual_timing_s"
    ].abs().max()
    <=
    TOL_TIEMPO_S
)


print(
    "\n✅ 42/42 estaciones "
    "alineables SIN interpolación."
)


# ======================================================================
# 6. GUARDAR SINCRONIZACIÓN ANTES DE Rrup
# ======================================================================

RUTA_SYNC = os.path.join(
    CARPETA,
    "Valparaiso2017_alineacion_EN_FINAL.csv"
)


sync.to_csv(
    RUTA_SYNC,
    index=False
)


print(
    "\nAuditoría sincronización:"
)

print(
    RUTA_SYNC
)


# ======================================================================
# 7. CARGAR FFM GEOJSON
# ======================================================================

with open(
    FFM_PATH,
    "r",
    encoding="utf-8"
) as f:

    ffm = json.load(
        f
    )


features = ffm.get(
    "features",
    []
)


poligonos = []


for feature in features:

    geom = feature.get(
        "geometry",
        {}
    )

    tipo = geom.get(
        "type"
    )


    if tipo == "Polygon":

        coords = geom.get(
            "coordinates",
            []
        )

        if len(coords):

            poligonos.append(
                coords[0]
            )


    elif tipo == "MultiPolygon":

        for poly in geom.get(
            "coordinates",
            []
        ):

            if len(poly):

                poligonos.append(
                    poly[0]
                )


print("\n" + "=" * 80)
print("FINITE-FAULT USGS")
print("=" * 80)

print(
    "Polígonos/subfallas:",
    len(poligonos)
)


# USGS reportó Nx = 21, Nz = 21
assert len(poligonos) == 441, (
    f"Esperábamos 441 subfallas; "
    f"se encontraron {len(poligonos)}"
)


# ======================================================================
# 8. PROFUNDIDAD FFM
# ======================================================================

z_all = []


for ring in poligonos:

    for p in ring:

        if len(p) >= 3:

            z_all.append(
                p[2]
            )


z_all = np.asarray(
    z_all,
    dtype=float
)


print(
    "Z raw min/max:",
    z_all.min(),
    z_all.max()
)


# Este FFM almacena profundidad en metros.
assert np.median(
    np.abs(
        z_all
    )
) > 1000


print(
    "Interpretación Z: "
    "metros de profundidad."
)


# ======================================================================
# 9. COORDENADAS GEOCÉNTRICAS ECEF
# ======================================================================
#
# Evitamos aproximaciones planas.
#
# WGS84 geográfico 3D:
# EPSG:4979
#
# WGS84 geocéntrico:
# EPSG:4978
#
# Falla:
# h = -profundidad
#
# Estación:
# h = 0
# ======================================================================

to_ecef = Transformer.from_crs(
    "EPSG:4979",
    "EPSG:4978",
    always_xy=True
)


# ======================================================================
# 10. CREAR TRIÁNGULOS 3D DE SUBFALLAS
# ======================================================================

triangulos = []


for ring in poligonos:

    arr = np.asarray(
        ring,
        dtype=float
    )


    # eliminar cierre duplicado
    if np.allclose(
        arr[0],
        arr[-1]
    ):

        arr = arr[:-1]


    assert len(arr) >= 3


    lon = arr[:, 0]
    lat = arr[:, 1]

    # profundidad positiva hacia abajo
    profundidad_m = arr[:, 2]

    h = (
        -profundidad_m
    )


    X, Y, Z = to_ecef.transform(
        lon,
        lat,
        h
    )


    xyz = np.column_stack(
        [
            X,
            Y,
            Z
        ]
    )


    # fan triangulation
    for j in range(
        1,
        len(xyz) - 1
    ):

        triangulos.append(
            (
                xyz[0],
                xyz[j],
                xyz[j + 1]
            )
        )


print(
    "Triángulos 3D:",
    len(triangulos)
)


# Cuadriláteros -> 2 triángulos
assert len(triangulos) == 882


# ======================================================================
# 11. DISTANCIA PUNTO - TRIÁNGULO
# ======================================================================

def distancia_punto_triangulo(
    p,
    a,
    b,
    c
):

    ab = b - a
    ac = c - a
    ap = p - a

    d1 = np.dot(
        ab,
        ap
    )

    d2 = np.dot(
        ac,
        ap
    )


    if (
        d1 <= 0
        and
        d2 <= 0
    ):

        return np.linalg.norm(
            p - a
        )


    bp = p - b

    d3 = np.dot(
        ab,
        bp
    )

    d4 = np.dot(
        ac,
        bp
    )


    if (
        d3 >= 0
        and
        d4 <= d3
    ):

        return np.linalg.norm(
            p - b
        )


    vc = (
        d1 * d4
        -
        d3 * d2
    )


    if (
        vc <= 0
        and
        d1 >= 0
        and
        d3 <= 0
    ):

        v = (
            d1
            /
            (d1 - d3)
        )

        proj = (
            a
            +
            v * ab
        )

        return np.linalg.norm(
            p - proj
        )


    cp = p - c

    d5 = np.dot(
        ab,
        cp
    )

    d6 = np.dot(
        ac,
        cp
    )


    if (
        d6 >= 0
        and
        d5 <= d6
    ):

        return np.linalg.norm(
            p - c
        )


    vb = (
        d5 * d2
        -
        d1 * d6
    )


    if (
        vb <= 0
        and
        d2 >= 0
        and
        d6 <= 0
    ):

        w = (
            d2
            /
            (d2 - d6)
        )

        proj = (
            a
            +
            w * ac
        )

        return np.linalg.norm(
            p - proj
        )


    va = (
        d3 * d6
        -
        d5 * d4
    )


    if (
        va <= 0
        and
        (d4 - d3) >= 0
        and
        (d5 - d6) >= 0
    ):

        bc = c - b

        w = (
            (d4 - d3)
            /
            (
                (d4 - d3)
                +
                (d5 - d6)
            )
        )

        proj = (
            b
            +
            w * bc
        )

        return np.linalg.norm(
            p - proj
        )


    denom = (
        va
        +
        vb
        +
        vc
    )


    assert denom != 0


    v = vb / denom
    w = vc / denom


    proj = (
        a
        +
        ab * v
        +
        ac * w
    )


    return np.linalg.norm(
        p - proj
    )


# ======================================================================
# 12. CALCULAR Rrup
# ======================================================================

rrups = []


for _, row in sync.iterrows():

    lon = row[
        "Longitud"
    ]

    lat = row[
        "Latitud"
    ]


    Xs, Ys, Zs = to_ecef.transform(
        lon,
        lat,
        0.0
    )


    p = np.array(
        [
            Xs,
            Ys,
            Zs
        ],
        dtype=float
    )


    dmin_m = np.inf


    for a, b, c in triangulos:

        d = distancia_punto_triangulo(
            p,
            a,
            b,
            c
        )

        if d < dmin_m:

            dmin_m = d


    rrups.append(
        dmin_m
        /
        1000.0
    )


sync[
    "Rrup_km"
] = rrups


assert np.isfinite(
    sync["Rrup_km"]
).all()


assert (
    sync["Rrup_km"]
    >
    0
).all()


print("\n" + "=" * 80)
print("Rrup — USGS FINITE-FAULT")
print("=" * 80)


display(
    sync[
        [
            "Estacion",
            "Latitud",
            "Longitud",
            "Rrup_km"
        ]
    ]
    .sort_values(
        "Rrup_km"
    )
    .round(4)
)


print(
    "\nRrup mínimo:",
    sync[
        "Rrup_km"
    ].min()
)

print(
    "Rrup máximo:",
    sync[
        "Rrup_km"
    ].max()
)


# ======================================================================
# 13. Vs30 HISTÓRICO — NEAREST
# ======================================================================

print("\n" + "=" * 80)
print("Vs30 — GLOBAL MOSAIC / NEAREST")
print("=" * 80)


ds = xr.open_dataset(
    VS30_PATH
)


if "z" in ds.data_vars:

    da = ds["z"]

else:

    assert len(
        ds.data_vars
    ) == 1

    da = ds[
        list(
            ds.data_vars
        )[0]
    ]


vs30 = []
raster_lat = []
raster_lon = []


for _, row in sync.iterrows():

    sel = da.sel(
        lat=row["Latitud"],
        lon=row["Longitud"],
        method="nearest"
    )


    valor = float(
        np.asarray(
            sel.values
        ).squeeze()
    )


    lat_r = float(
        np.asarray(
            sel["lat"].values
        ).squeeze()
    )


    lon_r = float(
        np.asarray(
            sel["lon"].values
        ).squeeze()
    )


    vs30.append(
        valor
    )

    raster_lat.append(
        lat_r
    )

    raster_lon.append(
        lon_r
    )


sync[
    "Vs30_m_s"
] = vs30

sync[
    "Vs30_raster_lat"
] = raster_lat

sync[
    "Vs30_raster_lon"
] = raster_lon


assert np.isfinite(
    sync[
        "Vs30_m_s"
    ]
).all()


assert (
    sync[
        "Vs30_m_s"
    ]
    >
    0
).all()


print(
    "N       :",
    len(sync)
)

print(
    "Mínimo  :",
    sync[
        "Vs30_m_s"
    ].min()
)

print(
    "Mediana :",
    sync[
        "Vs30_m_s"
    ].median()
)

print(
    "Media   :",
    sync[
        "Vs30_m_s"
    ].mean()
)

print(
    "Máximo  :",
    sync[
        "Vs30_m_s"
    ].max()
)


# ======================================================================
# 14. MATRIZ CONGELADA PRE-SA
# ======================================================================

matriz = sync[
    [
        "Estacion",
        "Latitud",
        "Longitud",
        "Rrup_km",
        "Vs30_m_s",
        "Fs_Hz",
        "Indice_inicio_E",
        "Indice_inicio_N",
        "N_comun",
        "Duracion_comun_s",
        "Residual_timing_s",
        "Clasificacion_sync",
        "Archivo_E",
        "Archivo_N",
        "SHA256_E",
        "SHA256_N"
    ]
].copy()


matriz.insert(
    3,
    "Mw",
    MW
)


matriz = matriz.sort_values(
    "Rrup_km"
).reset_index(
    drop=True
)


print("\n" + "=" * 80)
print("MATRIZ CONGELADA PRE-SA")
print("=" * 80)


display(
    matriz[
        [
            "Estacion",
            "Latitud",
            "Longitud",
            "Mw",
            "Rrup_km",
            "Vs30_m_s",
            "Fs_Hz",
            "N_comun",
            "Duracion_comun_s",
            "Residual_timing_s",
            "Clasificacion_sync"
        ]
    ].round(
        {
            "Latitud":
                5,

            "Longitud":
                5,

            "Rrup_km":
                3,

            "Vs30_m_s":
                3,

            "Duracion_comun_s":
                3,

            "Residual_timing_s":
                7
        }
    )
)


# ======================================================================
# 15. GUARDAR
# ======================================================================

RUTA_FINAL_SYNC = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_sync_Rrup_Vs30.csv"
)

RUTA_PRE_SA = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_PRE_Sa.csv"
)


sync.to_csv(
    RUTA_FINAL_SYNC,
    index=False
)

matriz.to_csv(
    RUTA_PRE_SA,
    index=False
)


hash_pre_sa = sha256_file(
    RUTA_PRE_SA
)


# ======================================================================
# 16. GATE FINAL
# ======================================================================

gate = (
    len(matriz) == 42
    and
    np.isfinite(
        matriz[
            [
                "Latitud",
                "Longitud",
                "Rrup_km",
                "Vs30_m_s"
            ]
        ].to_numpy()
    ).all()
    and
    (
        matriz[
            "Residual_timing_s"
        ].abs()
        <=
        TOL_TIEMPO_S
    ).all()
)


print("\n" + "=" * 80)
print("GATE PRE-SA DEFINITIVO")
print("=" * 80)


if gate:

    print("✅ PASS")

    print(
        "42/42 estaciones conservadas."
    )

    print(
        "No se interpoló ninguna señal."
    )

    print(
        "No se hizo resampling."
    )

    print(
        "Rrup congelado."
    )

    print(
        "Vs30 congelado."
    )

    print(
        "\nSiguiente etapa:"
    )

    print(
        "RotD50(T=1.0 s, 5%)."
    )

else:

    print("❌ FAIL")


print("\nArchivos:")

print(
    RUTA_FINAL_SYNC
)

print(
    RUTA_PRE_SA
)

print(
    "\nSHA256 PRE-Sa:"
)

print(
    hash_pre_sa
)

print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — CORRECCIÓN AUDITADA DE COORDENADAS M13L
#
# PRE-SA V2
#
# MOTIVO:
# El header de los acelerogramas M13L contiene coordenadas incompatibles
# con la estación M13L / Municipalidad de Parral.
#
# Fuente externa congelada ANTES de ejecutar V5.1:
# Leyton et al. (2018), Seismological Research Letters
# DOI: 10.1785/0220170156
#
# M13L:
# Lat = -36.1411
# Lon = -71.8242
#
# IMPORTANTE:
# - NO modifica señales.
# - NO modifica Mw.
# - NO modifica ninguna otra estación.
# - NO usa Vs30=309 del artículo como predictor.
# - Vs30 se vuelve a extraer del MISMO global_vs30.grd / nearest.
# - Rrup se vuelve a calcular con el MISMO FFM USGS.
# - NO calcula Sa.
# - NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import hashlib
import numpy as np
import pandas as pd
import xarray as xr

from pyproj import Transformer


# ======================================================================
# 1. RUTAS
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

RUTA_V1 = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_PRE_Sa.csv"
)

META_DIR = os.path.join(
    CARPETA,
    "Metadata_estaciones_CSN"
)

FFM_PATH = os.path.join(
    CARPETA,
    "FFM.geojson"
)

VS30_PATH = os.path.join(
    BASE,
    "global_vs30.grd"
)

assert os.path.exists(RUTA_V1)
assert os.path.exists(FFM_PATH)
assert os.path.exists(VS30_PATH)
assert os.path.isdir(META_DIR)


# Coordenada M13L congelada
M13L_LAT = -36.1411
M13L_LON = -71.8242


print("=" * 80)
print("VALPARAÍSO 2017 — AUDITORÍA DE COORDENADAS / PRE-SA V2")
print("=" * 80)


# ======================================================================
# 2. HASH
# ======================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            bloque = f.read(
                1024 * 1024
            )

            if not bloque:
                break

            h.update(bloque)

    return h.hexdigest()


print(
    "\nSHA256 PRE-Sa V1:",
    sha256_file(RUTA_V1)
)


# ======================================================================
# 3. CARGAR MATRIZ V1
# ======================================================================

df = pd.read_csv(
    RUTA_V1
)

assert len(df) == 42
assert df["Estacion"].nunique() == 42


print("\nM13L EN PRE-SA V1:")

display(
    df.loc[
        df["Estacion"] == "M13L",
        [
            "Estacion",
            "Latitud",
            "Longitud",
            "Mw",
            "Rrup_km",
            "Vs30_m_s"
        ]
    ]
)


# ======================================================================
# 4. AUDITAR HEADERS CONTRA FICHAS CSN
# ======================================================================
#
# Las páginas CSN muestran lat/lon redondeadas aproximadamente a 0.1°.
# Por ello NO se usan para Rrup.
#
# Se usan aquí únicamente como gate para detectar errores gruesos de
# metadata en los headers.
# ======================================================================

def extraer_coordenadas_html(path):

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        texto = f.read()


    mlat = re.search(
        r"<strong>Latitud:</strong>\s*"
        r"(-?[0-9.]+)",
        texto,
        re.I
    )

    mlon = re.search(
        r"<strong>Longitud:</strong>\s*"
        r"(-?[0-9.]+)",
        texto,
        re.I
    )


    if (
        mlat is None
        or
        mlon is None
    ):

        return np.nan, np.nan


    return (
        float(mlat.group(1)),
        float(mlon.group(1))
    )


audit = []


for _, row in df.iterrows():

    codigo = row["Estacion"]

    html = os.path.join(
        META_DIR,
        f"{codigo}.html"
    )

    assert os.path.exists(
        html
    ), html


    lat_page, lon_page = (
        extraer_coordenadas_html(
            html
        )
    )


    dlat = (
        row["Latitud"]
        -
        lat_page
    )

    dlon = (
        row["Longitud"]
        -
        lon_page
    )


    # Una página redondeada a 0.1°
    # debería diferir como máximo ~0.05°.
    discrepancia = (
        abs(dlat) > 0.055
        or
        abs(dlon) > 0.055
    )


    audit.append(
        {
            "Estacion":
                codigo,

            "Lat_header":
                row["Latitud"],

            "Lon_header":
                row["Longitud"],

            "Lat_pagina_CSN":
                lat_page,

            "Lon_pagina_CSN":
                lon_page,

            "Delta_lat":
                dlat,

            "Delta_lon":
                dlon,

            "Discrepancia_gruesa":
                discrepancia
        }
    )


audit = pd.DataFrame(
    audit
)


print("\n" + "=" * 80)
print("AUDITORÍA HEADER VS FICHA CSN")
print("=" * 80)


print(
    "Discrepancias gruesas:",
    int(
        audit[
            "Discrepancia_gruesa"
        ].sum()
    )
)


display(
    audit.loc[
        audit[
            "Discrepancia_gruesa"
        ]
    ]
)


# Esperamos que M13L sea la única
discrepantes = set(
    audit.loc[
        audit[
            "Discrepancia_gruesa"
        ],
        "Estacion"
    ]
)


assert discrepantes == {
    "M13L"
}, (
    "Aparecieron discrepancias adicionales: "
    f"{discrepantes}"
)


print(
    "\n✅ M13L es la única discrepancia "
    "gruesa detectada."
)


# ======================================================================
# 5. CORREGIR SOLO COORDENADAS DE M13L
# ======================================================================

df_v2 = df.copy()


idx = df_v2.index[
    df_v2[
        "Estacion"
    ] == "M13L"
]


assert len(idx) == 1

i = idx[0]


lat_old = df_v2.loc[
    i,
    "Latitud"
]

lon_old = df_v2.loc[
    i,
    "Longitud"
]


df_v2.loc[
    i,
    "Latitud"
] = M13L_LAT

df_v2.loc[
    i,
    "Longitud"
] = M13L_LON


print("\n" + "=" * 80)
print("CORRECCIÓN M13L")
print("=" * 80)

print(
    "Anterior:",
    lat_old,
    lon_old
)

print(
    "Corregida:",
    M13L_LAT,
    M13L_LON
)


# ======================================================================
# 6. CARGAR FFM
# ======================================================================

with open(
    FFM_PATH,
    "r",
    encoding="utf-8"
) as f:

    ffm = json.load(f)


features = ffm[
    "features"
]


poligonos = []


for feature in features:

    geom = feature.get(
        "geometry",
        {}
    )

    tipo = geom.get(
        "type"
    )


    if tipo == "Polygon":

        poligonos.append(
            geom[
                "coordinates"
            ][0]
        )


    elif tipo == "MultiPolygon":

        for poly in geom[
            "coordinates"
        ]:

            poligonos.append(
                poly[0]
            )


assert len(poligonos) == 441


# ======================================================================
# 7. TRANSFORMACIÓN ECEF
# ======================================================================

to_ecef = Transformer.from_crs(
    "EPSG:4979",
    "EPSG:4978",
    always_xy=True
)


triangulos = []


for ring in poligonos:

    arr = np.asarray(
        ring,
        dtype=float
    )


    if np.allclose(
        arr[0],
        arr[-1]
    ):

        arr = arr[:-1]


    lon = arr[:, 0]
    lat = arr[:, 1]

    profundidad_m = arr[:, 2]

    h = -profundidad_m


    X, Y, Z = to_ecef.transform(
        lon,
        lat,
        h
    )


    xyz = np.column_stack(
        [
            X,
            Y,
            Z
        ]
    )


    for j in range(
        1,
        len(xyz) - 1
    ):

        triangulos.append(
            (
                xyz[0],
                xyz[j],
                xyz[j + 1]
            )
        )


assert len(triangulos) == 882


# ======================================================================
# 8. DISTANCIA PUNTO-TRIÁNGULO
# ======================================================================

def distancia_punto_triangulo(
    p,
    a,
    b,
    c
):

    ab = b - a
    ac = c - a
    ap = p - a

    d1 = np.dot(ab, ap)
    d2 = np.dot(ac, ap)


    if d1 <= 0 and d2 <= 0:

        return np.linalg.norm(
            p - a
        )


    bp = p - b

    d3 = np.dot(ab, bp)
    d4 = np.dot(ac, bp)


    if d3 >= 0 and d4 <= d3:

        return np.linalg.norm(
            p - b
        )


    vc = d1*d4 - d3*d2


    if (
        vc <= 0
        and d1 >= 0
        and d3 <= 0
    ):

        v = d1 / (
            d1 - d3
        )

        proj = (
            a
            +
            v * ab
        )

        return np.linalg.norm(
            p - proj
        )


    cp = p - c

    d5 = np.dot(ab, cp)
    d6 = np.dot(ac, cp)


    if d6 >= 0 and d5 <= d6:

        return np.linalg.norm(
            p - c
        )


    vb = d5*d2 - d1*d6


    if (
        vb <= 0
        and d2 >= 0
        and d6 <= 0
    ):

        w = d2 / (
            d2 - d6
        )

        proj = (
            a
            +
            w * ac
        )

        return np.linalg.norm(
            p - proj
        )


    va = d3*d6 - d5*d4


    if (
        va <= 0
        and
        (d4 - d3) >= 0
        and
        (d5 - d6) >= 0
    ):

        bc = c - b

        w = (
            (d4 - d3)
            /
            (
                (d4 - d3)
                +
                (d5 - d6)
            )
        )

        proj = (
            b
            +
            w * bc
        )

        return np.linalg.norm(
            p - proj
        )


    denom = (
        va
        +
        vb
        +
        vc
    )


    v = vb / denom
    w = vc / denom


    proj = (
        a
        +
        ab*v
        +
        ac*w
    )


    return np.linalg.norm(
        p - proj
    )


# ======================================================================
# 9. RECALCULAR Rrup SOLO PARA M13L
# ======================================================================

Xs, Ys, Zs = to_ecef.transform(
    M13L_LON,
    M13L_LAT,
    0.0
)


p = np.array(
    [
        Xs,
        Ys,
        Zs
    ],
    dtype=float
)


dmin_m = np.inf


for a, b, c in triangulos:

    d = distancia_punto_triangulo(
        p,
        a,
        b,
        c
    )

    if d < dmin_m:

        dmin_m = d


rrup_new = (
    dmin_m
    /
    1000.0
)


assert np.isfinite(
    rrup_new
)

assert rrup_new > 0


# ======================================================================
# 10. RECALCULAR Vs30 SOLO PARA M13L
# ======================================================================

ds = xr.open_dataset(
    VS30_PATH
)


da = (
    ds["z"]
    if "z" in ds.data_vars
    else
    ds[
        list(
            ds.data_vars
        )[0]
    ]
)


sel = da.sel(
    lat=M13L_LAT,
    lon=M13L_LON,
    method="nearest"
)


vs30_new = float(
    np.asarray(
        sel.values
    ).squeeze()
)


assert np.isfinite(
    vs30_new
)

assert vs30_new > 0


rrup_old = df_v2.loc[
    i,
    "Rrup_km"
]

vs30_old = df_v2.loc[
    i,
    "Vs30_m_s"
]


df_v2.loc[
    i,
    "Rrup_km"
] = rrup_new

df_v2.loc[
    i,
    "Vs30_m_s"
] = vs30_new


# ======================================================================
# 11. DOCUMENTAR FUENTE DE COORDENADAS
# ======================================================================

df_v2[
    "Fuente_coordenadas"
] = (
    "Header_CSN"
)


df_v2.loc[
    i,
    "Fuente_coordenadas"
] = (
    "Leyton_et_al_2018_supplement_"
    "DOI_10.1785_0220170156"
)


df_v2[
    "Correccion_metadata"
] = False


df_v2.loc[
    i,
    "Correccion_metadata"
] = True


# ======================================================================
# 12. COMPARACIÓN M13L V1 VS V2
# ======================================================================

print("\n" + "=" * 80)
print("IMPACTO DE LA CORRECCIÓN — M13L")
print("=" * 80)


comparacion = pd.DataFrame(
    {
        "Variable": [
            "Latitud",
            "Longitud",
            "Rrup_km",
            "Vs30_m_s"
        ],

        "PRE_Sa_V1": [
            lat_old,
            lon_old,
            rrup_old,
            vs30_old
        ],

        "PRE_Sa_V2": [
            M13L_LAT,
            M13L_LON,
            rrup_new,
            vs30_new
        ]
    }
)


display(
    comparacion
)


# ======================================================================
# 13. VERIFICAR QUE SOLO M13L CAMBIÓ
# ======================================================================

otras = (
    df["Estacion"]
    !=
    "M13L"
)


for columna in [
    "Latitud",
    "Longitud",
    "Mw",
    "Rrup_km",
    "Vs30_m_s"
]:

    assert np.allclose(
        df.loc[
            otras,
            columna
        ],
        df_v2.loc[
            otras,
            columna
        ],
        rtol=0,
        atol=0,
        equal_nan=True
    ), (
        f"Cambió indebidamente "
        f"la columna {columna}"
    )


print(
    "\n✅ Las otras 41 estaciones "
    "permanecen exactamente iguales."
)


# ======================================================================
# 14. ORDENAR DE NUEVO POR Rrup
# ======================================================================

df_v2 = (
    df_v2
    .sort_values(
        "Rrup_km"
    )
    .reset_index(
        drop=True
    )
)


# ======================================================================
# 15. GUARDAR
# ======================================================================

RUTA_AUDIT = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_coordenadas_header_vs_CSN.csv"
)

RUTA_V2 = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_PRE_Sa_V2_COORD_CORREGIDA.csv"
)


audit.to_csv(
    RUTA_AUDIT,
    index=False
)


df_v2.to_csv(
    RUTA_V2,
    index=False
)


print("\n" + "=" * 80)
print("PRE-SA V2 — MATRIZ CORREGIDA")
print("=" * 80)


display(
    df_v2[
        [
            "Estacion",
            "Latitud",
            "Longitud",
            "Mw",
            "Rrup_km",
            "Vs30_m_s",
            "Fuente_coordenadas",
            "Correccion_metadata"
        ]
    ].round(
        {
            "Latitud": 5,
            "Longitud": 5,
            "Rrup_km": 3,
            "Vs30_m_s": 3
        }
    )
)


print("\nSHA256 PRE-Sa V2:")

print(
    sha256_file(
        RUTA_V2
    )
)


# ======================================================================
# 16. GATE
# ======================================================================

gate = (
    len(df_v2) == 42
    and
    df_v2[
        "Estacion"
    ].nunique() == 42
    and
    df_v2[
        [
            "Rrup_km",
            "Vs30_m_s"
        ]
    ].notna().all().all()
    and
    (
        df_v2[
            "Rrup_km"
        ] > 0
    ).all()
    and
    (
        df_v2[
            "Vs30_m_s"
        ] > 0
    ).all()
    and
    int(
        df_v2[
            "Correccion_metadata"
        ].sum()
    ) == 1
)


print("\n" + "=" * 80)
print("GATE PRE-SA V2")
print("=" * 80)


if gate:

    print("✅ PASS")

    print(
        "42 estaciones conservadas."
    )

    print(
        "Solo M13L fue corregida."
    )

    print(
        "Rrup de M13L recalculado."
    )

    print(
        "Vs30 de M13L reextraído "
        "del raster histórico nearest."
    )

    print(
        "V5.1 NO fue ejecutado."
    )

    print(
        "\nAhora sí:"
    )

    print(
        "siguiente etapa = "
        "RotD50(T=1.0 s, 5%)."
    )

else:

    print("❌ FAIL")


print("\nArchivos:")
print(RUTA_AUDIT)
print(RUTA_V2)


In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — CÁLCULO AUDITADO DE RotD50
# T = 1.0 s
# amortiguamiento = 5 %
#
# INPUT CONGELADO:
# PRE-SA V2
# SHA256 esperado:
# 3284ed662d37dde88ab83c88dacad8ef82976c6057f9aaf7eb3e643755c436ba
#
# PROCEDIMIENTO PRINCIPAL:
# - componentes E/N alineadas mediante índices enteros ya congelados;
# - remover media de cada componente;
# - SIN filtro;
# - SIN taper;
# - SIN resampling;
# - SIN interpolación entre componentes;
# - Newmark promedio de aceleración:
#       beta = 1/4
#       gamma = 1/2
# - PSA(T=1.0 s, ξ=5 %)
# - rotaciones 0° ... 179°, paso 1°
# - RotD50 = mediana de PSA sobre orientaciones
#
# AUDITORÍAS:
# 1. señal raw;
# 2. detrend lineal;
# 3. resolución angular 0.5°;
# 4. integración con dt/2 mediante interpolación lineal SOLO
#    como prueba numérica de convergencia;
# 5. control de respuesta máxima cerca de extremos del registro.
#
# IMPORTANTE:
# - NO ejecuta V5.1.
# - NO elimina estaciones.
# - NO modifica Mw, Rrup o Vs30.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import hashlib
import numpy as np
import pandas as pd

from scipy.signal import detrend


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

RUTA_PRE_SA = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_PRE_Sa_V2_COORD_CORREGIDA.csv"
)

HASH_ESPERADO = (
    "3284ed662d37dde88ab83c88dacad8ef"
    "82976c6057f9aaf7eb3e643755c436ba"
)

T = 1.0
XI = 0.05

G = 9.80665

BETA = 0.25
GAMMA = 0.50

ANGULOS_1 = np.arange(
    0.0,
    180.0,
    1.0
)

ANGULOS_05 = np.arange(
    0.0,
    180.0,
    0.5
)


# ======================================================================
# 2. HASH
# ======================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            b = f.read(
                1024 * 1024
            )

            if not b:
                break

            h.update(b)

    return h.hexdigest()


assert os.path.exists(
    RUTA_PRE_SA
)

hash_actual = sha256_file(
    RUTA_PRE_SA
)


print("=" * 80)
print("VALPARAÍSO 2017 — RotD50(T=1.0 s, 5 %)")
print("=" * 80)

print(
    "SHA256 PRE-Sa V2:",
    hash_actual
)


assert (
    hash_actual
    ==
    HASH_ESPERADO
), (
    "STOP: el archivo PRE-Sa V2 "
    "ya no coincide con la versión congelada."
)


print(
    "✓ PRE-Sa V2 verificado."
)


# ======================================================================
# 3. CARGAR MATRIZ CONGELADA
# ======================================================================

df = pd.read_csv(
    RUTA_PRE_SA
)

assert len(df) == 42
assert df["Estacion"].nunique() == 42


columnas_necesarias = [
    "Estacion",
    "Latitud",
    "Longitud",
    "Mw",
    "Rrup_km",
    "Vs30_m_s",
    "Fs_Hz",
    "Indice_inicio_E",
    "Indice_inicio_N",
    "N_comun",
    "Duracion_comun_s",
    "Archivo_E",
    "Archivo_N"
]


for c in columnas_necesarias:

    assert c in df.columns, c


# ======================================================================
# 4. CARGA DE ACELEROGRAMAS
# ======================================================================

def cargar_aceleracion(path):

    assert os.path.exists(
        path
    ), path

    x = np.loadtxt(
        path,
        comments="#"
    )

    x = np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)

    assert len(x) > 0
    assert np.isfinite(x).all()

    return x


def segmento_alineado(row):

    E = cargar_aceleracion(
        row["Archivo_E"]
    )

    N = cargar_aceleracion(
        row["Archivo_N"]
    )

    iE = int(
        row["Indice_inicio_E"]
    )

    iN = int(
        row["Indice_inicio_N"]
    )

    n = int(
        row["N_comun"]
    )


    E = E[
        iE:
        iE + n
    ]

    N = N[
        iN:
        iN + n
    ]


    assert len(E) == n
    assert len(N) == n


    return E, N


# ======================================================================
# 5. NEWMARK — RESPUESTA DE DESPLAZAMIENTO RELATIVO
# ======================================================================
#
# Ecuación:
#
# u'' + 2 ξ ω u' + ω² u = -ag(t)
#
# Newmark promedio de aceleración:
# beta  = 1/4
# gamma = 1/2
#
# PSA = ω² * max(|u|)
# ======================================================================

def newmark_u(
    ag,
    dt,
    T=1.0,
    xi=0.05
):

    ag = np.asarray(
        ag,
        dtype=np.float64
    )


    omega = (
        2.0
        *
        np.pi
        /
        T
    )

    k = omega**2

    c = (
        2.0
        *
        xi
        *
        omega
    )


    n = len(
        ag
    )


    u = np.zeros(
        n,
        dtype=np.float64
    )

    v = np.zeros(
        n,
        dtype=np.float64
    )

    ar = np.zeros(
        n,
        dtype=np.float64
    )


    # condición inicial
    ar[0] = (
        -ag[0]
        -
        c * v[0]
        -
        k * u[0]
    )


    a0 = (
        1.0
        /
        (
            BETA
            *
            dt**2
        )
    )

    a1 = (
        GAMMA
        /
        (
            BETA
            *
            dt
        )
    )

    a2 = (
        1.0
        /
        (
            BETA
            *
            dt
        )
    )

    a3 = (
        1.0
        /
        (
            2.0
            *
            BETA
        )
        -
        1.0
    )

    a4 = (
        GAMMA
        /
        BETA
        -
        1.0
    )

    a5 = (
        dt
        *
        0.5
        *
        (
            GAMMA
            /
            BETA
            -
            2.0
        )
    )


    k_hat = (
        k
        +
        a0
        +
        c * a1
    )


    for i in range(
        n - 1
    ):

        p_next = (
            -ag[
                i + 1
            ]
        )


        p_eff = (
            p_next
            +
            a0 * u[i]
            +
            a2 * v[i]
            +
            a3 * ar[i]
            +
            c
            *
            (
                a1 * u[i]
                +
                a4 * v[i]
                +
                a5 * ar[i]
            )
        )


        u_next = (
            p_eff
            /
            k_hat
        )


        ar_next = (
            a0
            *
            (
                u_next
                -
                u[i]
            )
            -
            a2 * v[i]
            -
            a3 * ar[i]
        )


        v_next = (
            v[i]
            +
            dt
            *
            (
                (
                    1.0
                    -
                    GAMMA
                )
                *
                ar[i]
                +
                GAMMA
                *
                ar_next
            )
        )


        u[
            i + 1
        ] = u_next

        v[
            i + 1
        ] = v_next

        ar[
            i + 1
        ] = ar_next


    return u


# ======================================================================
# 6. RotD A PARTIR DE RESPUESTAS E/N
# ======================================================================
#
# Por linealidad del oscilador:
#
# u(theta) =
# uE cos(theta) + uN sin(theta)
#
# Por ello NO necesitamos integrar 180 veces.
# ======================================================================

def rotd_desde_respuestas(
    uE,
    uN,
    T,
    angulos_deg
):

    omega = (
        2.0
        *
        np.pi
        /
        T
    )


    factores = []


    for ang in angulos_deg:

        rad = np.deg2rad(
            ang
        )

        ur = (
            uE
            *
            np.cos(rad)
            +
            uN
            *
            np.sin(rad)
        )


        psa = (
            omega**2
            *
            np.max(
                np.abs(
                    ur
                )
            )
        )


        factores.append(
            psa
            /
            G
        )


    factores = np.asarray(
        factores,
        dtype=float
    )


    return {
        "RotD00_g":
            float(
                np.min(
                    factores
                )
            ),

        "RotD50_g":
            float(
                np.median(
                    factores
                )
            ),

        "RotD100_g":
            float(
                np.max(
                    factores
                )
            ),

        "Sa_por_angulo_g":
            factores
    }


# ======================================================================
# 7. FUNCIÓN COMPLETA PARA UNA ESTACIÓN
# ======================================================================

def calcular_estacion(
    row
):

    codigo = row[
        "Estacion"
    ]

    fs = float(
        row[
            "Fs_Hz"
        ]
    )

    dt = (
        1.0
        /
        fs
    )


    E0, N0 = segmento_alineado(
        row
    )


    # --------------------------------------------------------------
    # A. RAW
    # --------------------------------------------------------------

    E_raw = E0.copy()
    N_raw = N0.copy()


    # --------------------------------------------------------------
    # B. PRIMARY = DEMEAN
    # --------------------------------------------------------------

    E = (
        E0
        -
        np.mean(
            E0
        )
    )

    N = (
        N0
        -
        np.mean(
            N0
        )
    )


    # --------------------------------------------------------------
    # C. DETREND LINEAL
    # --------------------------------------------------------------

    E_lin = detrend(
        E0,
        type="linear"
    )

    N_lin = detrend(
        N0,
        type="linear"
    )


    # --------------------------------------------------------------
    # RESPONSE — RAW
    # --------------------------------------------------------------

    uE_raw = newmark_u(
        E_raw,
        dt,
        T,
        XI
    )

    uN_raw = newmark_u(
        N_raw,
        dt,
        T,
        XI
    )


    R_raw = rotd_desde_respuestas(
        uE_raw,
        uN_raw,
        T,
        ANGULOS_1
    )


    # --------------------------------------------------------------
    # RESPONSE — PRIMARY / DEMEAN
    # --------------------------------------------------------------

    uE = newmark_u(
        E,
        dt,
        T,
        XI
    )

    uN = newmark_u(
        N,
        dt,
        T,
        XI
    )


    R = rotd_desde_respuestas(
        uE,
        uN,
        T,
        ANGULOS_1
    )


    # --------------------------------------------------------------
    # RESPONSE — DETREND
    # --------------------------------------------------------------

    uE_lin = newmark_u(
        E_lin,
        dt,
        T,
        XI
    )

    uN_lin = newmark_u(
        N_lin,
        dt,
        T,
        XI
    )


    R_lin = rotd_desde_respuestas(
        uE_lin,
        uN_lin,
        T,
        ANGULOS_1
    )


    # --------------------------------------------------------------
    # ANGULAR RESOLUTION 0.5°
    # --------------------------------------------------------------

    R_05 = rotd_desde_respuestas(
        uE,
        uN,
        T,
        ANGULOS_05
    )


    # --------------------------------------------------------------
    # CONVERGENCIA TEMPORAL: dt/2
    #
    # ÚNICAMENTE auditoría numérica.
    # La señal primaria sigue siendo la original.
    # --------------------------------------------------------------

    n = len(E)


    E_half = np.empty(
        2*n - 1,
        dtype=float
    )

    N_half = np.empty(
        2*n - 1,
        dtype=float
    )


    E_half[
        0::2
    ] = E

    N_half[
        0::2
    ] = N


    E_half[
        1::2
    ] = (
        0.5
        *
        (
            E[:-1]
            +
            E[1:]
        )
    )

    N_half[
        1::2
    ] = (
        0.5
        *
        (
            N[:-1]
            +
            N[1:]
        )
    )


    uE_half = newmark_u(
        E_half,
        dt / 2.0,
        T,
        XI
    )

    uN_half = newmark_u(
        N_half,
        dt / 2.0,
        T,
        XI
    )


    R_half = rotd_desde_respuestas(
        uE_half,
        uN_half,
        T,
        ANGULOS_1
    )


    # --------------------------------------------------------------
    # SA COMPONENTES AS-ORIENTED — QC
    # --------------------------------------------------------------

    omega = (
        2.0
        *
        np.pi
        /
        T
    )


    Sa_E_g = (
        omega**2
        *
        np.max(
            np.abs(
                uE
            )
        )
        /
        G
    )


    Sa_N_g = (
        omega**2
        *
        np.max(
            np.abs(
                uN
            )
        )
        /
        G
    )


    # --------------------------------------------------------------
    # ORIENTACIÓN MÁS CERCANA AL RotD50
    # --------------------------------------------------------------

    vals = R[
        "Sa_por_angulo_g"
    ]


    idx_med = int(
        np.argmin(
            np.abs(
                vals
                -
                R["RotD50_g"]
            )
        )
    )


    theta_med = float(
        ANGULOS_1[
            idx_med
        ]
    )


    rad = np.deg2rad(
        theta_med
    )


    u_med = (
        uE
        *
        np.cos(rad)
        +
        uN
        *
        np.sin(rad)
    )


    idx_peak = int(
        np.argmax(
            np.abs(
                u_med
            )
        )
    )


    t_peak = (
        idx_peak
        *
        dt
    )


    duracion = (
        (
            len(
                u_med
            )
            -
            1
        )
        *
        dt
    )


    margen_borde = min(
        t_peak,
        duracion
        -
        t_peak
    )


    # --------------------------------------------------------------
    # SENSIBILIDADES
    # --------------------------------------------------------------

    delta_raw_pct = (
        100.0
        *
        (
            R_raw[
                "RotD50_g"
            ]
            /
            R[
                "RotD50_g"
            ]
            -
            1.0
        )
    )


    delta_lin_pct = (
        100.0
        *
        (
            R_lin[
                "RotD50_g"
            ]
            /
            R[
                "RotD50_g"
            ]
            -
            1.0
        )
    )


    delta_ang_pct = (
        100.0
        *
        (
            R_05[
                "RotD50_g"
            ]
            /
            R[
                "RotD50_g"
            ]
            -
            1.0
        )
    )


    delta_half_pct = (
        100.0
        *
        (
            R_half[
                "RotD50_g"
            ]
            /
            R[
                "RotD50_g"
            ]
            -
            1.0
        )
    )


    return {

        "Estacion":
            codigo,

        "Fs_Hz":
            fs,

        "N_comun":
            len(
                E
            ),

        "Duracion_s":
            len(E)
            /
            fs,

        "Media_E_raw_m_s2":
            float(
                np.mean(
                    E0
                )
            ),

        "Media_N_raw_m_s2":
            float(
                np.mean(
                    N0
                )
            ),

        "PGA_E_g":
            float(
                np.max(
                    np.abs(
                        E0
                    )
                )
                /
                G
            ),

        "PGA_N_g":
            float(
                np.max(
                    np.abs(
                        N0
                    )
                )
                /
                G
            ),

        "Sa_E_T1_g":
            float(
                Sa_E_g
            ),

        "Sa_N_T1_g":
            float(
                Sa_N_g
            ),

        "RotD00_T1_g":
            R[
                "RotD00_g"
            ],

        "RotD50_T1_g":
            R[
                "RotD50_g"
            ],

        "RotD100_T1_g":
            R[
                "RotD100_g"
            ],

        "RotD100_div_RotD50":
            (
                R[
                    "RotD100_g"
                ]
                /
                R[
                    "RotD50_g"
                ]
            ),

        "RotD50_raw_g":
            R_raw[
                "RotD50_g"
            ],

        "RotD50_detrend_lineal_g":
            R_lin[
                "RotD50_g"
            ],

        "Delta_raw_vs_primary_pct":
            delta_raw_pct,

        "Delta_detrend_vs_primary_pct":
            delta_lin_pct,

        "RotD50_0p5deg_g":
            R_05[
                "RotD50_g"
            ],

        "Delta_0p5deg_pct":
            delta_ang_pct,

        "RotD50_dt_mitad_g":
            R_half[
                "RotD50_g"
            ],

        "Delta_dt_mitad_pct":
            delta_half_pct,

        "Angulo_cercano_RotD50_deg":
            theta_med,

        "Tiempo_pico_RotD50_s":
            t_peak,

        "Margen_pico_a_borde_s":
            margen_borde,

        "Flag_pico_lt_5s_borde":
            bool(
                margen_borde
                <
                5.0
            ),

        "Flag_duracion_lt_60s":
            bool(
                (
                    len(E)
                    /
                    fs
                )
                <
                60.0
            )
    }


# ======================================================================
# 8. EJECUTAR LAS 42 ESTACIONES
# ======================================================================

resultados = []


print("\n" + "=" * 80)
print("CÁLCULO RotD50")
print("=" * 80)


for i, row in df.iterrows():

    print(
        f"[{i+1:02d}/42] "
        f"{row['Estacion']}"
    )

    r = calcular_estacion(
        row
    )

    resultados.append(
        r
    )


sa = pd.DataFrame(
    resultados
)


# ======================================================================
# 9. CONTROLES BÁSICOS
# ======================================================================

assert len(sa) == 42

assert sa[
    "Estacion"
].nunique() == 42


columnas_sa = [
    "RotD00_T1_g",
    "RotD50_T1_g",
    "RotD100_T1_g"
]


assert np.isfinite(
    sa[
        columnas_sa
    ].to_numpy()
).all()


assert (
    sa[
        columnas_sa
    ]
    >
    0
).all().all()


# Definición física:
# RotD00 <= RotD50 <= RotD100
assert (
    sa[
        "RotD00_T1_g"
    ]
    <=
    sa[
        "RotD50_T1_g"
    ]
).all()


assert (
    sa[
        "RotD50_T1_g"
    ]
    <=
    sa[
        "RotD100_T1_g"
    ]
).all()


# ======================================================================
# 10. RESULTADOS
# ======================================================================

print("\n" + "=" * 80)
print("RotD50(T=1.0 s, 5 %) — 42 ESTACIONES")
print("=" * 80)


display(
    sa[
        [
            "Estacion",
            "Fs_Hz",
            "Duracion_s",
            "PGA_E_g",
            "PGA_N_g",
            "Sa_E_T1_g",
            "Sa_N_T1_g",
            "RotD00_T1_g",
            "RotD50_T1_g",
            "RotD100_T1_g",
            "RotD100_div_RotD50"
        ]
    ].round(6)
)


# ======================================================================
# 11. AUDITORÍA DE PREPROCESAMIENTO
# ======================================================================

print("\n" + "=" * 80)
print("SENSIBILIDAD AL PREPROCESAMIENTO")
print("=" * 80)


display(
    sa[
        [
            "Estacion",
            "RotD50_T1_g",
            "RotD50_raw_g",
            "Delta_raw_vs_primary_pct",
            "RotD50_detrend_lineal_g",
            "Delta_detrend_vs_primary_pct"
        ]
    ].round(6)
)


print(
    "\nMáximo |raw vs primary| [%]:",
    sa[
        "Delta_raw_vs_primary_pct"
    ].abs().max()
)


print(
    "Máximo |detrend vs primary| [%]:",
    sa[
        "Delta_detrend_vs_primary_pct"
    ].abs().max()
)


# ======================================================================
# 12. CONVERGENCIA NUMÉRICA
# ======================================================================

print("\n" + "=" * 80)
print("CONVERGENCIA NUMÉRICA")
print("=" * 80)


display(
    sa[
        [
            "Estacion",
            "RotD50_T1_g",
            "RotD50_0p5deg_g",
            "Delta_0p5deg_pct",
            "RotD50_dt_mitad_g",
            "Delta_dt_mitad_pct"
        ]
    ].round(7)
)


print(
    "\nMáximo |efecto resolución angular| [%]:",
    sa[
        "Delta_0p5deg_pct"
    ].abs().max()
)


print(
    "Máximo |efecto dt/2| [%]:",
    sa[
        "Delta_dt_mitad_pct"
    ].abs().max()
)


# ======================================================================
# 13. CONTROL DE EXTREMOS / DURACIÓN
# ======================================================================

print("\n" + "=" * 80)
print("CONTROL DE DURACIÓN Y BORDE")
print("=" * 80)


flags = sa[
    (
        sa[
            "Flag_pico_lt_5s_borde"
        ]
    )
    |
    (
        sa[
            "Flag_duracion_lt_60s"
        ]
    )
].copy()


print(
    "Estaciones con alguna bandera:",
    len(flags)
)


display(
    flags[
        [
            "Estacion",
            "Duracion_s",
            "RotD50_T1_g",
            "Angulo_cercano_RotD50_deg",
            "Tiempo_pico_RotD50_s",
            "Margen_pico_a_borde_s",
            "Flag_pico_lt_5s_borde",
            "Flag_duracion_lt_60s"
        ]
    ].round(6)
)


# ======================================================================
# 14. RESUMEN ESTADÍSTICO Sa
# ======================================================================

print("\n" + "=" * 80)
print("RESUMEN RotD50")
print("=" * 80)


print(
    sa[
        "RotD50_T1_g"
    ].describe()
)


print(
    "\nMínimo:",
    sa[
        "RotD50_T1_g"
    ].min()
)


print(
    "Máximo:",
    sa[
        "RotD50_T1_g"
    ].max()
)


print(
    "Mediana:",
    sa[
        "RotD50_T1_g"
    ].median()
)


# ======================================================================
# 15. UNIR CON Mw / Rrup / Vs30
# ======================================================================

final = df.merge(
    sa,
    on="Estacion",
    how="left",
    validate="one_to_one"
)


assert len(final) == 42


# Nombre estandarizado del objetivo observado
final[
    "Sa_RotD50_T1_g"
] = final[
    "RotD50_T1_g"
]


# ======================================================================
# 16. GUARDAR RESULTADOS
# ======================================================================

RUTA_AUDITORIA = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_RotD50_T1.csv"
)

RUTA_FINAL = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_CANDIDATO.csv"
)


sa.to_csv(
    RUTA_AUDITORIA,
    index=False
)


final.to_csv(
    RUTA_FINAL,
    index=False
)


hash_final = sha256_file(
    RUTA_FINAL
)


# ======================================================================
# 17. GATE ESPECTRAL PRELIMINAR
# ======================================================================
#
# IMPORTANTE:
# PASS aquí significa solamente:
# cálculo internamente consistente.
#
# TODAVÍA falta la comparación independiente
# con CESMD antes de congelar Sa.
# ======================================================================

gate_basico = (
    len(final) == 42
    and
    np.isfinite(
        final[
            "Sa_RotD50_T1_g"
        ]
    ).all()
    and
    (
        final[
            "Sa_RotD50_T1_g"
        ]
        >
        0
    ).all()
)


gate_angular = (
    sa[
        "Delta_0p5deg_pct"
    ].abs().max()
    <
    0.5
)


gate_integracion = (
    sa[
        "Delta_dt_mitad_pct"
    ].abs().max()
    <
    1.0
)


print("\n" + "=" * 80)
print("GATE ESPECTRAL INTERNO")
print("=" * 80)


print(
    "Datos completos          :",
    "PASS"
    if gate_basico
    else "FAIL"
)


print(
    "Resolución angular       :",
    "PASS"
    if gate_angular
    else "REVISAR"
)


print(
    "Convergencia integración :",
    "PASS"
    if gate_integracion
    else "REVISAR"
)


if (
    gate_basico
    and
    gate_angular
    and
    gate_integracion
):

    print(
        "\n✅ CÁLCULO INTERNO PASS"
    )

    print(
        "RotD50 candidato calculado "
        "para 42 estaciones."
    )

    print(
        "\nNO está todavía congelado como "
        "Sa observado definitivo."
    )

    print(
        "Falta auditoría independiente CESMD."
    )

else:

    print(
        "\n⚠ DETENER ANTES DE CONGELAR Sa."
    )


print("\nArchivos guardados:")

print(
    RUTA_AUDITORIA
)

print(
    RUTA_FINAL
)


print(
    "\nSHA256 archivo candidato:"
)

print(
    hash_final
)


print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — AUDITORÍA INDEPENDIENTE CESMD
# ETAPA 1: DESCUBRIMIENTO DE LOS 5 REGISTROS
#
# OBJETIVO:
# - consultar CESMD para us10008kce;
# - identificar exactamente las 5 estaciones disponibles;
# - recuperar coordenadas y parámetros CESMD;
# - cruzarlas objetivamente con las 42 estaciones CSN;
# - identificar disponibilidad de datos procesados;
#
# NO modifica Sa CSN.
# NO congela todavía RotD50.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import json
import requests
import numpy as np
import pandas as pd

from math import radians, sin, cos, asin, sqrt


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

EVENT_ID = "us10008kce"

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

RUTA_CANDIDATO = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_CANDIDATO.csv"
)

assert os.path.exists(
    RUTA_CANDIDATO
)

csn = pd.read_csv(
    RUTA_CANDIDATO
)

assert len(csn) == 42


print("=" * 80)
print("VALPARAÍSO 2017 — CESMD / AUDITORÍA INDEPENDIENTE")
print("=" * 80)

print(
    "Evento:",
    EVENT_ID
)


# ======================================================================
# 2. CONSULTAR CESMD
# ======================================================================

URL = (
    "https://strongmotioncenter.org/"
    "wserv/records/query"
)

# Usar primero el conjunto mínimo de parámetros
# documentados por CESMD.
params = {
    "eventid": EVENT_ID,
    "rettype": "metadata",
    "format": "json",
    "nodata": 404
}


r = requests.get(
    URL,
    params=params,
    timeout=90
)


print("\nHTTP inicial:", r.status_code)
print("URL final:", r.url)


# Intento alternativo únicamente si el servidor
# exige network code en mayúscula.
if r.status_code != 200:

    params[
        "eventid"
    ] = EVENT_ID.upper()

    r = requests.get(
        URL,
        params=params,
        timeout=90
    )

    print(
        "HTTP segundo intento:",
        r.status_code
    )

    print(
        "URL segundo intento:",
        r.url
    )


r.raise_for_status()

cesmd = r.json()


# ======================================================================
# 3. GUARDAR RESPUESTA ÍNTEGRA
# ======================================================================

RUTA_JSON = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_metadata_original.json"
)


with open(
    RUTA_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cesmd,
        f,
        indent=2,
        ensure_ascii=False
    )


print("\nRespuesta guardada:")
print(RUTA_JSON)


# ======================================================================
# 4. INFORMACIÓN BÁSICA
# ======================================================================

print("\n" + "=" * 80)
print("RESPUESTA CESMD")
print("=" * 80)

print(
    "Tipo objeto:",
    type(cesmd).__name__
)


if isinstance(
    cesmd,
    dict
):

    print(
        "Keys principales:",
        list(
            cesmd.keys()
        )
    )

    print(
        "count:",
        cesmd.get(
            "count"
        )
    )

    print(
        "status:",
        cesmd.get(
            "status"
        )
    )


# ======================================================================
# 5. RECORRER TODO EL JSON
# ======================================================================

todos_dicts = []


def recorrer(
    obj,
    path="root"
):

    if isinstance(
        obj,
        dict
    ):

        todos_dicts.append(
            (
                path,
                obj
            )
        )

        for k, v in obj.items():

            recorrer(
                v,
                f"{path}.{k}"
            )


    elif isinstance(
        obj,
        list
    ):

        for i, v in enumerate(
            obj
        ):

            recorrer(
                v,
                f"{path}[{i}]"
            )


recorrer(
    cesmd
)


print(
    "\nDiccionarios encontrados:",
    len(todos_dicts)
)


# ======================================================================
# 6. MOSTRAR DICCIONARIOS QUE PARECEN ESTACIONES
# ======================================================================

candidatos_estacion = []


for path, d in todos_dicts:

    claves = set(
        d.keys()
    )


    if (
        "code" in claves
        and
        (
            "latitude" in claves
            or
            "longitude" in claves
        )
    ):

        candidatos_estacion.append(
            (
                path,
                d
            )
        )


print("\n" + "=" * 80)
print("OBJETOS CESMD QUE PARECEN ESTACIONES")
print("=" * 80)

print(
    "N =",
    len(candidatos_estacion)
)


for i, (
    path,
    d
) in enumerate(
    candidatos_estacion
):

    print(
        f"\n--- Candidato {i+1} ---"
    )

    print(
        "PATH:",
        path
    )

    for k in [
        "network",
        "code",
        "name",
        "location",
        "latitude",
        "longitude",
        "elevation",
        "Vs30",
        "channels",
        "numRecorders",
        "status"
    ]:

        if k in d:

            print(
                f"{k}:",
                d[k]
            )


# ======================================================================
# 7. MOSTRAR DICCIONARIOS CON PARÁMETROS DE REGISTRO
# ======================================================================

objetos_record = []


for path, d in todos_dicts:

    claves = set(
        d.keys()
    )


    if any(
        k in claves
        for k in [
            "sa10",
            "pgav2",
            "pgav1",
            "epidist",
            "fault_dist"
        ]
    ):

        objetos_record.append(
            (
                path,
                d
            )
        )


print("\n" + "=" * 80)
print("OBJETOS CESMD CON PARÁMETROS DE MOVIMIENTO")
print("=" * 80)

print(
    "N =",
    len(objetos_record)
)


for i, (
    path,
    d
) in enumerate(
    objetos_record
):

    print(
        f"\n--- Record {i+1} ---"
    )

    print(
        "PATH:",
        path
    )

    for k in [
        "epidist",
        "fault_dist",
        "pgav1",
        "pgav2",
        "pgv",
        "pgd",
        "sa03",
        "sa10",
        "sa30"
    ]:

        if k in d:

            print(
                f"{k}:",
                d[k]
            )


# ======================================================================
# 8. FUNCIÓN HAVERSINE PARA MATCH ESPACIAL
# ======================================================================

def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2
):

    R = 6371.0088

    lat1 = radians(
        lat1
    )

    lon1 = radians(
        lon1
    )

    lat2 = radians(
        lat2
    )

    lon2 = radians(
        lon2
    )


    dlat = (
        lat2
        -
        lat1
    )

    dlon = (
        lon2
        -
        lon1
    )


    a = (
        sin(
            dlat / 2
        )**2
        +
        cos(
            lat1
        )
        *
        cos(
            lat2
        )
        *
        sin(
            dlon / 2
        )**2
    )


    return (
        2
        *
        R
        *
        asin(
            sqrt(a)
        )
    )


# ======================================================================
# 9. EXTRAER ESTACIONES CESMD
# ======================================================================

filas_est = []


for path, d in candidatos_estacion:

    try:

        lat = float(
            d["latitude"]
        )

        lon = float(
            d["longitude"]
        )

    except Exception:

        continue


    filas_est.append(
        {
            "CESMD_path":
                path,

            "CESMD_network":
                d.get(
                    "network"
                ),

            "CESMD_code":
                str(
                    d.get(
                        "code"
                    )
                ),

            "CESMD_name":
                d.get(
                    "name"
                ),

            "CESMD_lat":
                lat,

            "CESMD_lon":
                lon,

            "CESMD_Vs30":
                d.get(
                    "Vs30"
                ),

            "CESMD_channels":
                d.get(
                    "channels"
                )
        }
    )


cesmd_est = pd.DataFrame(
    filas_est
)


# eliminar duplicados exactos
if len(
    cesmd_est
):

    cesmd_est = (
        cesmd_est
        .drop_duplicates(
            subset=[
                "CESMD_network",
                "CESMD_code",
                "CESMD_lat",
                "CESMD_lon"
            ]
        )
        .reset_index(
            drop=True
        )
    )


print("\n" + "=" * 80)
print("ESTACIONES CESMD EXTRAÍDAS")
print("=" * 80)

print(
    "N únicas:",
    len(
        cesmd_est
    )
)


display(
    cesmd_est
)


# ======================================================================
# 10. MATCH CESMD ↔ CSN POR COORDENADAS
# ======================================================================

matches = []


for _, ce in cesmd_est.iterrows():

    distancias = []


    for _, ch in csn.iterrows():

        d = haversine_km(
            ce[
                "CESMD_lat"
            ],
            ce[
                "CESMD_lon"
            ],
            ch[
                "Latitud"
            ],
            ch[
                "Longitud"
            ]
        )

        distancias.append(
            (
                d,
                ch[
                    "Estacion"
                ]
            )
        )


    distancias = sorted(
        distancias
    )


    d1, est1 = distancias[
        0
    ]


    d2, est2 = distancias[
        1
    ]


    matches.append(
        {
            "CESMD_network":
                ce[
                    "CESMD_network"
                ],

            "CESMD_code":
                ce[
                    "CESMD_code"
                ],

            "CESMD_name":
                ce[
                    "CESMD_name"
                ],

            "CESMD_lat":
                ce[
                    "CESMD_lat"
                ],

            "CESMD_lon":
                ce[
                    "CESMD_lon"
                ],

            "CSN_match":
                est1,

            "Dist_match_km":
                d1,

            "Segundo_CSN":
                est2,

            "Dist_segundo_km":
                d2
        }
    )


match = pd.DataFrame(
    matches
)


print("\n" + "=" * 80)
print("MATCH ESPACIAL CESMD ↔ CSN")
print("=" * 80)


display(
    match.round(
        {
            "CESMD_lat": 6,
            "CESMD_lon": 6,
            "Dist_match_km": 4,
            "Dist_segundo_km": 4
        }
    )
)


# ======================================================================
# 11. INTENTAR ASOCIAR sa10
# ======================================================================
#
# IMPORTANTE:
# sa10 todavía NO se interpreta como RotD50.
# Solo se documenta para decidir el siguiente paso.
# ======================================================================

registros = []


for path, d in objetos_record:

    registros.append(
        {
            "path":
                path,

            "epidist":
                d.get(
                    "epidist"
                ),

            "fault_dist":
                d.get(
                    "fault_dist"
                ),

            "pgav1":
                d.get(
                    "pgav1"
                ),

            "pgav2":
                d.get(
                    "pgav2"
                ),

            "sa03":
                d.get(
                    "sa03"
                ),

            "sa10":
                d.get(
                    "sa10"
                ),

            "sa30":
                d.get(
                    "sa30"
                )
        }
    )


record_df = pd.DataFrame(
    registros
)


print("\n" + "=" * 80)
print("PARÁMETROS CESMD — SOLO AUDITORÍA")
print("=" * 80)

display(
    record_df
)


# ======================================================================
# 12. CONSULTAR PÁGINA HTML CLÁSICA CESMD
# ======================================================================

URL_HTML = (
    "https://www.strongmotioncenter.org/"
    "cgi-bin/CESMD/apktable.pl"
)


rh = requests.get(
    URL_HTML,
    params={
        "iqrid":
            EVENT_ID
    },
    timeout=90
)


print("\n" + "=" * 80)
print("PÁGINA CLÁSICA CESMD")
print("=" * 80)

print(
    "HTTP:",
    rh.status_code
)

print(
    "URL:",
    rh.url
)

print(
    "Tamaño HTML:",
    len(
        rh.text
    )
)


RUTA_HTML = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_event.html"
)


if rh.status_code == 200:

    with open(
        RUTA_HTML,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            rh.text
        )


    try:

        tablas = pd.read_html(
            rh.text
        )

    except Exception:

        tablas = []


    print(
        "Tablas HTML encontradas:",
        len(
            tablas
        )
    )


    for i, tabla in enumerate(
        tablas
    ):

        print(
            f"\nTabla {i}:",
            tabla.shape
        )

        display(
            tabla.head(
                20
            )
        )


# ======================================================================
# 13. GUARDAR
# ======================================================================

RUTA_EST = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_estaciones.csv"
)

RUTA_MATCH = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_match_CSN.csv"
)

RUTA_RECORD = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_parametros_registros.csv"
)


cesmd_est.to_csv(
    RUTA_EST,
    index=False
)

match.to_csv(
    RUTA_MATCH,
    index=False
)

record_df.to_csv(
    RUTA_RECORD,
    index=False
)


# ======================================================================
# 14. GATE DE DESCUBRIMIENTO
# ======================================================================

print("\n" + "=" * 80)
print("GATE CESMD — ETAPA 1")
print("=" * 80)


print(
    "Estaciones CESMD extraídas:",
    len(
        cesmd_est
    )
)


if len(
    cesmd_est
) == 5:

    print(
        "✅ Se recuperaron las 5 "
        "estaciones esperadas."
    )

else:

    print(
        "⚠ La estructura del JSON requiere "
        "ajustar el parser."
    )


if len(
    match
):

    print(
        "Máxima distancia del match:",
        match[
            "Dist_match_km"
        ].max(),
        "km"
    )


print(
    "\nIMPORTANTE:"
)

print(
    "sa10 de CESMD todavía NO se "
    "considera equivalente a RotD50."
)

print(
    "Primero identificaremos el producto "
    "procesado / Vol3 correspondiente."
)

print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — AUDITORÍA INDEPENDIENTE CESMD
# ETAPA 1: DESCUBRIMIENTO DE LOS 5 REGISTROS
#
# OBJETIVO:
# - consultar CESMD para us10008kce;
# - identificar exactamente las 5 estaciones disponibles;
# - recuperar coordenadas y parámetros CESMD;
# - cruzarlas objetivamente con las 42 estaciones CSN;
# - identificar disponibilidad de datos procesados;
#
# NO modifica Sa CSN.
# NO congela todavía RotD50.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import json
import requests
import numpy as np
import pandas as pd

from math import radians, sin, cos, asin, sqrt


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

EVENT_ID = "us10008kce"

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

RUTA_CANDIDATO = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_CANDIDATO.csv"
)

assert os.path.exists(
    RUTA_CANDIDATO
)

csn = pd.read_csv(
    RUTA_CANDIDATO
)

assert len(csn) == 42


print("=" * 80)
print("VALPARAÍSO 2017 — CESMD / AUDITORÍA INDEPENDIENTE")
print("=" * 80)

print(
    "Evento:",
    EVENT_ID
)


# ======================================================================
# 2. CONSULTAR CESMD
# ======================================================================

URL = (
    "https://strongmotioncenter.org/"
    "wserv/records/query"
)

# Usar primero el conjunto mínimo de parámetros
# documentados por CESMD.
params = {
    "eventid": EVENT_ID,
    "rettype": "metadata",
    "format": "json",
    "nodata": 404
}


r = requests.get(
    URL,
    params=params,
    timeout=90
)


print("\nHTTP inicial:", r.status_code)
print("URL final:", r.url)


# Intento alternativo únicamente si el servidor
# exige network code en mayúscula.
if r.status_code != 200:

    params[
        "eventid"
    ] = EVENT_ID.upper()

    r = requests.get(
        URL,
        params=params,
        timeout=90
    )

    print(
        "HTTP segundo intento:",
        r.status_code
    )

    print(
        "URL segundo intento:",
        r.url
    )


r.raise_for_status()

cesmd = r.json()


# ======================================================================
# 3. GUARDAR RESPUESTA ÍNTEGRA
# ======================================================================

RUTA_JSON = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_metadata_original.json"
)


with open(
    RUTA_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cesmd,
        f,
        indent=2,
        ensure_ascii=False
    )


print("\nRespuesta guardada:")
print(RUTA_JSON)


# ======================================================================
# 4. INFORMACIÓN BÁSICA
# ======================================================================

print("\n" + "=" * 80)
print("RESPUESTA CESMD")
print("=" * 80)

print(
    "Tipo objeto:",
    type(cesmd).__name__
)


if isinstance(
    cesmd,
    dict
):

    print(
        "Keys principales:",
        list(
            cesmd.keys()
        )
    )

    print(
        "count:",
        cesmd.get(
            "count"
        )
    )

    print(
        "status:",
        cesmd.get(
            "status"
        )
    )


# ======================================================================
# 5. RECORRER TODO EL JSON
# ======================================================================

todos_dicts = []


def recorrer(
    obj,
    path="root"
):

    if isinstance(
        obj,
        dict
    ):

        todos_dicts.append(
            (
                path,
                obj
            )
        )

        for k, v in obj.items():

            recorrer(
                v,
                f"{path}.{k}"
            )


    elif isinstance(
        obj,
        list
    ):

        for i, v in enumerate(
            obj
        ):

            recorrer(
                v,
                f"{path}[{i}]"
            )


recorrer(
    cesmd
)


print(
    "\nDiccionarios encontrados:",
    len(todos_dicts)
)


# ======================================================================
# 6. MOSTRAR DICCIONARIOS QUE PARECEN ESTACIONES
# ======================================================================

candidatos_estacion = []


for path, d in todos_dicts:

    claves = set(
        d.keys()
    )


    if (
        "code" in claves
        and
        (
            "latitude" in claves
            or
            "longitude" in claves
        )
    ):

        candidatos_estacion.append(
            (
                path,
                d
            )
        )


print("\n" + "=" * 80)
print("OBJETOS CESMD QUE PARECEN ESTACIONES")
print("=" * 80)

print(
    "N =",
    len(candidatos_estacion)
)


for i, (
    path,
    d
) in enumerate(
    candidatos_estacion
):

    print(
        f"\n--- Candidato {i+1} ---"
    )

    print(
        "PATH:",
        path
    )

    for k in [
        "network",
        "code",
        "name",
        "location",
        "latitude",
        "longitude",
        "elevation",
        "Vs30",
        "channels",
        "numRecorders",
        "status"
    ]:

        if k in d:

            print(
                f"{k}:",
                d[k]
            )


# ======================================================================
# 7. MOSTRAR DICCIONARIOS CON PARÁMETROS DE REGISTRO
# ======================================================================

objetos_record = []


for path, d in todos_dicts:

    claves = set(
        d.keys()
    )


    if any(
        k in claves
        for k in [
            "sa10",
            "pgav2",
            "pgav1",
            "epidist",
            "fault_dist"
        ]
    ):

        objetos_record.append(
            (
                path,
                d
            )
        )


print("\n" + "=" * 80)
print("OBJETOS CESMD CON PARÁMETROS DE MOVIMIENTO")
print("=" * 80)

print(
    "N =",
    len(objetos_record)
)


for i, (
    path,
    d
) in enumerate(
    objetos_record
):

    print(
        f"\n--- Record {i+1} ---"
    )

    print(
        "PATH:",
        path
    )

    for k in [
        "epidist",
        "fault_dist",
        "pgav1",
        "pgav2",
        "pgv",
        "pgd",
        "sa03",
        "sa10",
        "sa30"
    ]:

        if k in d:

            print(
                f"{k}:",
                d[k]
            )


# ======================================================================
# 8. FUNCIÓN HAVERSINE PARA MATCH ESPACIAL
# ======================================================================

def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2
):

    R = 6371.0088

    lat1 = radians(
        lat1
    )

    lon1 = radians(
        lon1
    )

    lat2 = radians(
        lat2
    )

    lon2 = radians(
        lon2
    )


    dlat = (
        lat2
        -
        lat1
    )

    dlon = (
        lon2
        -
        lon1
    )


    a = (
        sin(
            dlat / 2
        )**2
        +
        cos(
            lat1
        )
        *
        cos(
            lat2
        )
        *
        sin(
            dlon / 2
        )**2
    )


    return (
        2
        *
        R
        *
        asin(
            sqrt(a)
        )
    )


# ======================================================================
# 9. EXTRAER ESTACIONES CESMD
# ======================================================================

filas_est = []


for path, d in candidatos_estacion:

    try:

        lat = float(
            d["latitude"]
        )

        lon = float(
            d["longitude"]
        )

    except Exception:

        continue


    filas_est.append(
        {
            "CESMD_path":
                path,

            "CESMD_network":
                d.get(
                    "network"
                ),

            "CESMD_code":
                str(
                    d.get(
                        "code"
                    )
                ),

            "CESMD_name":
                d.get(
                    "name"
                ),

            "CESMD_lat":
                lat,

            "CESMD_lon":
                lon,

            "CESMD_Vs30":
                d.get(
                    "Vs30"
                ),

            "CESMD_channels":
                d.get(
                    "channels"
                )
        }
    )


cesmd_est = pd.DataFrame(
    filas_est
)


# eliminar duplicados exactos
if len(
    cesmd_est
):

    cesmd_est = (
        cesmd_est
        .drop_duplicates(
            subset=[
                "CESMD_network",
                "CESMD_code",
                "CESMD_lat",
                "CESMD_lon"
            ]
        )
        .reset_index(
            drop=True
        )
    )


print("\n" + "=" * 80)
print("ESTACIONES CESMD EXTRAÍDAS")
print("=" * 80)

print(
    "N únicas:",
    len(
        cesmd_est
    )
)


display(
    cesmd_est
)


# ======================================================================
# 10. MATCH CESMD ↔ CSN POR COORDENADAS
# ======================================================================

matches = []


for _, ce in cesmd_est.iterrows():

    distancias = []


    for _, ch in csn.iterrows():

        d = haversine_km(
            ce[
                "CESMD_lat"
            ],
            ce[
                "CESMD_lon"
            ],
            ch[
                "Latitud"
            ],
            ch[
                "Longitud"
            ]
        )

        distancias.append(
            (
                d,
                ch[
                    "Estacion"
                ]
            )
        )


    distancias = sorted(
        distancias
    )


    d1, est1 = distancias[
        0
    ]


    d2, est2 = distancias[
        1
    ]


    matches.append(
        {
            "CESMD_network":
                ce[
                    "CESMD_network"
                ],

            "CESMD_code":
                ce[
                    "CESMD_code"
                ],

            "CESMD_name":
                ce[
                    "CESMD_name"
                ],

            "CESMD_lat":
                ce[
                    "CESMD_lat"
                ],

            "CESMD_lon":
                ce[
                    "CESMD_lon"
                ],

            "CSN_match":
                est1,

            "Dist_match_km":
                d1,

            "Segundo_CSN":
                est2,

            "Dist_segundo_km":
                d2
        }
    )


match = pd.DataFrame(
    matches
)


print("\n" + "=" * 80)
print("MATCH ESPACIAL CESMD ↔ CSN")
print("=" * 80)


display(
    match.round(
        {
            "CESMD_lat": 6,
            "CESMD_lon": 6,
            "Dist_match_km": 4,
            "Dist_segundo_km": 4
        }
    )
)


# ======================================================================
# 11. INTENTAR ASOCIAR sa10
# ======================================================================
#
# IMPORTANTE:
# sa10 todavía NO se interpreta como RotD50.
# Solo se documenta para decidir el siguiente paso.
# ======================================================================

registros = []


for path, d in objetos_record:

    registros.append(
        {
            "path":
                path,

            "epidist":
                d.get(
                    "epidist"
                ),

            "fault_dist":
                d.get(
                    "fault_dist"
                ),

            "pgav1":
                d.get(
                    "pgav1"
                ),

            "pgav2":
                d.get(
                    "pgav2"
                ),

            "sa03":
                d.get(
                    "sa03"
                ),

            "sa10":
                d.get(
                    "sa10"
                ),

            "sa30":
                d.get(
                    "sa30"
                )
        }
    )


record_df = pd.DataFrame(
    registros
)


print("\n" + "=" * 80)
print("PARÁMETROS CESMD — SOLO AUDITORÍA")
print("=" * 80)

display(
    record_df
)


# ======================================================================
# 12. CONSULTAR PÁGINA HTML CLÁSICA CESMD
# ======================================================================

URL_HTML = (
    "https://www.strongmotioncenter.org/"
    "cgi-bin/CESMD/apktable.pl"
)


rh = requests.get(
    URL_HTML,
    params={
        "iqrid":
            EVENT_ID
    },
    timeout=90
)


print("\n" + "=" * 80)
print("PÁGINA CLÁSICA CESMD")
print("=" * 80)

print(
    "HTTP:",
    rh.status_code
)

print(
    "URL:",
    rh.url
)

print(
    "Tamaño HTML:",
    len(
        rh.text
    )
)


RUTA_HTML = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_event.html"
)


if rh.status_code == 200:

    with open(
        RUTA_HTML,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            rh.text
        )


    try:

        tablas = pd.read_html(
            rh.text
        )

    except Exception:

        tablas = []


    print(
        "Tablas HTML encontradas:",
        len(
            tablas
        )
    )


    for i, tabla in enumerate(
        tablas
    ):

        print(
            f"\nTabla {i}:",
            tabla.shape
        )

        display(
            tabla.head(
                20
            )
        )


# ======================================================================
# 13. GUARDAR
# ======================================================================

RUTA_EST = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_estaciones.csv"
)

RUTA_MATCH = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_match_CSN.csv"
)

RUTA_RECORD = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_parametros_registros.csv"
)


cesmd_est.to_csv(
    RUTA_EST,
    index=False
)

match.to_csv(
    RUTA_MATCH,
    index=False
)

record_df.to_csv(
    RUTA_RECORD,
    index=False
)


# ======================================================================
# 14. GATE DE DESCUBRIMIENTO
# ======================================================================

print("\n" + "=" * 80)
print("GATE CESMD — ETAPA 1")
print("=" * 80)


print(
    "Estaciones CESMD extraídas:",
    len(
        cesmd_est
    )
)


if len(
    cesmd_est
) == 5:

    print(
        "✅ Se recuperaron las 5 "
        "estaciones esperadas."
    )

else:

    print(
        "⚠ La estructura del JSON requiere "
        "ajustar el parser."
    )


if len(
    match
):

    print(
        "Máxima distancia del match:",
        match[
            "Dist_match_km"
        ].max(),
        "km"
    )


print(
    "\nIMPORTANTE:"
)

print(
    "sa10 de CESMD todavía NO se "
    "considera equivalente a RotD50."
)

print(
    "Primero identificaremos el producto "
    "procesado / Vol3 correspondiente."
)

print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — CESMD FALLBACK OFICIAL
# Descubrimiento desde el archivo clásico CESMD
#
# OBJETIVO:
# 1. abrir archivo CESMD 2017;
# 2. localizar inequívocamente us10008kce;
# 3. seguir el enlace interno del propio CESMD;
# 4. recuperar las 5 estaciones;
# 5. cruzarlas con las 42 estaciones CSN;
# 6. guardar todo para la auditoría espectral posterior.
#
# NO modifica Sa CSN.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import requests
import numpy as np
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from math import radians, sin, cos, asin, sqrt


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

EVENT_ID = "us10008kce"

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

RUTA_CANDIDATO = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_CANDIDATO.csv"
)

assert os.path.exists(
    RUTA_CANDIDATO
)

csn = pd.read_csv(
    RUTA_CANDIDATO
)

assert len(csn) == 42

BASE_CESMD = (
    "https://www.strongmotioncenter.org"
)

ARCHIVE_URL = (
    BASE_CESMD
    +
    "/cgi-bin/CESMD/archive.pl?Archives=2017"
)


session = requests.Session()

session.headers.update(
    {
        "User-Agent":
            "Mozilla/5.0 "
            "(academic reproducibility audit)"
    }
)


print("=" * 80)
print("VALPARAÍSO 2017 — CESMD FALLBACK OFICIAL")
print("=" * 80)

print("Evento:", EVENT_ID)
print("Archivo:", ARCHIVE_URL)


# ======================================================================
# 2. DESCARGAR ARCHIVO 2017
# ======================================================================

r = session.get(
    ARCHIVE_URL,
    timeout=90
)

print(
    "\nHTTP archive:",
    r.status_code
)

r.raise_for_status()


RUTA_ARCHIVE = os.path.join(
    CARPETA,
    "CESMD_archive_2017.html"
)

with open(
    RUTA_ARCHIVE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        r.text
    )


soup = BeautifulSoup(
    r.text,
    "html.parser"
)


# ======================================================================
# 3. LOCALIZAR FILA us10008kce
# ======================================================================

fila_evento = None


for tr in soup.find_all(
    "tr"
):

    texto = tr.get_text(
        " ",
        strip=True
    )

    if EVENT_ID.lower() in texto.lower():

        fila_evento = tr
        break


if fila_evento is None:

    raise RuntimeError(
        "No se encontró us10008kce "
        "en el archivo CESMD 2017."
    )


texto_fila = fila_evento.get_text(
    " ",
    strip=True
)


print("\n" + "=" * 80)
print("FILA CESMD DEL EVENTO")
print("=" * 80)

print(
    texto_fila
)


# Debemos confirmar elementos básicos
assert "Valparaiso" in texto_fila

assert (
    EVENT_ID.lower()
    in
    texto_fila.lower()
)


# ======================================================================
# 4. EXTRAER ENLACES DE LA FILA
# ======================================================================

links_evento = []


for a in fila_evento.find_all(
    "a",
    href=True
):

    url = urljoin(
        ARCHIVE_URL,
        a[
            "href"
        ]
    )

    links_evento.append(
        {
            "texto":
                a.get_text(
                    " ",
                    strip=True
                ),

            "href":
                a[
                    "href"
                ],

            "url":
                url
        }
    )


df_links = pd.DataFrame(
    links_evento
)


print("\n" + "=" * 80)
print("ENLACES CESMD DEL EVENTO")
print("=" * 80)

display(
    df_links
)


# ======================================================================
# 5. GENERAR CANDIDATOS DE PÁGINA DE DETALLE
# ======================================================================

urls_candidatas = []


# Primero: enlaces reales encontrados
for x in links_evento:

    url = x[
        "url"
    ]

    if url not in urls_candidatas:

        urls_candidatas.append(
            url
        )


# Fallback clásico conocido de CESMD
fallbacks = [

    (
        BASE_CESMD
        +
        "/cgi-bin/CESMD/apktable.pl?"
        f"iqrid={EVENT_ID}"
    ),

    (
        BASE_CESMD
        +
        "/cgi-bin/CESMD/apktable.pl?"
        "iqrid=10008kce"
    )
]


for url in fallbacks:

    if url not in urls_candidatas:

        urls_candidatas.append(
            url
        )


# ======================================================================
# 6. PROBAR PÁGINAS
# ======================================================================

pagina_evento = None
url_evento = None


print("\n" + "=" * 80)
print("PRUEBA DE PÁGINAS CESMD")
print("=" * 80)


for url in urls_candidatas:

    try:

        rr = session.get(
            url,
            timeout=90
        )

        texto = rr.text


        print(
            "\nHTTP",
            rr.status_code,
            "|",
            url
        )

        print(
            "Tamaño:",
            len(texto)
        )


        if rr.status_code != 200:

            continue


        criterio_1 = (
            "Valparaiso"
            in texto
        )

        criterio_2 = (
            "Peak Ground Motion"
            in texto
            or
            "Station Name"
            in texto
            or
            "Statn"
            in texto
        )


        if (
            criterio_1
            and
            criterio_2
        ):

            pagina_evento = texto
            url_evento = url

            print(
                "✅ Página de detalle detectada."
            )

            break


    except Exception as e:

        print(
            "Error:",
            repr(e)
        )


if pagina_evento is None:

    raise RuntimeError(
        "No fue posible identificar "
        "la página de detalle CESMD."
    )


print("\nPágina seleccionada:")
print(
    url_evento
)


# ======================================================================
# 7. GUARDAR PÁGINA CESMD
# ======================================================================

RUTA_EVENTO = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_event_CLASSIC.html"
)


with open(
    RUTA_EVENTO,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        pagina_evento
    )


# ======================================================================
# 8. EXTRAER TEXTO DE LA PÁGINA
# ======================================================================

soup_ev = BeautifulSoup(
    pagina_evento,
    "html.parser"
)


texto_ev = soup_ev.get_text(
    "\n"
)


lineas = [

    re.sub(
        r"\s+",
        " ",
        x
    ).strip()

    for x in texto_ev.splitlines()

    if x.strip()
]


print("\n" + "=" * 80)
print("CABECERA CESMD")
print("=" * 80)


for linea in lineas:

    if (
        "Valparaiso"
        in linea
        or
        "Event Id"
        in linea
        or
        "Peak Ground Motion"
        in linea
        or
        "Table Last Updated"
        in linea
    ):

        print(
            linea
        )


# ======================================================================
# 9. MOSTRAR BLOQUE DE LA TABLA
# ======================================================================

print("\n" + "=" * 80)
print("BLOQUE DE REGISTROS CESMD")
print("=" * 80)


idx_header = None


for i, linea in enumerate(
    lineas
):

    if (
        "Station Name"
        in linea
        or
        (
            "Network"
            in linea
            and
            "Statn"
            in linea
        )
    ):

        idx_header = i
        break


if idx_header is not None:

    for linea in lineas[
        idx_header:
        idx_header + 25
    ]:

        print(
            linea
        )

else:

    print(
        "No se detectó automáticamente "
        "la cabecera tabular."
    )


# ======================================================================
# 10. PRIMER INTENTO: pd.read_html
# ======================================================================

tablas = []


try:

    tablas = pd.read_html(
        pagina_evento
    )

except Exception as e:

    print(
        "\npd.read_html:",
        repr(e)
    )


print(
    "\nTablas HTML detectadas:",
    len(tablas)
)


for i, tabla in enumerate(
    tablas
):

    print(
        f"\nTabla {i}:",
        tabla.shape
    )

    display(
        tabla.head(20)
    )


# ======================================================================
# 11. PARSER DE FILAS DE MOVIMIENTO FUERTE
# ======================================================================
#
# Formato clásico aproximado:
#
# C1 RED VA01 Nombre estación ...
# 33.xxx 71.xxx distancia(...) PGA ...
#
# La página identifica las columnas como
# S.Lat y W.Long para Chile.
# ======================================================================

texto_plano = soup_ev.get_text(
    "\n"
)


usar_lat_sur = (
    "S.Lat"
    in texto_plano
)

usar_lon_oeste = (
    "W.Long"
    in texto_plano
)


filas = []


# Regex deliberadamente flexible:
#
# 1 Network ID
# 2 Network name/token
# 3 station code
# 4 station name
# 5 lat
# 6 lon
# 7 epicentral distance
# 8 fault-distance text
#
patron = re.compile(
    r"^\s*"
    r"([A-Za-z0-9]{1,4})\s+"
    r"([A-Za-z0-9_-]{1,20})\s+"
    r"([A-Za-z0-9_-]{2,12})\s+"
    r"(.+?)\s+"
    r"(\d{1,2}\.\d+)\s+"
    r"(\d{2,3}\.\d+)\s+"
    r"(\d+(?:\.\d+)?)"
    r"\s*\(\s*([^)]+)\)"
    r"(.*)$"
)


for linea_original in texto_ev.splitlines():

    linea = re.sub(
        r"\s+",
        " ",
        linea_original
    ).strip()


    m = patron.match(
        linea
    )


    if not m:

        continue


    netid = m.group(1)
    netname = m.group(2)
    code = m.group(3)
    name = m.group(4)

    lat = float(
        m.group(5)
    )

    lon = float(
        m.group(6)
    )

    epidist = float(
        m.group(7)
    )

    fault_txt = (
        m.group(8)
        .strip()
    )

    resto = (
        m.group(9)
        .strip()
    )


    if usar_lat_sur:

        lat = -abs(
            lat
        )


    if usar_lon_oeste:

        lon = -abs(
            lon
        )


    # Solo aceptar coordenadas geográficamente
    # compatibles con Chile central / norte-central.
    if not (
        -40 < lat < -25
        and
        -75 < lon < -65
    ):

        continue


    filas.append(
        {
            "CESMD_netid":
                netid,

            "CESMD_network_name":
                netname,

            "CESMD_code":
                code.upper(),

            "CESMD_station_name":
                name.strip(),

            "CESMD_lat":
                lat,

            "CESMD_lon":
                lon,

            "CESMD_epidist_km":
                epidist,

            "CESMD_fault_distance_text":
                fault_txt,

            "CESMD_rest_of_row":
                resto,

            "CESMD_original_line":
                linea
        }
    )


cesmd5 = pd.DataFrame(
    filas
)


if len(cesmd5):

    cesmd5 = (
        cesmd5
        .drop_duplicates(
            subset=[
                "CESMD_netid",
                "CESMD_code"
            ]
        )
        .reset_index(
            drop=True
        )
    )


print("\n" + "=" * 80)
print("ESTACIONES CESMD PARSEADAS")
print("=" * 80)

print(
    "N =",
    len(
        cesmd5
    )
)


display(
    cesmd5
)


# ======================================================================
# 12. HAVERSINE
# ======================================================================

def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2
):

    R = 6371.0088

    p1 = radians(
        lat1
    )

    p2 = radians(
        lat2
    )

    dphi = radians(
        lat2
        -
        lat1
    )

    dlambda = radians(
        lon2
        -
        lon1
    )


    a = (
        sin(
            dphi / 2
        )**2
        +
        cos(p1)
        *
        cos(p2)
        *
        sin(
            dlambda / 2
        )**2
    )


    return (
        2
        *
        R
        *
        asin(
            sqrt(a)
        )
    )


# ======================================================================
# 13. MATCH CON CSN
# ======================================================================

matches = []


for _, ce in cesmd5.iterrows():

    code = ce[
        "CESMD_code"
    ]


    # ----------------------------------------------------------
    # Primero intentar match por código
    # ----------------------------------------------------------

    exacto = csn[
        csn[
            "Estacion"
        ].str.upper()
        ==
        code.upper()
    ]


    if len(exacto) == 1:

        ch = exacto.iloc[
            0
        ]

        dist = haversine_km(
            ce[
                "CESMD_lat"
            ],
            ce[
                "CESMD_lon"
            ],
            ch[
                "Latitud"
            ],
            ch[
                "Longitud"
            ]
        )

        metodo = (
            "CODIGO_EXACTO"
        )

        csn_match = ch[
            "Estacion"
        ]


    else:

        # ------------------------------------------------------
        # Fallback espacial
        # ------------------------------------------------------

        dists = []


        for _, ch in csn.iterrows():

            d = haversine_km(
                ce[
                    "CESMD_lat"
                ],
                ce[
                    "CESMD_lon"
                ],
                ch[
                    "Latitud"
                ],
                ch[
                    "Longitud"
                ]
            )

            dists.append(
                (
                    d,
                    ch[
                        "Estacion"
                    ]
                )
            )


        dists.sort()

        dist, csn_match = (
            dists[0]
        )

        metodo = (
            "COORDENADAS"
        )


    matches.append(
        {
            "CESMD_code":
                code,

            "CESMD_name":
                ce[
                    "CESMD_station_name"
                ],

            "CESMD_lat":
                ce[
                    "CESMD_lat"
                ],

            "CESMD_lon":
                ce[
                    "CESMD_lon"
                ],

            "CSN_match":
                csn_match,

            "Metodo_match":
                metodo,

            "Distancia_match_km":
                dist
        }
    )


match = pd.DataFrame(
    matches
)


print("\n" + "=" * 80)
print("MATCH CESMD ↔ CSN")
print("=" * 80)


display(
    match.round(
        {
            "CESMD_lat": 5,
            "CESMD_lon": 5,
            "Distancia_match_km": 4
        }
    )
)


# ======================================================================
# 14. EXTRAER TODOS LOS LINKS DE LA PÁGINA CESMD
# ======================================================================

links_detalle = []


for a in soup_ev.find_all(
    "a",
    href=True
):

    texto = a.get_text(
        " ",
        strip=True
    )

    href = a[
        "href"
    ]

    url = urljoin(
        url_evento,
        href
    )


    links_detalle.append(
        {
            "texto":
                texto,

            "href":
                href,

            "url":
                url
        }
    )


links_detalle = pd.DataFrame(
    links_detalle
).drop_duplicates()


RUTA_LINKS = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_links_evento.csv"
)


links_detalle.to_csv(
    RUTA_LINKS,
    index=False
)


print("\n" + "=" * 80)
print("LINKS POTENCIALMENTE ÚTILES")
print("=" * 80)


if len(links_detalle):

    mask = (
        links_detalle[
            "texto"
        ].str.contains(
            "data|download|record|processed|plot|spectrum|spectra",
            case=False,
            na=False
        )
        |
        links_detalle[
            "href"
        ].str.contains(
            "data|download|record|processed|plot|spect|select",
            case=False,
            na=False
        )
    )


    display(
        links_detalle[
            mask
        ].head(100)
    )


# ======================================================================
# 15. GUARDAR
# ======================================================================

RUTA_EST = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_5_estaciones_CLASSIC.csv"
)

RUTA_MATCH = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_match_CSN_CLASSIC.csv"
)


cesmd5.to_csv(
    RUTA_EST,
    index=False
)

match.to_csv(
    RUTA_MATCH,
    index=False
)


# ======================================================================
# 16. GATE
# ======================================================================

print("\n" + "=" * 80)
print("GATE CESMD — FALLBACK CLÁSICO")
print("=" * 80)


print(
    "Estaciones parseadas:",
    len(
        cesmd5
    )
)


if len(
    cesmd5
) == 5:

    print(
        "✅ PASS: 5/5 estaciones CESMD recuperadas."
    )

else:

    print(
        "⚠ Parser no recuperó exactamente 5."
    )

    print(
        "NO avanzar todavía a comparación espectral."
    )


if len(
    match
):

    print(
        "Máxima distancia match:",
        match[
            "Distancia_match_km"
        ].max(),
        "km"
    )


print(
    "\nPágina CESMD utilizada:"
)

print(
    url_evento
)


print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — CESMD
# RECUPERACIÓN ROBUSTA DESDE LA PÁGINA OFICIAL DEL EVENTO
#
# Corrige el fallo de detección de la celda anterior.
#
# OBJETIVOS:
# 1. aceptar directamente la URL oficial entregada por CESMD archive;
# 2. detectar cuáles de las 42 estaciones CSN aparecen en el HTML;
# 3. identificar la tabla que contiene los registros;
# 4. extraer links, formularios e inputs asociados;
# 5. preparar la auditoría de productos procesados.
#
# NO calcula Sa nueva.
# NO modifica RotD50.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import hashlib
import requests
import numpy as np
import pandas as pd

from io import StringIO
from bs4 import BeautifulSoup
from urllib.parse import urljoin


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

RUTA_CANDIDATO = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_CANDIDATO.csv"
)

assert os.path.exists(
    RUTA_CANDIDATO
)

csn = pd.read_csv(
    RUTA_CANDIDATO
)

assert len(csn) == 42
assert csn["Estacion"].nunique() == 42


URL_EVENTO = (
    "https://www.strongmotioncenter.org/"
    "cgi-bin/CESMD/iqr_dist_DM2.pl?"
    "IQRID=ValparaisoChile_24Apr2017_10008kce"
    "&SFlag=0&Flag=2"
)


session = requests.Session()

session.headers.update(
    {
        "User-Agent":
            "Mozilla/5.0 "
            "(academic reproducibility audit)"
    }
)


print("=" * 80)
print("VALPARAÍSO 2017 — CESMD / RECUPERACIÓN ROBUSTA")
print("=" * 80)

print(
    "URL oficial:",
    URL_EVENTO
)


# ======================================================================
# 2. DESCARGAR DIRECTAMENTE LA PÁGINA OFICIAL
# ======================================================================

r = session.get(
    URL_EVENTO,
    timeout=90
)


print(
    "\nHTTP:",
    r.status_code
)

print(
    "Tamaño HTML:",
    len(
        r.text
    )
)


r.raise_for_status()


assert len(
    r.text
) > 10000, (
    "La respuesta es demasiado pequeña "
    "para ser la página de datos esperada."
)


html = r.text


RUTA_HTML = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_event_OFICIAL.html"
)


with open(
    RUTA_HTML,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        html
    )


print(
    "HTML guardado:",
    RUTA_HTML
)


# ======================================================================
# 3. PARSEAR
# ======================================================================

soup = BeautifulSoup(
    html,
    "html.parser"
)


print("\nTítulo HTML:")

if soup.title:

    print(
        soup.title.get_text(
            " ",
            strip=True
        )
    )

else:

    print(
        "(sin title)"
    )


# ======================================================================
# 4. TEXTO LIMPIO — PRIMERAS LÍNEAS
# ======================================================================

texto = soup.get_text(
    "\n"
)


lineas = [
    re.sub(
        r"\s+",
        " ",
        x
    ).strip()

    for x in texto.splitlines()

    if x.strip()
]


print("\n" + "=" * 80)
print("PRIMERAS LÍNEAS DE LA PÁGINA")
print("=" * 80)


for x in lineas[:80]:

    print(
        x
    )


# ======================================================================
# 5. BUSCAR LAS 42 ESTACIONES CSN DENTRO DEL HTML
# ======================================================================

codigos_csn = sorted(
    csn[
        "Estacion"
    ]
    .astype(str)
    .str.upper()
    .unique()
)


html_upper = html.upper()


encontradas = []


for codigo in codigos_csn:

    patron = (
        r"(?<![A-Z0-9])"
        +
        re.escape(
            codigo
        )
        +
        r"(?![A-Z0-9])"
    )


    matches = list(
        re.finditer(
            patron,
            html_upper
        )
    )


    if len(matches):

        contextos = []


        for m in matches[:5]:

            ini = max(
                0,
                m.start() - 250
            )

            fin = min(
                len(html),
                m.end() + 350
            )


            contexto = html[
                ini:
                fin
            ]


            # quitar tags para lectura
            contexto_txt = BeautifulSoup(
                contexto,
                "html.parser"
            ).get_text(
                " ",
                strip=True
            )


            contexto_txt = re.sub(
                r"\s+",
                " ",
                contexto_txt
            )


            contextos.append(
                contexto_txt
            )


        encontradas.append(
            {
                "Estacion":
                    codigo,

                "N_apariciones_HTML":
                    len(matches),

                "Contexto":
                    " || ".join(
                        contextos
                    )
            }
        )


est_html = pd.DataFrame(
    encontradas
)


print("\n" + "=" * 80)
print("ESTACIONES CSN DETECTADAS EN HTML CESMD")
print("=" * 80)

print(
    "N estaciones distintas:",
    len(
        est_html
    )
)


display(
    est_html
)


# ======================================================================
# 6. LEER TODAS LAS TABLAS HTML
# ======================================================================

try:

    tablas = pd.read_html(
        StringIO(
            html
        )
    )

except Exception as e:

    print(
        "Error pd.read_html:",
        repr(e)
    )

    tablas = []


print("\n" + "=" * 80)
print("TABLAS HTML")
print("=" * 80)

print(
    "Número de tablas:",
    len(
        tablas
    )
)


resumen_tablas = []


for i, tabla in enumerate(
    tablas
):

    txt_tabla = (
        tabla
        .astype(str)
        .to_string()
        .upper()
    )


    codigos_en_tabla = []


    for codigo in codigos_csn:

        patron = (
            r"(?<![A-Z0-9])"
            +
            re.escape(
                codigo
            )
            +
            r"(?![A-Z0-9])"
        )


        if re.search(
            patron,
            txt_tabla
        ):

            codigos_en_tabla.append(
                codigo
            )


    resumen_tablas.append(
        {
            "Tabla":
                i,

            "Filas":
                tabla.shape[0],

            "Columnas":
                tabla.shape[1],

            "N_codigos_CSN":
                len(
                    codigos_en_tabla
                ),

            "Codigos_CSN":
                ",".join(
                    codigos_en_tabla
                )
        }
    )


resumen_tablas = pd.DataFrame(
    resumen_tablas
)


display(
    resumen_tablas
)


# ======================================================================
# 7. MOSTRAR TABLAS QUE CONTIENEN ESTACIONES CSN
# ======================================================================

tablas_utiles = (
    resumen_tablas.loc[
        resumen_tablas[
            "N_codigos_CSN"
        ] > 0,
        "Tabla"
    ]
    .astype(int)
    .tolist()
)


print("\n" + "=" * 80)
print("TABLAS CANDIDATAS DE REGISTROS")
print("=" * 80)


for idx in tablas_utiles:

    print(
        f"\n--- TABLA {idx} ---"
    )

    display(
        tablas[
            idx
        ]
    )


# ======================================================================
# 8. ELEGIR AUTOMÁTICAMENTE LA TABLA CON MÁS ESTACIONES
# ======================================================================

if len(
    resumen_tablas
):

    idx_mejor = int(
        resumen_tablas.sort_values(
            [
                "N_codigos_CSN",
                "Filas"
            ],
            ascending=[
                False,
                False
            ]
        ).iloc[0][
            "Tabla"
        ]
    )


    mejor = tablas[
        idx_mejor
    ]


    n_codigos_mejor = int(
        resumen_tablas.loc[
            resumen_tablas[
                "Tabla"
            ] == idx_mejor,
            "N_codigos_CSN"
        ].iloc[0]
    )


    print("\n" + "=" * 80)
    print("TABLA CESMD SELECCIONADA")
    print("=" * 80)

    print(
        "Índice:",
        idx_mejor
    )

    print(
        "Dimensión:",
        mejor.shape
    )

    print(
        "Códigos CSN detectados:",
        n_codigos_mejor
    )


    display(
        mejor
    )

else:

    idx_mejor = None
    mejor = None
    n_codigos_mejor = 0


# ======================================================================
# 9. INSPECCIONAR FILAS HTML <tr> QUE CONTIENEN CÓDIGOS CSN
# ======================================================================

filas_html = []


for tr_idx, tr in enumerate(
    soup.find_all(
        "tr"
    )
):

    texto_tr = re.sub(
        r"\s+",
        " ",
        tr.get_text(
            " ",
            strip=True
        )
    )


    texto_tr_upper = texto_tr.upper()


    encontrados_tr = []


    for codigo in codigos_csn:

        patron = (
            r"(?<![A-Z0-9])"
            +
            re.escape(
                codigo
            )
            +
            r"(?![A-Z0-9])"
        )


        if re.search(
            patron,
            texto_tr_upper
        ):

            encontrados_tr.append(
                codigo
            )


    if not encontrados_tr:

        continue


    # enlaces dentro de la fila
    links = []


    for a in tr.find_all(
        "a",
        href=True
    ):

        links.append(
            {
                "texto":
                    a.get_text(
                        " ",
                        strip=True
                    ),

                "href":
                    a[
                        "href"
                    ],

                "url":
                    urljoin(
                        URL_EVENTO,
                        a[
                            "href"
                        ]
                    )
            }
        )


    # inputs dentro de la fila
    inputs = []


    for inp in tr.find_all(
        "input"
    ):

        inputs.append(
            {
                "type":
                    inp.get(
                        "type"
                    ),

                "name":
                    inp.get(
                        "name"
                    ),

                "value":
                    inp.get(
                        "value"
                    ),

                "checked":
                    inp.has_attr(
                        "checked"
                    )
            }
        )


    filas_html.append(
        {
            "TR_index":
                tr_idx,

            "Codigos_CSN":
                ",".join(
                    encontrados_tr
                ),

            "Texto_fila":
                texto_tr,

            "Links":
                json.dumps(
                    links,
                    ensure_ascii=False
                ),

            "Inputs":
                json.dumps(
                    inputs,
                    ensure_ascii=False
                )
        }
    )


filas_html = pd.DataFrame(
    filas_html
)


print("\n" + "=" * 80)
print("FILAS HTML ASOCIADAS A ESTACIONES CSN")
print("=" * 80)


display(
    filas_html
)


# ======================================================================
# 10. TODOS LOS FORMULARIOS DE LA PÁGINA
# ======================================================================

formularios = []


for fi, form in enumerate(
    soup.find_all(
        "form"
    )
):

    action = form.get(
        "action",
        ""
    )

    method = form.get(
        "method",
        "get"
    )


    inputs = []


    for inp in form.find_all(
        "input"
    ):

        inputs.append(
            {
                "type":
                    inp.get(
                        "type"
                    ),

                "name":
                    inp.get(
                        "name"
                    ),

                "value":
                    inp.get(
                        "value"
                    ),

                "checked":
                    inp.has_attr(
                        "checked"
                    )
            }
        )


    texto_form = re.sub(
        r"\s+",
        " ",
        form.get_text(
            " ",
            strip=True
        )
    )


    formularios.append(
        {
            "Form_index":
                fi,

            "Method":
                method,

            "Action":
                action,

            "Action_abs":
                urljoin(
                    URL_EVENTO,
                    action
                ),

            "Texto":
                texto_form[:1000],

            "Inputs":
                json.dumps(
                    inputs,
                    ensure_ascii=False
                )
        }
    )


formularios = pd.DataFrame(
    formularios
)


print("\n" + "=" * 80)
print("FORMULARIOS CESMD")
print("=" * 80)


display(
    formularios
)


# ======================================================================
# 11. TODOS LOS LINKS POTENCIALMENTE ÚTILES
# ======================================================================

links = []


for a in soup.find_all(
    "a",
    href=True
):

    texto_a = re.sub(
        r"\s+",
        " ",
        a.get_text(
            " ",
            strip=True
        )
    )


    href = a[
        "href"
    ]


    links.append(
        {
            "Texto":
                texto_a,

            "Href":
                href,

            "URL":
                urljoin(
                    URL_EVENTO,
                    href
                )
        }
    )


links = pd.DataFrame(
    links
).drop_duplicates()


mask_util = (
    links[
        "Texto"
    ].str.contains(
        (
            "record|data|download|processed|"
            "raw|plot|spect|station|select|"
            "vol1|vol2|vol3"
        ),
        case=False,
        na=False
    )
    |
    links[
        "Href"
    ].str.contains(
        (
            "record|data|download|processed|"
            "raw|plot|spect|station|select|"
            "vol1|vol2|vol3"
        ),
        case=False,
        na=False
    )
)


links_utiles = links[
    mask_util
].copy()


print("\n" + "=" * 80)
print("LINKS POTENCIALMENTE ÚTILES")
print("=" * 80)


display(
    links_utiles
)


# ======================================================================
# 12. EXTRAER CÓDIGOS DEFINITIVOS CESMD ↔ CSN
# ======================================================================

if len(
    est_html
):

    estaciones_cesmd = (
        est_html[
            [
                "Estacion",
                "N_apariciones_HTML"
            ]
        ]
        .copy()
        .sort_values(
            "Estacion"
        )
        .reset_index(
            drop=True
        )
    )

else:

    estaciones_cesmd = pd.DataFrame(
        columns=[
            "Estacion",
            "N_apariciones_HTML"
        ]
    )


print("\n" + "=" * 80)
print("CANDIDATAS CESMD IDENTIFICADAS")
print("=" * 80)


display(
    estaciones_cesmd
)


# ======================================================================
# 13. GUARDAR AUDITORÍA
# ======================================================================

RUTA_EST = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_estaciones_detectadas_HTML.csv"
)

RUTA_TABLAS = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_resumen_tablas.csv"
)

RUTA_FILAS = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_filas_estaciones_HTML.csv"
)

RUTA_FORMS = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_formularios.csv"
)

RUTA_LINKS = os.path.join(
    CARPETA,
    "Valparaiso2017_CESMD_links_utiles.csv"
)


estaciones_cesmd.to_csv(
    RUTA_EST,
    index=False
)

resumen_tablas.to_csv(
    RUTA_TABLAS,
    index=False
)

filas_html.to_csv(
    RUTA_FILAS,
    index=False
)

formularios.to_csv(
    RUTA_FORMS,
    index=False
)

links_utiles.to_csv(
    RUTA_LINKS,
    index=False
)


# ======================================================================
# 14. GATE
# ======================================================================

print("\n" + "=" * 80)
print("GATE CESMD — RECUPERACIÓN HTML")
print("=" * 80)


print(
    "Estaciones CSN encontradas en HTML:",
    len(
        estaciones_cesmd
    )
)


print(
    "Máximo de códigos CSN "
    "en una tabla:",
    n_codigos_mejor
)


if len(
    estaciones_cesmd
) == 5:

    print(
        "\n✅ PASS"
    )

    print(
        "Las 5 estaciones CESMD fueron "
        "identificadas dentro de la página oficial."
    )

else:

    print(
        "\n⚠ REVISAR"
    )

    print(
        "No se detectaron exactamente 5 "
        "códigos CSN."
    )

    print(
        "Usaremos TABLAS/FILAS/FORMULARIOS "
        "para ajustar el parser."
    )


print(
    "\nV5.1 NO FUE EJECUTADO."
)


In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — CESMD
# DESCARGA AUDITADA DE PRODUCTOS PROCESADOS
#
# Estaciones CESMD confirmadas:
# VA01, VA05, MT02, VA06, MT01
#
# OBJETIVO:
# 1. solicitar Processed Data para las 5 estaciones;
# 2. inspeccionar la página Download Summary;
# 3. detectar automáticamente archivos ZIP/procesados;
# 4. descargar el paquete si CESMD lo permite;
# 5. extraerlo y mostrar inventario.
#
# NO modifica Sa CSN.
# NO recalcula RotD50.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import io
import json
import zipfile
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

CESMD_DIR = os.path.join(
    CARPETA,
    "CESMD_Procesados"
)

os.makedirs(
    CESMD_DIR,
    exist_ok=True
)


BASE_CESMD = (
    "https://www.strongmotioncenter.org/"
    "cgi-bin/CESMD/"
)

SELECTPAGE = urljoin(
    BASE_CESMD,
    "selectpage.pl"
)


IQRID = (
    "ValparaisoChile_24Apr2017_10008kce"
)


ESTACIONES = [
    "VA01",
    "VA05",
    "MT02",
    "VA06",
    "MT01"
]


NETWORK_PREFIX = "C1"


print("=" * 80)
print("VALPARAÍSO 2017 — CESMD PROCESSED DATA")
print("=" * 80)

print(
    "Estaciones:",
    ESTACIONES
)


# ======================================================================
# 2. SESIÓN
# ======================================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent":
            "Mozilla/5.0 "
            "(academic reproducibility audit)"
    }
)


# ======================================================================
# 3. CONSTRUIR PARÁMETROS DEL FORMULARIO
# ======================================================================

params = {}


for estacion in ESTACIONES:

    clave = (
        f"{IQRID}/"
        f"{NETWORK_PREFIX}{estacion}"
    )

    params[
        clave
    ] = "on"


# Solicitar SOLO processed
params[
    "Processed"
] = "on"


print("\nParámetros enviados:")

for k, v in params.items():

    print(
        k,
        "=",
        v
    )


# ======================================================================
# 4. SOLICITAR DOWNLOAD SUMMARY
# ======================================================================

r = session.get(
    SELECTPAGE,
    params=params,
    timeout=90
)


print("\n" + "=" * 80)
print("DOWNLOAD SUMMARY")
print("=" * 80)

print(
    "HTTP:",
    r.status_code
)

print(
    "URL:",
    r.url
)

print(
    "Content-Type:",
    r.headers.get(
        "Content-Type"
    )
)

print(
    "Tamaño:",
    len(
        r.content
    )
)


r.raise_for_status()


# ======================================================================
# 5. SI CESMD DEVUELVE ZIP DIRECTAMENTE
# ======================================================================

es_zip_directo = (
    r.content[:2]
    ==
    b"PK"
)


if es_zip_directo:

    print(
        "\n✅ CESMD devolvió ZIP directamente."
    )

    RUTA_ZIP = os.path.join(
        CESMD_DIR,
        "Valparaiso2017_CESMD_5estaciones_processed.zip"
    )


    with open(
        RUTA_ZIP,
        "wb"
    ) as f:

        f.write(
            r.content
        )


    with zipfile.ZipFile(
        RUTA_ZIP
    ) as z:

        z.extractall(
            CESMD_DIR
        )


    print(
        "ZIP guardado:",
        RUTA_ZIP
    )


else:

    # ==============================================================
    # 6. GUARDAR HTML DOWNLOAD SUMMARY
    # ==============================================================

    html = r.text


    RUTA_SUMMARY = os.path.join(
        CESMD_DIR,
        "Valparaiso2017_CESMD_download_summary.html"
    )


    with open(
        RUTA_SUMMARY,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            html
        )


    print(
        "\nHTML guardado:",
        RUTA_SUMMARY
    )


    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    # ==============================================================
    # 7. TEXTO RELEVANTE
    # ==============================================================

    texto = soup.get_text(
        "\n"
    )


    lineas = [
        re.sub(
            r"\s+",
            " ",
            x
        ).strip()

        for x in texto.splitlines()

        if x.strip()
    ]


    print("\n" + "=" * 80)
    print("TEXTO DOWNLOAD SUMMARY")
    print("=" * 80)


    for linea in lineas:

        if any(
            palabra.lower()
            in
            linea.lower()

            for palabra in [
                "Processed",
                "Raw",
                "Valparaiso",
                "VA01",
                "VA05",
                "VA06",
                "MT01",
                "MT02",
                "download",
                "Vol2",
                "Vol3"
            ]
        ):

            print(
                linea
            )


    # ==============================================================
    # 8. CONFIRMAR QUE ESTAMOS EN PROCESSED
    # ==============================================================

    texto_upper = texto.upper()


    assert (
        "PROCESSED"
        in
        texto_upper
    ), (
        "STOP: la página no parece "
        "corresponder a Processed Data."
    )


    print(
        "\n✓ Página Processed confirmada."
    )


    # ==============================================================
    # 9. CONFIRMAR LAS CINCO ESTACIONES
    # ==============================================================

    detectadas = []


    for estacion in ESTACIONES:

        if estacion.upper() in texto_upper:

            detectadas.append(
                estacion
            )


    print(
        "Estaciones detectadas:",
        detectadas
    )


    assert set(
        detectadas
    ) == set(
        ESTACIONES
    ), (
        "STOP: no aparecen las cinco "
        "estaciones esperadas."
    )


    # ==============================================================
    # 10. INSPECCIONAR FORMULARIOS
    # ==============================================================

    forms_info = []


    forms = soup.find_all(
        "form"
    )


    print("\n" + "=" * 80)
    print("FORMULARIOS DOWNLOAD SUMMARY")
    print("=" * 80)

    print(
        "N formularios:",
        len(
            forms
        )
    )


    for fi, form in enumerate(
        forms
    ):

        action = form.get(
            "action",
            ""
        )

        method = form.get(
            "method",
            "GET"
        ).upper()


        inputs = []


        for inp in form.find_all(
            "input"
        ):

            inputs.append(
                {
                    "type":
                        inp.get(
                            "type",
                            ""
                        ),

                    "name":
                        inp.get(
                            "name"
                        ),

                    "value":
                        inp.get(
                            "value"
                        ),

                    "checked":
                        inp.has_attr(
                            "checked"
                        )
                }
            )


        forms_info.append(
            {
                "Form":
                    fi,

                "Method":
                    method,

                "Action":
                    action,

                "Action_abs":
                    urljoin(
                        r.url,
                        action
                    ),

                "Texto":
                    re.sub(
                        r"\s+",
                        " ",
                        form.get_text(
                            " ",
                            strip=True
                        )
                    )[:1500],

                "Inputs":
                    json.dumps(
                        inputs,
                        ensure_ascii=False
                    )
            }
        )


    forms_df = pd.DataFrame(
        forms_info
    )


    display(
        forms_df
    )


    # ==============================================================
    # 11. EXTRAER TODOS LOS INPUTS CON .ZIP
    # ==============================================================

    archivos_zip = []


    for fi, form in enumerate(
        forms
    ):

        for inp in form.find_all(
            "input"
        ):

            name = inp.get(
                "name"
            )

            value = inp.get(
                "value"
            )


            texto_input = (
                str(name)
                +
                " "
                +
                str(value)
            )


            if ".zip" in texto_input.lower():

                archivos_zip.append(
                    {
                        "Form":
                            fi,

                        "type":
                            inp.get(
                                "type"
                            ),

                        "name":
                            name,

                        "value":
                            value,

                        "checked":
                            inp.has_attr(
                                "checked"
                            )
                    }
                )


    zip_df = pd.DataFrame(
        archivos_zip
    )


    print("\n" + "=" * 80)
    print("ARCHIVOS ZIP DETECTADOS")
    print("=" * 80)


    display(
        zip_df
    )


    # ==============================================================
    # 12. LINKS DIRECTOS A ZIP
    # ==============================================================

    links_zip = []


    for a in soup.find_all(
        "a",
        href=True
    ):

        href = a[
            "href"
        ]

        texto_a = a.get_text(
            " ",
            strip=True
        )


        if (
            ".zip"
            in
            href.lower()
            or
            ".zip"
            in
            texto_a.lower()
        ):

            links_zip.append(
                {
                    "Texto":
                        texto_a,

                    "Href":
                        href,

                    "URL":
                        urljoin(
                            r.url,
                            href
                        )
                }
            )


    links_zip_df = pd.DataFrame(
        links_zip
    )


    print("\n" + "=" * 80)
    print("LINKS ZIP DIRECTOS")
    print("=" * 80)


    display(
        links_zip_df
    )


    # ==============================================================
    # 13. ELEGIR FORMULARIO DE DESCARGA
    # ==============================================================

    candidatos_form = []


    for fi, form in enumerate(
        forms
    ):

        action = form.get(
            "action",
            ""
        )


        inputs = form.find_all(
            "input"
        )


        n_zip = 0


        for inp in inputs:

            texto_input = (
                str(
                    inp.get(
                        "name"
                    )
                )
                +
                " "
                +
                str(
                    inp.get(
                        "value"
                    )
                )
            )


            if ".zip" in texto_input.lower():

                n_zip += 1


        score = 0


        if (
            "download"
            in
            action.lower()
        ):

            score += 10


        if (
            "zip"
            in
            action.lower()
        ):

            score += 10


        score += (
            5
            *
            n_zip
        )


        candidatos_form.append(
            (
                score,
                fi,
                n_zip
            )
        )


    candidatos_form.sort(
        reverse=True
    )


    print(
        "\nRanking formularios:",
        candidatos_form
    )


    # ==============================================================
    # 14. INTENTAR DESCARGA AUTOMÁTICA
    # ==============================================================

    descargado = False


    if (
        len(
            candidatos_form
        )
        and
        candidatos_form[0][0] > 0
    ):

        score, fi, n_zip = (
            candidatos_form[0]
        )


        form = forms[
            fi
        ]


        action = urljoin(
            r.url,
            form.get(
                "action",
                ""
            )
        )


        method = form.get(
            "method",
            "GET"
        ).upper()


        payload = {}


        # ----------------------------------------------
        # hidden inputs
        # ----------------------------------------------

        for inp in form.find_all(
            "input"
        ):

            tipo = (
                inp.get(
                    "type",
                    ""
                )
                .lower()
            )

            name = inp.get(
                "name"
            )

            value = inp.get(
                "value",
                ""
            )


            if not name:

                continue


            if tipo == "hidden":

                payload[
                    name
                ] = value


        # ----------------------------------------------
        # seleccionar TODOS los ZIP de la página
        # ----------------------------------------------

        for inp in form.find_all(
            "input"
        ):

            name = inp.get(
                "name"
            )

            value = inp.get(
                "value",
                "on"
            )


            if not name:

                continue


            if (
                ".zip"
                in
                name.lower()
                or
                ".zip"
                in
                str(
                    value
                ).lower()
            ):

                payload[
                    name
                ] = (
                    value
                    if value
                    else "on"
                )


        print("\n" + "=" * 80)
        print("INTENTO DE DESCARGA PROCESADA")
        print("=" * 80)

        print(
            "Formulario:",
            fi
        )

        print(
            "Action:",
            action
        )

        print(
            "Method:",
            method
        )

        print(
            "ZIP fields:",
            n_zip
        )


        for k in payload:

            print(
                " ",
                k,
                "=",
                payload[k]
            )


        if method == "POST":

            rd = session.post(
                action,
                data=payload,
                timeout=180
            )

        else:

            rd = session.get(
                action,
                params=payload,
                timeout=180
            )


        print(
            "\nHTTP descarga:",
            rd.status_code
        )

        print(
            "Content-Type:",
            rd.headers.get(
                "Content-Type"
            )
        )

        print(
            "Tamaño:",
            len(
                rd.content
            )
        )


        rd.raise_for_status()


        # ----------------------------------------------
        # ¿ZIP?
        # ----------------------------------------------

        if rd.content[:2] == b"PK":

            RUTA_ZIP = os.path.join(
                CESMD_DIR,
                "Valparaiso2017_CESMD_5estaciones_processed.zip"
            )


            with open(
                RUTA_ZIP,
                "wb"
            ) as f:

                f.write(
                    rd.content
                )


            with zipfile.ZipFile(
                RUTA_ZIP
            ) as z:

                z.extractall(
                    CESMD_DIR
                )


            print(
                "\n✅ ZIP descargado:"
            )

            print(
                RUTA_ZIP
            )

            descargado = True


        else:

            # guardar respuesta para inspección
            RUTA_RESP = os.path.join(
                CESMD_DIR,
                "CESMD_respuesta_descarga.html"
            )


            with open(
                RUTA_RESP,
                "wb"
            ) as f:

                f.write(
                    rd.content
                )


            print(
                "\nLa respuesta no fue ZIP."
            )

            print(
                "Guardada para auditoría:"
            )

            print(
                RUTA_RESP
            )


# ======================================================================
# 15. INVENTARIO FINAL DE ARCHIVOS
# ======================================================================

inventario = []


for raiz, dirs, files in os.walk(
    CESMD_DIR
):

    for archivo in files:

        ruta = os.path.join(
            raiz,
            archivo
        )


        inventario.append(
            {
                "Archivo":
                    archivo,

                "Ruta":
                    ruta,

                "Tamano_bytes":
                    os.path.getsize(
                        ruta
                    )
            }
        )


inv = pd.DataFrame(
    inventario
)


print("\n" + "=" * 80)
print("INVENTARIO CESMD PROCESADO")
print("=" * 80)


display(
    inv
)


# ======================================================================
# 16. IDENTIFICAR VOL2 / VOL3 / ESPECTROS
# ======================================================================

if len(inv):

    mask_espectral = (
        inv[
            "Archivo"
        ].str.contains(
            (
                "V3|VOL3|SPECT|"
                "RESP|PSA|RSA"
            ),
            case=False,
            regex=True,
            na=False
        )
    )


    print("\n" + "=" * 80)
    print("CANDIDATOS ESPECTRALES / VOL3")
    print("=" * 80)


    display(
        inv[
            mask_espectral
        ]
    )


# ======================================================================
# 17. GUARDAR INVENTARIO
# ======================================================================

RUTA_INV = os.path.join(
    CESMD_DIR,
    "Valparaiso2017_CESMD_inventario_processed.csv"
)


inv.to_csv(
    RUTA_INV,
    index=False
)


# ======================================================================
# 18. GATE
# ======================================================================

print("\n" + "=" * 80)
print("GATE CESMD — PRODUCTOS PROCESADOS")
print("=" * 80)


if descargado:

    print(
        "✅ DESCARGA PROCESADA COMPLETADA."
    )

    print(
        "Siguiente paso:"
    )

    print(
        "identificar Vol3/espectros y "
        "comparar Sa(T=1.0 s) por componente."
    )

else:

    print(
        "⚠ La estructura fue identificada, "
        "pero el ZIP no se descargó automáticamente."
    )

    print(
        "NO es un fallo de la validación."
    )

    print(
        "Usaremos la respuesta HTML/formulario "
        "para completar la descarga."
    )


print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — CESMD
# DESCARGA REAL DE LOS 5 PAQUETES PROCESADOS
#
# Descubierto directamente del HTML oficial:
# c1va01p.zip
# c1va05p.zip
# c1mt02p.zip
# c1va06p.zip
# c1mt01p.zip
#
# Estrategia:
# 1. extraer nombres ZIP desde download_summary.html;
# 2. intentar descarga directa desde NCESMD/data;
# 3. si está protegida, reproducir llamada login_DM1.pl;
# 4. guardar/extractar cualquier ZIP válido;
# 5. inventariar Vol2 / Vol3 / espectros.
#
# NO calcula nuevas Sa.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import json
import time
import zipfile
import hashlib
import requests
import pandas as pd

from urllib.parse import unquote, urlencode


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

CESMD_DIR = os.path.join(
    CARPETA,
    "CESMD_Procesados"
)

SUMMARY = os.path.join(
    CESMD_DIR,
    "Valparaiso2017_CESMD_download_summary.html"
)

os.makedirs(
    CESMD_DIR,
    exist_ok=True
)

assert os.path.exists(
    SUMMARY
)


BASE_STATIC = (
    "https://www.strongmotioncenter.org/"
    "NCESMD/data/"
)

BASE_CGI = (
    "https://www.strongmotioncenter.org/"
    "cgi-bin/CESMD/"
)


print("=" * 80)
print("VALPARAÍSO 2017 — CESMD PROCESSED ZIP")
print("=" * 80)


# ======================================================================
# 2. LEER HTML OFICIAL Y EXTRAER ZIP
# ======================================================================

with open(
    SUMMARY,
    "r",
    encoding="iso-8859-1",
    errors="ignore"
) as f:

    html = f.read()


# patrones URL encoded:
# valparaisochile_24apr2017_10008kce%2Fc1va01p.zip=on

matches = re.findall(
    r"([A-Za-z0-9_]+)%2F"
    r"([A-Za-z0-9_-]+p\.zip)=on",
    html,
    flags=re.I
)


assert len(matches) > 0, (
    "No se encontraron ZIP procesados "
    "en el HTML."
)


pares = []


for carpeta_evento, zip_name in matches:

    item = (
        carpeta_evento.lower(),
        zip_name.lower()
    )

    if item not in pares:

        pares.append(
            item
        )


print("\nZIP descubiertos:")

for carpeta_evento, zip_name in pares:

    print(
        carpeta_evento,
        "/",
        zip_name
    )


assert len(pares) == 5, (
    f"Esperábamos 5 ZIP y encontramos {len(pares)}"
)


# ======================================================================
# 3. VALIDAR ESTACIONES ESPERADAS
# ======================================================================

esperados = {
    "c1va01p.zip",
    "c1va05p.zip",
    "c1mt02p.zip",
    "c1va06p.zip",
    "c1mt01p.zip"
}


encontrados = {
    z
    for _, z in pares
}


assert encontrados == esperados, (
    "Los ZIP descubiertos no coinciden "
    "con las cinco estaciones congeladas."
)


print(
    "\n✅ Los 5 ZIP coinciden con "
    "VA01, VA05, MT02, VA06 y MT01."
)


# ======================================================================
# 4. SESIÓN
# ======================================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent":
            "Mozilla/5.0 "
            "(academic reproducibility audit)",

        "Referer":
            (
                BASE_CGI
                +
                "selectpage.pl"
            )
    }
)


# ======================================================================
# 5. HASH
# ======================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            b = f.read(
                1024 * 1024
            )

            if not b:
                break

            h.update(b)

    return h.hexdigest()


# ======================================================================
# 6. INTENTAR DESCARGA DIRECTA
# ======================================================================

resultados = []


print("\n" + "=" * 80)
print("INTENTO 1 — RUTA DIRECTA NCESMD/data")
print("=" * 80)


for carpeta_evento, zip_name in pares:

    url = (
        BASE_STATIC
        +
        carpeta_evento
        +
        "/"
        +
        zip_name
    )


    print(
        "\nProbando:",
        url
    )


    try:

        r = session.get(
            url,
            timeout=90,
            allow_redirects=True
        )


        ctype = r.headers.get(
            "Content-Type",
            ""
        )


        es_zip = (
            r.content[:2]
            ==
            b"PK"
        )


        print(
            "HTTP:",
            r.status_code
        )

        print(
            "Content-Type:",
            ctype
        )

        print(
            "Bytes:",
            len(r.content)
        )

        print(
            "ZIP:",
            es_zip
        )


        if (
            r.status_code == 200
            and
            es_zip
        ):

            ruta_zip = os.path.join(
                CESMD_DIR,
                zip_name
            )


            with open(
                ruta_zip,
                "wb"
            ) as f:

                f.write(
                    r.content
                )


            resultados.append(
                {
                    "ZIP":
                        zip_name,

                    "Metodo":
                        "DIRECT_STATIC",

                    "HTTP":
                        r.status_code,

                    "Descargado":
                        True,

                    "Ruta":
                        ruta_zip,

                    "SHA256":
                        sha256_file(
                            ruta_zip
                        )
                }
            )


        else:

            resultados.append(
                {
                    "ZIP":
                        zip_name,

                    "Metodo":
                        "DIRECT_STATIC",

                    "HTTP":
                        r.status_code,

                    "Descargado":
                        False,

                    "Ruta":
                        None,

                    "SHA256":
                        None
                }
            )


    except Exception as e:

        print(
            "Error:",
            repr(e)
        )


        resultados.append(
            {
                "ZIP":
                    zip_name,

                "Metodo":
                    "DIRECT_STATIC",

                "HTTP":
                    None,

                "Descargado":
                    False,

                "Ruta":
                    None,

                "SHA256":
                    None
            }
        )


    time.sleep(
        1
    )


res = pd.DataFrame(
    resultados
)


print("\nResultado intento directo:")

display(
    res
)


# ======================================================================
# 7. ¿CUÁNTOS FALTAN?
# ======================================================================

descargados = set(
    res.loc[
        res[
            "Descargado"
        ],
        "ZIP"
    ]
)


faltantes = sorted(
    esperados
    -
    descargados
)


print(
    "\nZIP descargados directamente:",
    len(descargados)
)

print(
    "ZIP faltantes:",
    faltantes
)


# ======================================================================
# 8. SI FALTAN, REPRODUCIR login_DM1.pl
# ======================================================================

if faltantes:

    print("\n" + "=" * 80)
    print("INTENTO 2 — login_DM1.pl")
    print("=" * 80)


    params_login = {}


    for carpeta_evento, zip_name in pares:

        params_login[
            f"{carpeta_evento}/{zip_name}"
        ] = "on"


    LOGIN_URL = (
        BASE_CGI
        +
        "login_DM1.pl"
    )


    rl = session.get(
        LOGIN_URL,
        params=params_login,
        timeout=90,
        allow_redirects=True
    )


    print(
        "HTTP:",
        rl.status_code
    )

    print(
        "URL final:",
        rl.url
    )

    print(
        "Content-Type:",
        rl.headers.get(
            "Content-Type"
        )
    )

    print(
        "Content-Disposition:",
        rl.headers.get(
            "Content-Disposition"
        )
    )

    print(
        "Bytes:",
        len(
            rl.content
        )
    )


    # --------------------------------------------------------------
    # Caso A: ZIP directo
    # --------------------------------------------------------------

    if rl.content[:2] == b"PK":

        ruta = os.path.join(
            CESMD_DIR,
            "CESMD_processed_login_response.zip"
        )


        with open(
            ruta,
            "wb"
        ) as f:

            f.write(
                rl.content
            )


        print(
            "\n✅ login_DM1 devolvió ZIP."
        )


    # --------------------------------------------------------------
    # Caso B: HTML
    # --------------------------------------------------------------

    else:

        ruta_html_login = os.path.join(
            CESMD_DIR,
            "CESMD_login_DM1_response.html"
        )


        with open(
            ruta_html_login,
            "wb"
        ) as f:

            f.write(
                rl.content
            )


        texto_login = rl.text


        print(
            "\nRespuesta HTML guardada:"
        )

        print(
            ruta_html_login
        )


        print("\n" + "=" * 80)
        print("DIAGNÓSTICO login_DM1.pl")
        print("=" * 80)


        indicadores_login = [
            "login",
            "password",
            "username",
            "email",
            "account",
            "sign in"
        ]


        encontrados_login = [
            x
            for x in indicadores_login
            if x in texto_login.lower()
        ]


        print(
            "Indicadores de autenticación:",
            encontrados_login
        )


        # Mostrar texto limpio relevante
        from bs4 import BeautifulSoup

        soup = BeautifulSoup(
            texto_login,
            "html.parser"
        )


        lineas = [
            re.sub(
                r"\s+",
                " ",
                x
            ).strip()

            for x in soup.get_text(
                "\n"
            ).splitlines()

            if x.strip()
        ]


        print(
            "\nPrimeras líneas:"
        )


        for x in lineas[:80]:

            print(
                x
            )


# ======================================================================
# 9. EXTRAER TODOS LOS ZIP DESCARGADOS
# ======================================================================

print("\n" + "=" * 80)
print("EXTRACCIÓN DE ZIP")
print("=" * 80)


zip_locales = [
    os.path.join(
        CESMD_DIR,
        f
    )

    for f in os.listdir(
        CESMD_DIR
    )

    if f.lower().endswith(
        ".zip"
    )
]


print(
    "ZIP locales:",
    len(zip_locales)
)


EXTRACT_DIR = os.path.join(
    CESMD_DIR,
    "extraidos"
)

os.makedirs(
    EXTRACT_DIR,
    exist_ok=True
)


for ruta_zip in zip_locales:

    print(
        "\nExtrayendo:",
        os.path.basename(
            ruta_zip
        )
    )


    try:

        with zipfile.ZipFile(
            ruta_zip
        ) as z:

            nombres = z.namelist()

            print(
                "Archivos internos:",
                len(nombres)
            )


            for n in nombres[:30]:

                print(
                    " ",
                    n
                )


            z.extractall(
                EXTRACT_DIR
            )


    except zipfile.BadZipFile:

        print(
            "⚠ No es ZIP válido:",
            ruta_zip
        )


# ======================================================================
# 10. INVENTARIO COMPLETO
# ======================================================================

inventario = []


for raiz, dirs, files in os.walk(
    CESMD_DIR
):

    for archivo in files:

        ruta = os.path.join(
            raiz,
            archivo
        )


        inventario.append(
            {
                "Archivo":
                    archivo,

                "Extension":
                    os.path.splitext(
                        archivo
                    )[1].lower(),

                "Ruta":
                    ruta,

                "Tamano_bytes":
                    os.path.getsize(
                        ruta
                    )
            }
        )


inv = pd.DataFrame(
    inventario
)


print("\n" + "=" * 80)
print("INVENTARIO FINAL CESMD")
print("=" * 80)


display(
    inv
)


# ======================================================================
# 11. BUSCAR ARCHIVOS ESPECTRALES / VOL3
# ======================================================================

if len(inv):

    mask = (
        inv[
            "Archivo"
        ].str.contains(
            (
                r"v3|vol3|"
                r"spect|response|resp|"
                r"sa|psa|rsp"
            ),
            case=False,
            regex=True,
            na=False
        )
    )


    espectrales = inv[
        mask
    ].copy()


else:

    espectrales = pd.DataFrame()


print("\n" + "=" * 80)
print("CANDIDATOS VOL3 / ESPECTROS")
print("=" * 80)


display(
    espectrales
)


# ======================================================================
# 12. GUARDAR INVENTARIO
# ======================================================================

RUTA_INV = os.path.join(
    CESMD_DIR,
    "Valparaiso2017_CESMD_inventario_descarga_real.csv"
)


inv.to_csv(
    RUTA_INV,
    index=False
)


# ======================================================================
# 13. GATE
# ======================================================================

n_zip_validos = 0


for ruta_zip in zip_locales:

    try:

        with zipfile.ZipFile(
            ruta_zip
        ) as z:

            z.testzip()

        n_zip_validos += 1

    except Exception:

        pass


print("\n" + "=" * 80)
print("GATE CESMD — DESCARGA REAL")
print("=" * 80)

print(
    "ZIP válidos descargados:",
    n_zip_validos
)


if n_zip_validos >= 5:

    print(
        "✅ PASS"
    )

    print(
        "Los cinco paquetes procesados "
        "están disponibles localmente."
    )

    print(
        "\nSiguiente paso:"
    )

    print(
        "parsear Vol3 y comparar "
        "Sa(T=1.0 s) por componente."
    )


elif n_zip_validos > 0:

    print(
        "⚠ PARCIAL"
    )

    print(
        "Se descargó al menos un paquete, "
        "pero faltan estaciones."
    )


else:

    print(
        "⚠ AUTENTICACIÓN / RUTA PROTEGIDA"
    )

    print(
        "CESMD exige completar el acceso "
        "antes de servir los ZIP."
    )

    print(
        "No cambia ningún resultado de la tesis."
    )


print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — AUDITORÍA ESPECTRAL CESMD DEFINITIVA
#
# Compara nuestro PSA(T=1.0 s, 5%) por componente
# contra los archivos CESMD .rs2 procesados.
#
# CESMD disponible:
# VA01: HNE + HNN
# VA05: HNE + HNN
# MT02: HNE + HNN
# VA06: HNE solamente
# MT01: HNE + HNN
#
# TOTAL ESPERADO = 9 horizontales.
#
# IMPORTANTE:
# - Nuestro objetivo es PSEUDO spectral acceleration (PSA).
# - CESMD .rs2 entrega Pseudo-Velocity Response Spectrum.
# - PSA = omega * PSV.
# - Como control independiente también se calcula PSA desde Sd:
#       PSA = omega^2 * Sd.
#
# NO usa Absolute Acceleration Spectrum como equivalente al PSA.
# NO modifica nuestros 42 RotD50.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import hashlib
import numpy as np
import pandas as pd


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

CESMD_DIR = os.path.join(
    CARPETA,
    "CESMD_Procesados",
    "extraidos"
)

RUTA_NUESTRO = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_CANDIDATO.csv"
)

assert os.path.isdir(CESMD_DIR)
assert os.path.exists(RUTA_NUESTRO)


T_OBJ = 1.0
XI_OBJ = 0.05

G_CM_S2 = 980.665

OMEGA = (
    2.0
    *
    np.pi
    /
    T_OBJ
)


df_nuestro = pd.read_csv(
    RUTA_NUESTRO
)

assert len(df_nuestro) == 42


print("=" * 80)
print("VALPARAÍSO 2017 — AUDITORÍA ESPECTRAL CESMD")
print("=" * 80)

print("T objetivo          :", T_OBJ, "s")
print("Amortiguamiento     :", XI_OBJ)
print("Objetivo             : PSA")
print("Comparaciones esperadas: 9 horizontales")


# ======================================================================
# 2. INVENTARIO .rs2
# ======================================================================

rs2_files = []


for raiz, dirs, files in os.walk(
    CESMD_DIR
):

    for f in files:

        if f.lower().endswith(
            ".rs2"
        ):

            rs2_files.append(
                os.path.join(
                    raiz,
                    f
                )
            )


rs2_files = sorted(
    rs2_files
)


print("\nArchivos .rs2 encontrados:")
print(len(rs2_files))


for x in rs2_files:

    print(
        os.path.basename(
            x
        )
    )


# Deben existir 14:
# 3+3+3+2+3
assert len(rs2_files) == 14, (
    f"Esperábamos 14 rs2; "
    f"se encontraron {len(rs2_files)}."
)


# ======================================================================
# 3. SELECCIONAR SOLO HORIZONTALES
# ======================================================================

horiz = []


for path in rs2_files:

    name = os.path.basename(
        path
    ).upper()


    if ".HNE." in name:

        componente = "E"

    elif ".HNN." in name:

        componente = "N"

    else:

        continue


    estacion = (
        name.split(".")[0]
    )


    horiz.append(
        {
            "Estacion":
                estacion,

            "Componente":
                componente,

            "Archivo":
                path
        }
    )


horiz = pd.DataFrame(
    horiz
)


print("\n" + "=" * 80)
print("HORIZONTALES CESMD")
print("=" * 80)


display(
    horiz
)


assert len(horiz) == 9, (
    f"Esperábamos 9 horizontales; "
    f"se encontraron {len(horiz)}."
)


esperados = {
    ("VA01", "E"),
    ("VA01", "N"),
    ("VA05", "E"),
    ("VA05", "N"),
    ("MT02", "E"),
    ("MT02", "N"),
    ("VA06", "E"),
    ("MT01", "E"),
    ("MT01", "N")
}


hallados = set(
    zip(
        horiz[
            "Estacion"
        ],
        horiz[
            "Componente"
        ]
    )
)


assert hallados == esperados, (
    "Las componentes CESMD no coinciden "
    "con las 9 esperadas."
)


print(
    "✅ Inventario horizontal CESMD = 9/9."
)


# ======================================================================
# 4. UTILIDADES
# ======================================================================

FLOAT_RE = re.compile(
    r"""
    [+-]?
    (?:
        (?:\d+\.\d*)
        |
        (?:\.\d+)
        |
        (?:\d+)
    )
    (?:[EeDd][+-]?\d+)?
    """,
    re.VERBOSE
)


def numeros_linea(linea):

    valores = []

    for x in FLOAT_RE.findall(
        linea
    ):

        try:

            valores.append(
                float(
                    x.replace(
                        "D",
                        "E"
                    ).replace(
                        "d",
                        "e"
                    )
                )
            )

        except Exception:

            pass


    return valores


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            b = f.read(
                1024 * 1024
            )

            if not b:
                break

            h.update(b)


    return h.hexdigest()


# ======================================================================
# 5. PARSER SMC / RS2
# ======================================================================
#
# Estructura documentada:
#
# - unidades
# - NDAMP, NPER, TFLAG
# - lista damping
# - lista periodos
#
# Para cada damping:
# - Relative Displacement
# - Relative Velocity
# - Pseudo-Velocity
# - Absolute Acceleration
#
# El parser NO presupone 91 periodos.
# Lee NDAMP y NPER directamente del archivo.
# ======================================================================

def parse_rs2(path):

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        lines = [
            x.rstrip("\n")
            for x in f
        ]


    # --------------------------------------------------------------
    # A. Encontrar línea NDAMP NPER TFLAG
    # --------------------------------------------------------------

    idx_meta = None
    ndamp = None
    nper = None
    tflag = None


    for i in range(
        len(lines)
    ):

        # Buscamos una línea de 3 enteros,
        # después de encontrar referencia a cm/sec.
        nums = numeros_linea(
            lines[i]
        )


        if len(nums) != 3:

            continue


        enteros = all(
            np.isclose(
                x,
                round(x)
            )
            for x in nums
        )


        if not enteros:

            continue


        a, b, c = [
            int(
                round(x)
            )
            for x in nums
        ]


        # restricciones físicas/formato
        if (
            1 <= a <= 20
            and
            5 <= b <= 1000
            and
            c in [0, 1]
        ):

            # revisar contexto previo:
            contexto = " ".join(
                lines[
                    max(
                        0,
                        i - 3
                    ):
                    i
                ]
            ).lower()


            if (
                "cm/sec"
                in contexto
                or
                "inch/sec"
                in contexto
                or
                "response"
                in "\n".join(
                    lines[:15]
                ).lower()
            ):

                idx_meta = i
                ndamp = a
                nper = b
                tflag = c

                break


    assert idx_meta is not None, (
        f"No se encontró NDAMP/NPER/TFLAG: {path}"
    )


    # --------------------------------------------------------------
    # B. Leer dampings + periodos
    # --------------------------------------------------------------

    valores = []

    j = (
        idx_meta
        +
        1
    )


    while (
        j < len(lines)
        and
        len(valores)
        <
        (
            ndamp
            +
            nper
        )
    ):

        # detener si aparece un encabezado espectral
        if "response spectrum" in lines[
            j
        ].lower():

            break


        valores.extend(
            numeros_linea(
                lines[j]
            )
        )

        j += 1


    assert len(valores) >= (
        ndamp
        +
        nper
    ), (
        f"No se pudieron recuperar damping+periodos: {path}"
    )


    damping = np.asarray(
        valores[
            :ndamp
        ],
        dtype=float
    )


    periodos = np.asarray(
        valores[
            ndamp:
            ndamp + nper
        ],
        dtype=float
    )


    assert len(damping) == ndamp
    assert len(periodos) == nper

    assert np.isfinite(
        damping
    ).all()

    assert np.isfinite(
        periodos
    ).all()


    # --------------------------------------------------------------
    # C. Parsear todos los bloques espectrales
    # --------------------------------------------------------------

    bloques = []


    tipo_patterns = {
        "SD":
            "relative displacement response spectrum",

        "SV":
            "relative velocity response spectrum",

        "PSV":
            "pseudo-velocity response spectrum",

        "ABS_ACC":
            "absolute acceleration response spectrum"
    }


    for i, linea in enumerate(
        lines
    ):

        low = linea.lower()


        tipo = None


        for clave, patron in (
            tipo_patterns.items()
        ):

            if patron in low:

                tipo = clave
                break


        if tipo is None:

            continue


        # damping escrito en el header
        nums_header = numeros_linea(
            linea
        )


        if len(nums_header) == 0:

            damp_header = np.nan

        else:

            damp_header = float(
                nums_header[-1]
            )


        # Normalizar:
        # puede aparecer como 0.05 o 5.0 %
        if (
            np.isfinite(
                damp_header
            )
            and
            damp_header > 1.0
        ):

            damp_frac = (
                damp_header
                /
                100.0
            )

        else:

            damp_frac = damp_header


        # leer exactamente NPER valores siguientes
        vals = []

        k = (
            i
            +
            1
        )


        while (
            k < len(lines)
            and
            len(vals) < nper
        ):

            # si por error llegamos al siguiente header
            if (
                "response spectrum"
                in
                lines[k].lower()
            ):

                break


            vals.extend(
                numeros_linea(
                    lines[k]
                )
            )

            k += 1


        assert len(vals) >= nper, (
            f"Bloque incompleto {tipo}: {path}"
        )


        vals = np.asarray(
            vals[:nper],
            dtype=float
        )


        bloques.append(
            {
                "tipo":
                    tipo,

                "damping_header":
                    damp_header,

                "damping_frac":
                    damp_frac,

                "values":
                    vals
            }
        )


    assert len(bloques) > 0, (
        f"No se encontraron espectros: {path}"
    )


    return {
        "ndamp":
            ndamp,

        "nper":
            nper,

        "tflag":
            tflag,

        "damping_list":
            damping,

        "periodos":
            periodos,

        "bloques":
            bloques
    }


# ======================================================================
# 6. EXTRAER T=1 s, 5 %
# ======================================================================

def obtener_bloque(
    parsed,
    tipo,
    xi_obj=0.05
):

    cand = [
        b
        for b in parsed[
            "bloques"
        ]
        if b[
            "tipo"
        ] == tipo
    ]


    assert len(cand) > 0, (
        f"No existe bloque {tipo}"
    )


    # Seleccionar damping más cercano a 5%.
    # Algunas implementaciones imprimen header ambiguo,
    # así que también verificamos contra damping_list.
    dist = [
        abs(
            b[
                "damping_frac"
            ]
            -
            xi_obj
        )
        if np.isfinite(
            b[
                "damping_frac"
            ]
        )
        else np.inf

        for b in cand
    ]


    idx = int(
        np.argmin(
            dist
        )
    )


    elegido = cand[
        idx
    ]


    assert abs(
        elegido[
            "damping_frac"
        ]
        -
        xi_obj
    ) < 1e-4, (
        f"No se encontró exactamente 5% "
        f"para {tipo}; encontrado "
        f"{elegido['damping_frac']}"
    )


    return elegido


def valor_T1(
    periodos,
    valores
):

    idx = np.where(
        np.isclose(
            periodos,
            T_OBJ,
            rtol=0,
            atol=1e-8
        )
    )[0]


    assert len(idx) == 1, (
        "CESMD no contiene exactamente "
        "T=1.0 s en la grilla."
    )


    i = int(
        idx[0]
    )


    return (
        i,
        float(
            valores[
                i
            ]
        )
    )


# ======================================================================
# 7. PARSEAR LAS 9 COMPONENTES
# ======================================================================

filas = []


for _, row in horiz.iterrows():

    estacion = row[
        "Estacion"
    ]

    comp = row[
        "Componente"
    ]

    path = row[
        "Archivo"
    ]


    print(
        f"\nProcesando {estacion}-{comp}: "
        f"{os.path.basename(path)}"
    )


    p = parse_rs2(
        path
    )


    print(
        "  NDAMP:",
        p[
            "ndamp"
        ]
    )

    print(
        "  NPER :",
        p[
            "nper"
        ]
    )

    print(
        "  Damping list:",
        p[
            "damping_list"
        ]
    )


    # ----------------------------------------------------------
    # PSV 5%
    # ----------------------------------------------------------

    b_psv = obtener_bloque(
        p,
        "PSV",
        XI_OBJ
    )


    idx_T,
    PSV_cm_s = valor_T1(
        p[
            "periodos"
        ],
        b_psv[
            "values"
        ]
    )


    # PSA a partir de PSV:
    # cm/s * 1/s = cm/s²
    PSA_from_PSV_g = (
        OMEGA
        *
        PSV_cm_s
        /
        G_CM_S2
    )


    # ----------------------------------------------------------
    # Sd 5% como auditoría independiente
    # ----------------------------------------------------------

    b_sd = obtener_bloque(
        p,
        "SD",
        XI_OBJ
    )


    _, Sd_cm = valor_T1(
        p[
            "periodos"
        ],
        b_sd[
            "values"
        ]
    )


    PSA_from_Sd_g = (
        OMEGA**2
        *
        Sd_cm
        /
        G_CM_S2
    )


    # ----------------------------------------------------------
    # Absolute acceleration — SOLO REFERENCIA
    # ----------------------------------------------------------

    b_abs = obtener_bloque(
        p,
        "ABS_ACC",
        XI_OBJ
    )


    _, AbsAcc_cm_s2 = valor_T1(
        p[
            "periodos"
        ],
        b_abs[
            "values"
        ]
    )


    AbsAcc_g = (
        AbsAcc_cm_s2
        /
        G_CM_S2
    )


    # ----------------------------------------------------------
    # Nuestro resultado
    # ----------------------------------------------------------

    nrow = df_nuestro[
        df_nuestro[
            "Estacion"
        ].str.upper()
        ==
        estacion.upper()
    ]


    assert len(nrow) == 1


    if comp == "E":

        nuestro_g = float(
            nrow.iloc[
                0
            ][
                "Sa_E_T1_g"
            ]
        )


    elif comp == "N":

        nuestro_g = float(
            nrow.iloc[
                0
            ][
                "Sa_N_T1_g"
            ]
        )


    else:

        raise ValueError(
            comp
        )


    # ----------------------------------------------------------
    # Diferencias
    # ----------------------------------------------------------

    delta_pct = (
        100.0
        *
        (
            nuestro_g
            /
            PSA_from_PSV_g
            -
            1.0
        )
    )


    ln_ratio = np.log(
        nuestro_g
        /
        PSA_from_PSV_g
    )


    cesmd_internal_pct = (
        100.0
        *
        (
            PSA_from_Sd_g
            /
            PSA_from_PSV_g
            -
            1.0
        )
    )


    filas.append(
        {
            "Estacion":
                estacion,

            "Componente":
                comp,

            "Archivo_RS2":
                os.path.basename(
                    path
                ),

            "SHA256_RS2":
                sha256_file(
                    path
                ),

            "NDAMP":
                p[
                    "ndamp"
                ],

            "NPER":
                p[
                    "nper"
                ],

            "T_s":
                T_OBJ,

            "Xi":
                XI_OBJ,

            "PSV_CESMD_cm_s":
                PSV_cm_s,

            "Sd_CESMD_cm":
                Sd_cm,

            "PSA_CESMD_from_PSV_g":
                PSA_from_PSV_g,

            "PSA_CESMD_from_Sd_g":
                PSA_from_Sd_g,

            "CESMD_Sd_vs_PSV_pct":
                cesmd_internal_pct,

            "AbsoluteAcc_CESMD_g":
                AbsAcc_g,

            "PSA_nuestro_g":
                nuestro_g,

            "Delta_nuestro_vs_CESMD_pct":
                delta_pct,

            "ln_ratio_nuestro_CESMD":
                ln_ratio
        }
    )


audit = pd.DataFrame(
    filas
)


# ======================================================================
# 8. RESULTADOS
# ======================================================================

print("\n" + "=" * 80)
print("COMPARACIÓN PSA T=1.0 s — CESMD VS NUESTRO")
print("=" * 80)


display(
    audit[
        [
            "Estacion",
            "Componente",
            "PSV_CESMD_cm_s",
            "Sd_CESMD_cm",
            "PSA_CESMD_from_PSV_g",
            "PSA_CESMD_from_Sd_g",
            "CESMD_Sd_vs_PSV_pct",
            "PSA_nuestro_g",
            "Delta_nuestro_vs_CESMD_pct",
            "AbsoluteAcc_CESMD_g"
        ]
    ].round(
        7
    )
)


# ======================================================================
# 9. CONSISTENCIA INTERNA CESMD
# ======================================================================

print("\n" + "=" * 80)
print("CONSISTENCIA INTERNA CESMD")
print("=" * 80)


print(
    "Máximo |PSA(Sd) vs PSA(PSV)| [%]:",
    audit[
        "CESMD_Sd_vs_PSV_pct"
    ].abs().max()
)


# ======================================================================
# 10. RESUMEN DE DIFERENCIAS
# ======================================================================

abs_delta = audit[
    "Delta_nuestro_vs_CESMD_pct"
].abs()


print("\n" + "=" * 80)
print("RESUMEN AUDITORÍA EXTERNA")
print("=" * 80)


print(
    "N componentes comparadas:",
    len(
        audit
    )
)

print(
    "Media Δ [%]:",
    audit[
        "Delta_nuestro_vs_CESMD_pct"
    ].mean()
)

print(
    "Mediana Δ [%]:",
    audit[
        "Delta_nuestro_vs_CESMD_pct"
    ].median()
)

print(
    "Media |Δ| [%]:",
    abs_delta.mean()
)

print(
    "Máximo |Δ| [%]:",
    abs_delta.max()
)


idx_max = abs_delta.idxmax()


print(
    "\nMayor diferencia:"
)

display(
    audit.loc[
        [
            idx_max
        ],
        [
            "Estacion",
            "Componente",
            "PSA_CESMD_from_PSV_g",
            "PSA_nuestro_g",
            "Delta_nuestro_vs_CESMD_pct"
        ]
    ]
)


# ======================================================================
# 11. RESULTADOS POR ESTACIÓN
# ======================================================================

print("\n" + "=" * 80)
print("RESUMEN POR ESTACIÓN")
print("=" * 80)


por_estacion = (
    audit
    .groupby(
        "Estacion",
        as_index=False
    )
    .agg(
        N_componentes=(
            "Componente",
            "count"
        ),

        Media_abs_delta_pct=(
            "Delta_nuestro_vs_CESMD_pct",
            lambda x:
                np.mean(
                    np.abs(
                        x
                    )
                )
        ),

        Max_abs_delta_pct=(
            "Delta_nuestro_vs_CESMD_pct",
            lambda x:
                np.max(
                    np.abs(
                        x
                    )
                )
        )
    )
)


display(
    por_estacion.round(
        5
    )
)


# ======================================================================
# 12. GATES DE AUDITORÍA
# ======================================================================
#
# Estos límites NO seleccionan registros.
# Son umbrales de concordancia de implementación.
#
# <= 2%  : concordancia muy alta
# <= 5%  : concordancia aceptable para fuentes/procesamientos distintos
# > 5%   : investigar antes de congelar Sa
#
# ======================================================================

max_internal = audit[
    "CESMD_Sd_vs_PSV_pct"
].abs().max()


max_external = abs_delta.max()


gate_cesmd_internal = (
    max_internal
    <
    0.5
)


gate_external_2 = (
    max_external
    <=
    2.0
)


gate_external_5 = (
    max_external
    <=
    5.0
)


print("\n" + "=" * 80)
print("GATE AUDITORÍA ESPECTRAL CESMD")
print("=" * 80)


print(
    "CESMD Sd ↔ PSV:",
    "PASS"
    if gate_cesmd_internal
    else "FAIL"
)


if gate_external_2:

    print(
        "Nuestro PSA ↔ CESMD:",
        "PASS — concordancia ≤2%"
    )


elif gate_external_5:

    print(
        "Nuestro PSA ↔ CESMD:",
        "PASS CON OBSERVACIÓN — concordancia ≤5%"
    )


else:

    print(
        "Nuestro PSA ↔ CESMD:",
        "REVISAR — alguna diferencia >5%"
    )


# ======================================================================
# 13. GUARDAR
# ======================================================================

RUTA_AUDIT = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_CESMD_PSA_T1_componentes.csv"
)


RUTA_RESUMEN = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_CESMD_PSA_T1_resumen_estacion.csv"
)


audit.to_csv(
    RUTA_AUDIT,
    index=False
)


por_estacion.to_csv(
    RUTA_RESUMEN,
    index=False
)


print("\nArchivos:")

print(
    RUTA_AUDIT
)

print(
    RUTA_RESUMEN
)


# ======================================================================
# 14. DECISIÓN
# ======================================================================

print("\n" + "=" * 80)
print("DECISIÓN PRE-FREEZE Sa")
print("=" * 80)


if (
    gate_cesmd_internal
    and
    gate_external_5
):

    print(
        "✅ AUDITORÍA ESPECTRAL SUPERADA."
    )

    print(
        "La implementación del PSA(T=1 s, 5%) "
        "queda externamente respaldada."
    )

    print(
        "\nTodavía NO ejecutar V5.1 en esta celda."
    )

    print(
        "Siguiente etapa:"
    )

    print(
        "congelar formalmente los 42 "
        "Sa_RotD50_T1_g y generar hash final."
    )


else:

    print(
        "⚠ DETENER."
    )

    print(
        "Investigar diferencias antes de "
        "congelar los 42 Sa."
    )


print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# CONTINUACIÓN — AUDITORÍA ESPECTRAL CESMD
# Corrige únicamente la asignación idx_T, PSV_cm_s.
# Requiere haber ejecutado previamente las funciones:
# parse_rs2, obtener_bloque, valor_T1, sha256_file
# y tener definidos: horiz, df_nuestro, T_OBJ, XI_OBJ,
# OMEGA y G_CM_S2.
# ======================================================================

import os
import numpy as np
import pandas as pd


# Comprobación de que seguimos en el mismo runtime
for nombre in [
    "horiz",
    "df_nuestro",
    "parse_rs2",
    "obtener_bloque",
    "valor_T1",
    "sha256_file",
    "OMEGA",
    "G_CM_S2"
]:
    assert nombre in globals(), (
        f"Falta {nombre}. "
        "Si reiniciaste el runtime, ejecuta la celda completa corregida."
    )


filas = []


for _, row in horiz.iterrows():

    estacion = row["Estacion"]
    comp = row["Componente"]
    path = row["Archivo"]

    print(
        f"\nProcesando {estacion}-{comp}: "
        f"{os.path.basename(path)}"
    )


    p = parse_rs2(
        path
    )


    print(
        "  NDAMP:",
        p["ndamp"]
    )

    print(
        "  NPER :",
        p["nper"]
    )

    print(
        "  Damping list:",
        p["damping_list"]
    )


    # ==============================================================
    # PSV — 5 %
    # ==============================================================

    b_psv = obtener_bloque(
        p,
        "PSV",
        XI_OBJ
    )


    # CORRECCIÓN AQUÍ:
    idx_T, PSV_cm_s = valor_T1(
        p["periodos"],
        b_psv["values"]
    )


    PSA_from_PSV_g = (
        OMEGA
        *
        PSV_cm_s
        /
        G_CM_S2
    )


    # ==============================================================
    # Sd — 5 %
    # ==============================================================

    b_sd = obtener_bloque(
        p,
        "SD",
        XI_OBJ
    )


    _, Sd_cm = valor_T1(
        p["periodos"],
        b_sd["values"]
    )


    PSA_from_Sd_g = (
        OMEGA**2
        *
        Sd_cm
        /
        G_CM_S2
    )


    # ==============================================================
    # Absolute Acceleration — solo referencia
    # ==============================================================

    b_abs = obtener_bloque(
        p,
        "ABS_ACC",
        XI_OBJ
    )


    _, AbsAcc_cm_s2 = valor_T1(
        p["periodos"],
        b_abs["values"]
    )


    AbsAcc_g = (
        AbsAcc_cm_s2
        /
        G_CM_S2
    )


    # ==============================================================
    # Nuestro PSA
    # ==============================================================

    nrow = df_nuestro[
        df_nuestro[
            "Estacion"
        ].str.upper()
        ==
        estacion.upper()
    ]


    assert len(nrow) == 1


    if comp == "E":

        nuestro_g = float(
            nrow.iloc[0][
                "Sa_E_T1_g"
            ]
        )

    elif comp == "N":

        nuestro_g = float(
            nrow.iloc[0][
                "Sa_N_T1_g"
            ]
        )

    else:

        raise ValueError(
            comp
        )


    # ==============================================================
    # Diferencias
    # ==============================================================

    delta_pct = (
        100.0
        *
        (
            nuestro_g
            /
            PSA_from_PSV_g
            -
            1.0
        )
    )


    ln_ratio = np.log(
        nuestro_g
        /
        PSA_from_PSV_g
    )


    cesmd_internal_pct = (
        100.0
        *
        (
            PSA_from_Sd_g
            /
            PSA_from_PSV_g
            -
            1.0
        )
    )


    filas.append(
        {
            "Estacion":
                estacion,

            "Componente":
                comp,

            "Archivo_RS2":
                os.path.basename(
                    path
                ),

            "SHA256_RS2":
                sha256_file(
                    path
                ),

            "NDAMP":
                p["ndamp"],

            "NPER":
                p["nper"],

            "T_s":
                T_OBJ,

            "Xi":
                XI_OBJ,

            "Indice_T1":
                idx_T,

            "PSV_CESMD_cm_s":
                PSV_cm_s,

            "Sd_CESMD_cm":
                Sd_cm,

            "PSA_CESMD_from_PSV_g":
                PSA_from_PSV_g,

            "PSA_CESMD_from_Sd_g":
                PSA_from_Sd_g,

            "CESMD_Sd_vs_PSV_pct":
                cesmd_internal_pct,

            "AbsoluteAcc_CESMD_g":
                AbsAcc_g,

            "PSA_nuestro_g":
                nuestro_g,

            "Delta_nuestro_vs_CESMD_pct":
                delta_pct,

            "ln_ratio_nuestro_CESMD":
                ln_ratio
        }
    )


audit = pd.DataFrame(
    filas
)


# ======================================================================
# RESULTADOS
# ======================================================================

print("\n" + "=" * 80)
print("COMPARACIÓN PSA T=1.0 s — CESMD VS NUESTRO")
print("=" * 80)


display(
    audit[
        [
            "Estacion",
            "Componente",
            "PSV_CESMD_cm_s",
            "Sd_CESMD_cm",
            "PSA_CESMD_from_PSV_g",
            "PSA_CESMD_from_Sd_g",
            "CESMD_Sd_vs_PSV_pct",
            "PSA_nuestro_g",
            "Delta_nuestro_vs_CESMD_pct",
            "AbsoluteAcc_CESMD_g"
        ]
    ].round(7)
)


# ======================================================================
# CONSISTENCIA INTERNA CESMD
# ======================================================================

max_internal = (
    audit[
        "CESMD_Sd_vs_PSV_pct"
    ]
    .abs()
    .max()
)


print("\n" + "=" * 80)
print("CONSISTENCIA INTERNA CESMD")
print("=" * 80)

print(
    "Máximo |PSA(Sd) vs PSA(PSV)| [%]:",
    max_internal
)


# ======================================================================
# RESUMEN EXTERNO
# ======================================================================

abs_delta = (
    audit[
        "Delta_nuestro_vs_CESMD_pct"
    ]
    .abs()
)


print("\n" + "=" * 80)
print("RESUMEN AUDITORÍA EXTERNA")
print("=" * 80)

print(
    "N componentes comparadas:",
    len(audit)
)

print(
    "Media Δ [%]:",
    audit[
        "Delta_nuestro_vs_CESMD_pct"
    ].mean()
)

print(
    "Mediana Δ [%]:",
    audit[
        "Delta_nuestro_vs_CESMD_pct"
    ].median()
)

print(
    "Media |Δ| [%]:",
    abs_delta.mean()
)

print(
    "Máximo |Δ| [%]:",
    abs_delta.max()
)


idx_max = abs_delta.idxmax()


print("\nMayor diferencia:")

display(
    audit.loc[
        [idx_max],
        [
            "Estacion",
            "Componente",
            "PSA_CESMD_from_PSV_g",
            "PSA_nuestro_g",
            "Delta_nuestro_vs_CESMD_pct"
        ]
    ]
)


# ======================================================================
# RESUMEN POR ESTACIÓN
# ======================================================================

por_estacion = (
    audit
    .groupby(
        "Estacion",
        as_index=False
    )
    .agg(
        N_componentes=(
            "Componente",
            "count"
        ),

        Media_abs_delta_pct=(
            "Delta_nuestro_vs_CESMD_pct",
            lambda x:
                np.mean(
                    np.abs(x)
                )
        ),

        Max_abs_delta_pct=(
            "Delta_nuestro_vs_CESMD_pct",
            lambda x:
                np.max(
                    np.abs(x)
                )
        )
    )
)


print("\n" + "=" * 80)
print("RESUMEN POR ESTACIÓN")
print("=" * 80)

display(
    por_estacion.round(5)
)


# ======================================================================
# GATES
# ======================================================================

max_external = (
    abs_delta.max()
)


gate_cesmd_internal = (
    max_internal
    <
    0.5
)


gate_external_2 = (
    max_external
    <=
    2.0
)


gate_external_5 = (
    max_external
    <=
    5.0
)


print("\n" + "=" * 80)
print("GATE AUDITORÍA ESPECTRAL CESMD")
print("=" * 80)


print(
    "CESMD Sd ↔ PSV:",
    "PASS"
    if gate_cesmd_internal
    else "FAIL"
)


if gate_external_2:

    print(
        "Nuestro PSA ↔ CESMD: "
        "PASS — concordancia ≤2%"
    )

elif gate_external_5:

    print(
        "Nuestro PSA ↔ CESMD: "
        "PASS CON OBSERVACIÓN — concordancia ≤5%"
    )

else:

    print(
        "Nuestro PSA ↔ CESMD: "
        "REVISAR — alguna diferencia >5%"
    )


# ======================================================================
# GUARDAR
# ======================================================================

RUTA_AUDIT = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_CESMD_PSA_T1_componentes.csv"
)

RUTA_RESUMEN = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_CESMD_PSA_T1_resumen_estacion.csv"
)


audit.to_csv(
    RUTA_AUDIT,
    index=False
)

por_estacion.to_csv(
    RUTA_RESUMEN,
    index=False
)


# ======================================================================
# DECISIÓN
# ======================================================================

print("\n" + "=" * 80)
print("DECISIÓN PRE-FREEZE Sa")
print("=" * 80)


if (
    gate_cesmd_internal
    and
    gate_external_5
):

    print(
        "✅ AUDITORÍA ESPECTRAL SUPERADA."
    )

    print(
        "La implementación de "
        "PSA(T=1 s, 5%) queda respaldada."
    )

    print(
        "\nSiguiente etapa:"
    )

    print(
        "congelar formalmente los 42 "
        "Sa_RotD50_T1_g."
    )

else:

    print(
        "⚠ DETENER."
    )

    print(
        "Investigar diferencias antes "
        "de congelar Sa."
    )


print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — AUDITORÍA DE ESCALA CSN vs CESMD
#
# Objetivo:
# demostrar si la discrepancia espectral ~factor 2 ya está presente
# en la aceleración de entrada.
#
# Compara:
#   1. PGA del registro CSN
#   2. "pk acc" del acelerograma procesado CESMD _a.smc
#   3. ratio CESMD/CSN en PGA
#   4. ratio CESMD/CSN en PSA(T=1 s)
#
# NO corrige ninguna señal.
# NO congela Sa.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import numpy as np
import pandas as pd


# ======================================================================
# 1. RUTAS
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

CESMD_DIR = os.path.join(
    CARPETA,
    "CESMD_Procesados",
    "extraidos"
)

RUTA_EVENTO = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_CANDIDATO.csv"
)

RUTA_AUDIT_ESPECTRAL = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_CESMD_PSA_T1_componentes.csv"
)

assert os.path.exists(RUTA_EVENTO)
assert os.path.exists(RUTA_AUDIT_ESPECTRAL)
assert os.path.isdir(CESMD_DIR)


df = pd.read_csv(
    RUTA_EVENTO
)

spec = pd.read_csv(
    RUTA_AUDIT_ESPECTRAL
)


G = 9.80665


# ======================================================================
# 2. COMPONENTES A COMPARAR
# ======================================================================

comparables = [
    ("MT01", "E"),
    ("MT01", "N"),
    ("MT02", "E"),
    ("MT02", "N"),
    ("VA01", "E"),
    ("VA01", "N"),
    ("VA05", "E"),
    ("VA05", "N"),
    ("VA06", "E")
]


# ======================================================================
# 3. BUSCAR ACELEROGRAMA CESMD _a.smc
# ======================================================================

def buscar_smc(estacion, componente):

    canal = (
        "HNE"
        if componente == "E"
        else "HNN"
    )

    encontrados = []


    for raiz, dirs, files in os.walk(
        CESMD_DIR
    ):

        for f in files:

            fu = f.upper()

            if (
                fu.startswith(
                    estacion.upper()
                    +
                    "."
                    +
                    canal
                    +
                    "."
                )
                and
                fu.endswith(
                    "_A.SMC"
                )
            ):

                encontrados.append(
                    os.path.join(
                        raiz,
                        f
                    )
                )


    assert len(encontrados) == 1, (
        f"{estacion}-{componente}: "
        f"se encontraron {len(encontrados)} "
        "archivos _a.smc"
    )


    return encontrados[0]


# ======================================================================
# 4. EXTRAER "pk acc" DEL HEADER SMC
# ======================================================================

def leer_pk_acc_smc(path):

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        lineas = [
            f.readline().rstrip("\n")
            for _ in range(11)
        ]


    texto = "\n".join(
        lineas
    )


    # Formato SMC:
    # epicentral dist= ... pk acc = ...
    m = re.search(
        r"pk\s*acc\s*=\s*"
        r"([+-]?"
        r"(?:\d+(?:\.\d*)?|\.\d+)"
        r"(?:[EeDd][+-]?\d+)?)",
        texto,
        flags=re.I
    )


    assert m is not None, (
        f"No se encontró pk acc en {path}\n"
        f"{texto}"
    )


    valor = float(
        m.group(1)
        .replace(
            "D",
            "E"
        )
        .replace(
            "d",
            "e"
        )
    )


    return valor, lineas


# ======================================================================
# 5. CARGAR SEÑAL CSN
# ======================================================================

def cargar_txt(path):

    x = np.loadtxt(
        path,
        comments="#"
    )

    x = np.asarray(
        x,
        dtype=float
    ).reshape(-1)

    assert np.isfinite(
        x
    ).all()


    return x


# ======================================================================
# 6. COMPARAR
# ======================================================================

filas = []


for estacion, comp in comparables:

    row = df[
        df[
            "Estacion"
        ].str.upper()
        ==
        estacion.upper()
    ]


    assert len(row) == 1

    row = row.iloc[0]


    # --------------------------------------------------------------
    # CSN
    # --------------------------------------------------------------

    if comp == "E":

        path_csn = row[
            "Archivo_E"
        ]

        i0 = int(
            row[
                "Indice_inicio_E"
            ]
        )

    else:

        path_csn = row[
            "Archivo_N"
        ]

        i0 = int(
            row[
                "Indice_inicio_N"
            ]
        )


    n = int(
        row[
            COL_N_COMUN
        ]
    )


    x = cargar_txt(
        path_csn
    )


    x = x[
        i0:
        i0 + n
    ]


    # CSN está en m/s²
    pga_csn_g = (
        np.max(
            np.abs(
                x
            )
        )
        /
        G
    )


    # --------------------------------------------------------------
    # CESMD corrected acceleration
    # --------------------------------------------------------------

    path_smc = buscar_smc(
        estacion,
        comp
    )


    pk_acc_cesmd_cm_s2, headers = (
        leer_pk_acc_smc(
            path_smc
        )
    )


    pga_cesmd_g = (
        abs(
            pk_acc_cesmd_cm_s2
        )
        /
        (
            G
            *
            100.0
        )
    )


    ratio_pga = (
        pga_cesmd_g
        /
        pga_csn_g
    )


    # --------------------------------------------------------------
    # PSA T=1 s ya auditado
    # --------------------------------------------------------------

    s = spec[
        (
            spec[
                "Estacion"
            ].str.upper()
            ==
            estacion.upper()
        )
        &
        (
            spec[
                "Componente"
            ].str.upper()
            ==
            comp.upper()
        )
    ]


    assert len(s) == 1

    s = s.iloc[0]


    psa_csn = float(
        s[
            "PSA_nuestro_g"
        ]
    )

    psa_cesmd = float(
        s[
            "PSA_CESMD_from_PSV_g"
        ]
    )


    ratio_psa = (
        psa_cesmd
        /
        psa_csn
    )


    diferencia_ratios_pct = (
        100.0
        *
        (
            ratio_psa
            /
            ratio_pga
            -
            1.0
        )
    )


    filas.append(
        {
            "Estacion":
                estacion,

            "Componente":
                comp,

            "PGA_CSN_g":
                pga_csn_g,

            "PGA_CESMD_g":
                pga_cesmd_g,

            "Ratio_PGA_CESMD_CSN":
                ratio_pga,

            "PSA_CSN_T1_g":
                psa_csn,

            "PSA_CESMD_T1_g":
                psa_cesmd,

            "Ratio_PSA_CESMD_CSN":
                ratio_psa,

            "Delta_ratio_PSA_vs_PGA_pct":
                diferencia_ratios_pct,

            "Archivo_CSN":
                path_csn,

            "Archivo_CESMD":
                path_smc
        }
    )


audit = pd.DataFrame(
    filas
)


# ======================================================================
# 7. RESULTADOS
# ======================================================================

print("=" * 80)
print("AUDITORÍA DE ESCALA — CSN vs CESMD")
print("=" * 80)


display(
    audit[
        [
            "Estacion",
            "Componente",
            "PGA_CSN_g",
            "PGA_CESMD_g",
            "Ratio_PGA_CESMD_CSN",
            "PSA_CSN_T1_g",
            "PSA_CESMD_T1_g",
            "Ratio_PSA_CESMD_CSN",
            "Delta_ratio_PSA_vs_PGA_pct"
        ]
    ].round(6)
)


# ======================================================================
# 8. CLASIFICAR EL FACTOR
# ======================================================================

def clasificar(r):

    if abs(r - 1.0) <= 0.05:

        return "FACTOR_1"

    elif abs(r - 2.0) <= 0.10:

        return "FACTOR_2"

    else:

        return "OTRO"


audit[
    "Clase_escala_PGA"
] = audit[
    "Ratio_PGA_CESMD_CSN"
].apply(
    clasificar
)


audit[
    "Clase_escala_PSA"
] = audit[
    "Ratio_PSA_CESMD_CSN"
].apply(
    clasificar
)


print("\n" + "=" * 80)
print("CLASIFICACIÓN DE ESCALA")
print("=" * 80)


display(
    audit[
        [
            "Estacion",
            "Componente",
            "Ratio_PGA_CESMD_CSN",
            "Clase_escala_PGA",
            "Ratio_PSA_CESMD_CSN",
            "Clase_escala_PSA"
        ]
    ].round(5)
)


print(
    "\nPGA:"
)

print(
    audit[
        "Clase_escala_PGA"
    ].value_counts()
)


print(
    "\nPSA:"
)

print(
    audit[
        "Clase_escala_PSA"
    ].value_counts()
)


# ======================================================================
# 9. VA01 VS INFORME OFICIAL CSN
# ======================================================================

va01 = audit[
    audit[
        "Estacion"
    ]
    ==
    "VA01"
]


max_pga_csn_va01 = va01[
    "PGA_CSN_g"
].max()


max_pga_cesmd_va01 = va01[
    "PGA_CESMD_g"
].max()


print("\n" + "=" * 80)
print("CONTROL VA01 — PUBLICACIÓN OFICIAL CSN")
print("=" * 80)

print(
    "Máximo horizontal CSN calculado:",
    max_pga_csn_va01,
    "g"
)

print(
    "Máximo horizontal CESMD:",
    max_pga_cesmd_va01,
    "g"
)

print(
    "CSN publicó para el evento:"
)

print(
    "aprox. 0.39–0.40 g en Valparaíso."
)


# ======================================================================
# 10. GATE DE DIAGNÓSTICO
# ======================================================================

mismo_patron = (
    audit[
        "Clase_escala_PGA"
    ].equals(
        audit[
            "Clase_escala_PSA"
        ]
    )
)


print("\n" + "=" * 80)
print("GATE DIAGNÓSTICO DE ESCALA")
print("=" * 80)


print(
    "Patrón PGA = patrón PSA:",
    mismo_patron
)


print(
    "Factor 2 en PGA:",
    int(
        (
            audit[
                "Clase_escala_PGA"
            ]
            ==
            "FACTOR_2"
        ).sum()
    ),
    "/ 9"
)


print(
    "Factor 1 en PGA:",
    int(
        (
            audit[
                "Clase_escala_PGA"
            ]
            ==
            "FACTOR_1"
        ).sum()
    ),
    "/ 9"
)


if mismo_patron:

    print(
        "\n✅ La discrepancia espectral "
        "ya está presente en la amplitud "
        "de aceleración."
    )

    print(
        "No se origina en el algoritmo "
        "de respuesta espectral."
    )

else:

    print(
        "\n⚠ El patrón PGA y PSA no coincide."
    )

    print(
        "Requiere investigación adicional."
    )


# ======================================================================
# 11. GUARDAR
# ======================================================================

RUTA_OUT = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_escala_CSN_vs_CESMD.csv"
)


audit.to_csv(
    RUTA_OUT,
    index=False
)


print("\nArchivo:")
print(RUTA_OUT)

print("\nV5.1 NO FUE EJECUTADO.")

In [ ]:
# ======================================================================
# RESOLVER COLUMNAS DUPLICADAS GENERADAS POR merge()
# ======================================================================

print("Columnas relacionadas con N_comun:")
print(
    [
        c
        for c in df.columns
        if "N_comun" in c
    ]
)

print("\nColumnas relacionadas con Fs_Hz:")
print(
    [
        c
        for c in df.columns
        if "Fs_Hz" in c
    ]
)


def resolver_columna_duplicada(
    dataframe,
    base
):

    # Caso simple: existe con el nombre original
    if base in dataframe.columns:

        return base


    candidatos = [
        c
        for c in dataframe.columns
        if c.startswith(
            base + "_"
        )
    ]


    assert len(candidatos) > 0, (
        f"No se encontró ninguna columna para {base}"
    )


    # Si existen dos copias por merge, deben ser idénticas
    if len(candidatos) >= 2:

        ref = dataframe[
            candidatos[0]
        ].to_numpy()


        for c in candidatos[1:]:

            otra = dataframe[
                c
            ].to_numpy()


            assert np.allclose(
                ref,
                otra,
                rtol=0,
                atol=0,
                equal_nan=True
            ), (
                f"STOP: {base} no coincide entre "
                f"{candidatos[0]} y {c}"
            )


        print(
            f"✅ {base}: columnas duplicadas "
            f"{candidatos} son idénticas."
        )


    return candidatos[0]


COL_N_COMUN = resolver_columna_duplicada(
    df,
    "N_comun"
)


print(
    "\nColumna utilizada para N_comun:",
    COL_N_COMUN
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — CSN RAW BINARY AUDIT
#
# Objetivo:
# - descargar el archivo binario ORIGINAL del evento desde CSN;
# - identificar formato;
# - extraerlo;
# - localizar archivos de VA01, VA05, MT01, MT02 y VA06;
#
# NO modifica Sa.
# NO corrige escalas.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import io
import gzip
import tarfile
import zipfile
import hashlib
import requests
import pandas as pd


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

RAW_DIR = os.path.join(
    CARPETA,
    "CSN_RAW"
)

os.makedirs(
    RAW_DIR,
    exist_ok=True
)


EVENT_HASH = (
    "6c5752b76db0f46280949a798662a224"
)

URL_RAW = (
    "https://evtdb.csn.uchile.cl/raw/"
    +
    EVENT_HASH
)


print("=" * 80)
print("VALPARAÍSO 2017 — CSN RAW BINARY")
print("=" * 80)

print("URL:", URL_RAW)


# ======================================================================
# 2. DESCARGAR
# ======================================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent":
            "Mozilla/5.0 "
            "(academic reproducibility audit)"
    }
)


r = session.get(
    URL_RAW,
    timeout=180,
    allow_redirects=True
)


print("\nHTTP:", r.status_code)
print("URL final:", r.url)
print("Content-Type:", r.headers.get("Content-Type"))
print("Content-Disposition:", r.headers.get("Content-Disposition"))
print("Bytes:", len(r.content))
print("Primeros 32 bytes:", r.content[:32])


r.raise_for_status()


# ======================================================================
# 3. NOMBRE DE ARCHIVO
# ======================================================================

cd = r.headers.get(
    "Content-Disposition",
    ""
)


m = re.search(
    r'filename="?([^";]+)',
    cd,
    flags=re.I
)


if m:

    nombre = m.group(1)

else:

    nombre = "20170424_213828_CSN_RAW.bin"


RUTA_RAW = os.path.join(
    RAW_DIR,
    nombre
)


with open(
    RUTA_RAW,
    "wb"
) as f:

    f.write(
        r.content
    )


print("\nGuardado:")
print(RUTA_RAW)


# ======================================================================
# 4. SHA256
# ======================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for bloque in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(bloque)

    return h.hexdigest()


print(
    "SHA256:",
    sha256_file(
        RUTA_RAW
    )
)


# ======================================================================
# 5. DETECTAR FORMATO
# ======================================================================

es_zip = zipfile.is_zipfile(
    RUTA_RAW
)

es_tar = tarfile.is_tarfile(
    RUTA_RAW
)


print("\n" + "=" * 80)
print("DETECCIÓN DE FORMATO")
print("=" * 80)

print("ZIP :", es_zip)
print("TAR :", es_tar)


EXTRACT_DIR = os.path.join(
    RAW_DIR,
    "extraidos"
)

os.makedirs(
    EXTRACT_DIR,
    exist_ok=True
)


# ======================================================================
# 6. EXTRAER
# ======================================================================

if es_zip:

    print("\nExtrayendo ZIP...")

    with zipfile.ZipFile(
        RUTA_RAW
    ) as z:

        nombres = z.namelist()

        print(
            "Archivos internos:",
            len(nombres)
        )

        for x in nombres[:100]:

            print(" ", x)

        z.extractall(
            EXTRACT_DIR
        )


elif es_tar:

    print("\nExtrayendo TAR...")

    with tarfile.open(
        RUTA_RAW
    ) as t:

        miembros = t.getmembers()

        print(
            "Archivos internos:",
            len(miembros)
        )

        for x in miembros[:100]:

            print(
                " ",
                x.name
            )

        t.extractall(
            EXTRACT_DIR
        )


else:

    print(
        "\n⚠ El endpoint devolvió un archivo "
        "que no es ZIP/TAR."
    )

    print(
        "Se conservará intacto para identificar "
        "su formato por cabecera/extensión."
    )


# ======================================================================
# 7. INVENTARIO
# ======================================================================

inventario = []


for raiz, dirs, files in os.walk(
    RAW_DIR
):

    for archivo in files:

        ruta = os.path.join(
            raiz,
            archivo
        )

        inventario.append(
            {
                "Archivo":
                    archivo,

                "Extension":
                    os.path.splitext(
                        archivo
                    )[1].lower(),

                "Ruta":
                    ruta,

                "Tamano_bytes":
                    os.path.getsize(
                        ruta
                    )
            }
        )


inv = pd.DataFrame(
    inventario
)


print("\n" + "=" * 80)
print("INVENTARIO CSN RAW")
print("=" * 80)

display(
    inv
)


# ======================================================================
# 8. BUSCAR LAS 5 ESTACIONES CESMD DE CONTROL
# ======================================================================

ESTACIONES = [
    "VA01",
    "VA05",
    "MT01",
    "MT02",
    "VA06"
]


relevantes = []


for _, row in inv.iterrows():

    nombre_upper = row[
        "Archivo"
    ].upper()


    encontrados = [
        est
        for est in ESTACIONES
        if est in nombre_upper
    ]


    if encontrados:

        r2 = row.to_dict()

        r2[
            "Estaciones_detectadas"
        ] = ",".join(
            encontrados
        )

        relevantes.append(
            r2
        )


rel = pd.DataFrame(
    relevantes
)


print("\n" + "=" * 80)
print("ARCHIVOS RAW DE LAS 5 ESTACIONES")
print("=" * 80)


if len(rel):

    display(
        rel
    )

else:

    print(
        "No aparecen códigos de estación "
        "directamente en los nombres."
    )


# ======================================================================
# 9. EXTENSIONES
# ======================================================================

print("\n" + "=" * 80)
print("EXTENSIONES EN EL RAW")
print("=" * 80)


if len(inv):

    print(
        inv[
            "Extension"
        ].value_counts(
            dropna=False
        )
    )


# ======================================================================
# 10. GUARDAR INVENTARIO
# ======================================================================

RUTA_INV = os.path.join(
    RAW_DIR,
    "Valparaiso2017_CSN_RAW_inventario.csv"
)


inv.to_csv(
    RUTA_INV,
    index=False
)


print("\nInventario guardado:")
print(RUTA_INV)


# ======================================================================
# 11. GATE
# ======================================================================

print("\n" + "=" * 80)
print("GATE CSN RAW — ETAPA 1")
print("=" * 80)


if es_zip or es_tar:

    print(
        "✅ Archivo bruto descargado "
        "y estructura extraída."
    )

else:

    print(
        "⚠ Archivo bruto descargado, "
        "pero formato aún por identificar."
    )


print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — IDENTIFICACIÓN FORENSE DE GUC___063
#
# OBJETIVOS:
# 1. identificar el formato real de 20170424_213828.GUC___063;
# 2. buscar códigos VA01, VA05, MT01, MT02 y VA06 dentro del binario;
# 3. probar lectura automática con ObsPy;
# 4. si ObsPy reconoce el archivo, inventariar trazas SIN modificar datos;
# 5. NO aplicar respuesta instrumental todavía.
#
# NO modifica los archivos.
# NO calcula Sa.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import sys
import hashlib
import subprocess
import importlib.util
import numpy as np
import pandas as pd


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

RAW_DIR = os.path.join(
    BASE,
    "Valparaiso2017",
    "CSN_RAW",
    "extraidos"
)

RUTA_GUC = os.path.join(
    RAW_DIR,
    "20170424_213828.GUC___063"
)

assert os.path.exists(RUTA_GUC)

ESTACIONES = [
    "VA01",
    "VA05",
    "MT01",
    "MT02",
    "VA06"
]


print("=" * 80)
print("VALPARAÍSO 2017 — IDENTIFICACIÓN DE GUC___063")
print("=" * 80)

print("Archivo:")
print(RUTA_GUC)

print(
    "Tamaño:",
    os.path.getsize(RUTA_GUC),
    "bytes"
)


# ======================================================================
# 2. SHA256
# ======================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for bloque in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(bloque)

    return h.hexdigest()


sha = sha256_file(
    RUTA_GUC
)


print(
    "\nSHA256:",
    sha
)


# ======================================================================
# 3. COMANDO UNIX file
# ======================================================================

print("\n" + "=" * 80)
print("IDENTIFICACIÓN CON 'file'")
print("=" * 80)


resultado_file = subprocess.run(
    [
        "file",
        "-b",
        RUTA_GUC
    ],
    capture_output=True,
    text=True
)


print(
    resultado_file.stdout.strip()
)


# ======================================================================
# 4. PRIMEROS BYTES
# ======================================================================

with open(
    RUTA_GUC,
    "rb"
) as f:

    primeros = f.read(
        256
    )


print("\n" + "=" * 80)
print("PRIMEROS 256 BYTES — HEX")
print("=" * 80)

print(
    primeros.hex(
        " "
    )
)


print("\n" + "=" * 80)
print("PRIMEROS 256 BYTES — REPRESENTACIÓN")
print("=" * 80)

print(
    repr(
        primeros
    )
)


# ======================================================================
# 5. STRINGS ASCII INICIALES
# ======================================================================

print("\n" + "=" * 80)
print("STRINGS ASCII")
print("=" * 80)


res_strings = subprocess.run(
    [
        "strings",
        "-n",
        "4",
        RUTA_GUC
    ],
    capture_output=True,
    text=True
)


strings_txt = res_strings.stdout


lineas_strings = [
    x.strip()
    for x in strings_txt.splitlines()
    if x.strip()
]


print(
    "Total strings detectados:",
    len(lineas_strings)
)


print("\nPrimeros 100:")

for x in lineas_strings[:100]:

    print(
        x
    )


# ======================================================================
# 6. BUSCAR CÓDIGOS DE LAS 5 ESTACIONES
# ======================================================================

print("\n" + "=" * 80)
print("BÚSQUEDA DE CÓDIGOS DE ESTACIÓN")
print("=" * 80)


with open(
    RUTA_GUC,
    "rb"
) as f:

    contenido = f.read()


hallazgos = []


for est in ESTACIONES:

    patron = est.encode(
        "ascii"
    )

    posiciones = []

    inicio = 0

    while True:

        pos = contenido.find(
            patron,
            inicio
        )

        if pos == -1:
            break

        posiciones.append(
            pos
        )

        inicio = pos + 1


    hallazgos.append(
        {
            "Estacion":
                est,

            "N_apariciones_ASCII":
                len(posiciones),

            "Offsets_bytes":
                ",".join(
                    str(x)
                    for x in posiciones[:20]
                )
        }
    )


hallazgos_df = pd.DataFrame(
    hallazgos
)


display(
    hallazgos_df
)


# ======================================================================
# 7. CONTEXTO BINARIO ALREDEDOR DE CADA CÓDIGO
# ======================================================================

print("\n" + "=" * 80)
print("CONTEXTO ALREDEDOR DE LOS CÓDIGOS")
print("=" * 80)


for _, row in hallazgos_df.iterrows():

    est = row[
        "Estacion"
    ]

    patron = est.encode(
        "ascii"
    )

    pos = contenido.find(
        patron
    )


    print(
        f"\n--- {est} ---"
    )


    if pos == -1:

        print(
            "No aparece literalmente en ASCII."
        )

        continue


    ini = max(
        0,
        pos - 100
    )

    fin = min(
        len(contenido),
        pos + 200
    )


    bloque = contenido[
        ini:
        fin
    ]


    print(
        "Offset:",
        pos
    )

    print(
        "HEX:"
    )

    print(
        bloque.hex(
            " "
        )
    )

    print(
        "\nASCII aproximado:"
    )

    printable = "".join(
        chr(b)
        if 32 <= b <= 126
        else "."
        for b in bloque
    )

    print(
        printable
    )


# ======================================================================
# 8. INSTALAR / IMPORTAR OBSPY
# ======================================================================

print("\n" + "=" * 80)
print("PRUEBA OBSPY")
print("=" * 80)


if importlib.util.find_spec(
    "obspy"
) is None:

    print(
        "ObsPy no instalado. Instalando..."
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "obspy"
        ]
    )


from obspy import read


print(
    "ObsPy disponible."
)


# ======================================================================
# 9. LECTURA AUTOMÁTICA — SIN FORZAR FORMATO
# ======================================================================

st_auto = None
error_auto = None


try:

    st_auto = read(
        RUTA_GUC
    )

except Exception as e:

    error_auto = repr(
        e
    )


print("\nLectura automática:")


if st_auto is not None:

    print(
        "✅ ObsPy reconoció el archivo."
    )

    print(
        "Número de trazas:",
        len(
            st_auto
        )
    )

else:

    print(
        "❌ No reconocido automáticamente."
    )

    print(
        error_auto
    )


# ======================================================================
# 10. SI FUNCIONA, INVENTARIAR TRAZAS
# ======================================================================

trazas = []


if st_auto is not None:

    for i, tr in enumerate(
        st_auto
    ):

        trazas.append(
            {
                "Indice":
                    i,

                "Network":
                    tr.stats.network,

                "Station":
                    tr.stats.station,

                "Location":
                    tr.stats.location,

                "Channel":
                    tr.stats.channel,

                "Starttime":
                    str(
                        tr.stats.starttime
                    ),

                "Endtime":
                    str(
                        tr.stats.endtime
                    ),

                "Sampling_rate_Hz":
                    float(
                        tr.stats.sampling_rate
                    ),

                "Npts":
                    int(
                        tr.stats.npts
                    ),

                "Min_raw":
                    float(
                        np.min(
                            tr.data
                        )
                    ),

                "Max_raw":
                    float(
                        np.max(
                            tr.data
                        )
                    ),

                "dtype":
                    str(
                        tr.data.dtype
                    )
            }
        )


trazas_df = pd.DataFrame(
    trazas
)


print("\n" + "=" * 80)
print("INVENTARIO DE TRAZAS OBSPY")
print("=" * 80)


if len(trazas_df):

    display(
        trazas_df
    )

else:

    print(
        "Sin trazas disponibles."
    )


# ======================================================================
# 11. BUSCAR LAS 5 ESTACIONES EN LAS TRAZAS
# ======================================================================

print("\n" + "=" * 80)
print("TRAZAS DE LAS 5 ESTACIONES")
print("=" * 80)


if len(trazas_df):

    mask = (
        trazas_df[
            "Station"
        ]
        .astype(str)
        .str.upper()
        .isin(
            ESTACIONES
        )
    )


    control = trazas_df[
        mask
    ].copy()


    display(
        control
    )


    print(
        "Estaciones encontradas:",
        sorted(
            control[
                "Station"
            ]
            .astype(str)
            .str.upper()
            .unique()
        )
    )


else:

    control = pd.DataFrame()


# ======================================================================
# 12. SI AUTO FALLA — PROBAR FORMATOS ESPECÍFICOS
# ======================================================================
#
# Solo lectura. No transformamos muestras.
#
# GCF = Güralp Compressed Format
# KINEMETRICS_EVT = Kinemetrics EVT
# MSEED = MiniSEED
#
# ======================================================================

pruebas = []


if st_auto is None:

    formatos = [
        "GCF",
        "KINEMETRICS_EVT",
        "MSEED"
    ]


    for formato in formatos:

        try:

            st = read(
                RUTA_GUC,
                format=formato
            )


            pruebas.append(
                {
                    "Formato":
                        formato,

                    "PASS":
                        True,

                    "N_trazas":
                        len(st),

                    "Error":
                        None
                }
            )


        except Exception as e:

            pruebas.append(
                {
                    "Formato":
                        formato,

                    "PASS":
                        False,

                    "N_trazas":
                        0,

                    "Error":
                        repr(e)
                }
            )


pruebas_df = pd.DataFrame(
    pruebas
)


print("\n" + "=" * 80)
print("PRUEBAS DE FORMATO EXPLÍCITO")
print("=" * 80)


if len(pruebas_df):

    display(
        pruebas_df
    )

else:

    print(
        "No necesarias: lectura automática exitosa."
    )


# ======================================================================
# 13. GUARDAR AUDITORÍA
# ======================================================================

RUTA_HALLAZGOS = os.path.join(
    RAW_DIR,
    "Valparaiso2017_GUC063_busqueda_estaciones.csv"
)

RUTA_TRAZAS = os.path.join(
    RAW_DIR,
    "Valparaiso2017_GUC063_inventario_trazas.csv"
)


hallazgos_df.to_csv(
    RUTA_HALLAZGOS,
    index=False
)


trazas_df.to_csv(
    RUTA_TRAZAS,
    index=False
)


# ======================================================================
# 14. GATE
# ======================================================================

print("\n" + "=" * 80)
print("GATE IDENTIFICACIÓN GUC___063")
print("=" * 80)


if st_auto is not None:

    estaciones_obspy = set(
        trazas_df[
            "Station"
        ]
        .astype(str)
        .str.upper()
    )


    presentes = [
        est
        for est in ESTACIONES
        if est in estaciones_obspy
    ]


    print(
        "Formato legible por ObsPy: SÍ"
    )

    print(
        "Estaciones de control encontradas:",
        presentes
    )


    if set(
        presentes
    ) == set(
        ESTACIONES
    ):

        print(
            "✅ PASS: las cinco estaciones "
            "están dentro de GUC___063."
        )

    else:

        print(
            "⚠ Lectura exitosa, pero no aparecen "
            "las cinco estaciones esperadas."
        )


else:

    print(
        "Formato legible automáticamente: NO"
    )

    print(
        "Revisar identificación explícita."
    )


print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017
# AUDITORÍA DE ESCALA DEFINITIVA:
# RAW MiniSEED -> CSN texto -> StationXML histórico
#
# OBJETIVO:
# Decidir objetivamente si la escala de los textos CSN o una escala
# aproximadamente 2x es consistente con la respuesta instrumental
# histórica de la red C1 para el 24/04/2017.
#
# NO corrige señales.
# NO modifica RotD50.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import re
import numpy as np
import pandas as pd

from obspy import read, UTCDateTime
from obspy.clients.fdsn import Client


# ======================================================================
# 1. CONFIGURACIÓN
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

RUTA_GUC = os.path.join(
    CARPETA,
    "CSN_RAW",
    "extraidos",
    "20170424_213828.GUC___063"
)

RUTA_CANDIDATO = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_CANDIDATO.csv"
)

RUTA_ESCALA = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_escala_CSN_vs_CESMD.csv"
)

META_DIR = os.path.join(
    CARPETA,
    "StationXML_2017"
)

os.makedirs(
    META_DIR,
    exist_ok=True
)

assert os.path.exists(RUTA_GUC)
assert os.path.exists(RUTA_CANDIDATO)


ESTACIONES = [
    "VA01",
    "VA05",
    "MT01",
    "MT02",
    "VA06"
]


COMPARABLES = [
    ("VA01", "E"),
    ("VA01", "N"),
    ("VA05", "E"),
    ("VA05", "N"),
    ("MT01", "E"),
    ("MT01", "N"),
    ("MT02", "E"),
    ("MT02", "N"),
    ("VA06", "E")
]


T_EVENTO = UTCDateTime(
    "2017-04-24T21:38:28"
)

G = 9.80665


print("=" * 80)
print("VALPARAÍSO 2017 — RAW / CSN / STATIONXML")
print("=" * 80)


# ======================================================================
# 2. LEER RAW MINISEED
# ======================================================================

st = read(
    RUTA_GUC
)

print(
    "Formato ObsPy:",
    sorted(
        set(
            getattr(
                tr.stats,
                "_format",
                "?"
            )
            for tr in st
        )
    )
)

print(
    "Trazas:",
    len(st)
)

assert len(st) == 63


# ======================================================================
# 3. LEER MATRIZ CON RUTAS DE TEXTOS CSN
# ======================================================================

df = pd.read_csv(
    RUTA_CANDIDATO
)

assert len(df) == 42


# ======================================================================
# 4. PARSER DE HEADER CSN
# ======================================================================

def leer_txt_csn(path):

    headers = []

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        for linea in f:

            if linea.startswith("#"):

                headers.append(
                    linea.strip()
                )

            else:

                break


    x = np.loadtxt(
        path,
        comments="#"
    )

    x = np.asarray(
        x,
        dtype=float
    ).reshape(-1)

    assert np.isfinite(x).all()


    texto = "\n".join(
        headers
    )


    fs = np.nan

    patrones_fs = [
        r"Frecuencia\s+de\s+muestreo\s*:\s*([0-9.]+)",
        r"Sampling\s+rate\s*:\s*([0-9.]+)"
    ]


    for p in patrones_fs:

        m = re.search(
            p,
            texto,
            flags=re.I
        )

        if m:

            fs = float(
                m.group(1)
            )

            break


    return {
        "data": x,
        "headers": headers,
        "fs": fs
    }


# ======================================================================
# 5. MEJOR ALINEACIÓN ENTERA RAW ↔ TEXTO
# ======================================================================

def alinear_lag(
    raw,
    txt,
    max_lag=20
):

    raw = np.asarray(
        raw,
        dtype=float
    )

    txt = np.asarray(
        txt,
        dtype=float
    )


    resultados = []


    for lag in range(
        -max_lag,
        max_lag + 1
    ):

        if lag >= 0:

            n = min(
                len(txt),
                len(raw) - lag
            )

            if n <= 100:
                continue

            xr = raw[
                lag:
                lag + n
            ]

            yt = txt[
                :n
            ]


        else:

            inicio_txt = -lag

            n = min(
                len(txt) - inicio_txt,
                len(raw)
            )

            if n <= 100:
                continue

            xr = raw[
                :n
            ]

            yt = txt[
                inicio_txt:
                inicio_txt + n
            ]


        xc = xr - np.mean(xr)
        yc = yt - np.mean(yt)


        den = (
            np.linalg.norm(xc)
            *
            np.linalg.norm(yc)
        )


        if den == 0:

            corr = np.nan

        else:

            corr = float(
                np.dot(
                    xc,
                    yc
                )
                /
                den
            )


        resultados.append(
            (
                lag,
                corr,
                xr,
                yt
            )
        )


    resultados = [
        r
        for r in resultados
        if np.isfinite(
            r[1]
        )
    ]


    assert resultados


    mejor = max(
        resultados,
        key=lambda z: abs(
            z[1]
        )
    )


    return mejor


# ======================================================================
# 6. REGRESIÓN TEXTO = a*RAW + b
# ======================================================================

def regresion_lineal(
    x,
    y
):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )


    A = np.column_stack(
        [
            x,
            np.ones(
                len(x)
            )
        ]
    )


    coef, _, _, _ = np.linalg.lstsq(
        A,
        y,
        rcond=None
    )


    a = float(
        coef[0]
    )

    b = float(
        coef[1]
    )


    yhat = (
        a * x
        +
        b
    )


    resid = (
        y
        -
        yhat
    )


    sse = np.sum(
        resid**2
    )

    sst = np.sum(
        (
            y
            -
            np.mean(y)
        )**2
    )


    r2 = (
        1.0
        -
        sse / sst
        if sst > 0
        else np.nan
    )


    rmse = float(
        np.sqrt(
            np.mean(
                resid**2
            )
        )
    )


    std_y = float(
        np.std(
            y
        )
    )


    nrmse = (
        rmse / std_y
        if std_y > 0
        else np.nan
    )


    return {
        "slope": a,
        "intercept": b,
        "r2": r2,
        "rmse": rmse,
        "nrmse": nrmse,
        "yhat": yhat
    }


# ======================================================================
# 7. DESCARGAR STATIONXML HISTÓRICO
# ======================================================================

print("\n" + "=" * 80)
print("DESCARGA STATIONXML — ÉPOCA 2017")
print("=" * 80)


client = Client(
    "EARTHSCOPE"
)


inventarios = {}


for estacion in ESTACIONES:

    print(
        "\nSolicitando:",
        estacion
    )


    try:

        inv = client.get_stations(
            network="C1",
            station=estacion,
            location="*",
            channel="HN?",
            starttime=T_EVENTO - 60,
            endtime=T_EVENTO + 60,
            level="response"
        )


        inventarios[
            estacion
        ] = inv


        ruta_xml = os.path.join(
            META_DIR,
            f"C1_{estacion}_20170424_response.xml"
        )


        inv.write(
            ruta_xml,
            format="STATIONXML"
        )


        canales = []


        for net in inv:

            for sta in net:

                for cha in sta:

                    canales.append(
                        (
                            net.code,
                            sta.code,
                            cha.location_code,
                            cha.code,
                            str(
                                cha.start_date
                            ),
                            str(
                                cha.end_date
                            )
                        )
                    )


        print(
            "  Canales:",
            canales
        )


    except Exception as e:

        print(
            "  ERROR:",
            repr(e)
        )


print(
    "\nStationXML recuperados:",
    len(inventarios),
    "/",
    len(ESTACIONES)
)


# ======================================================================
# 8. OBTENER SENSIBILIDAD HISTÓRICA
# ======================================================================

def sensibilidad_historica(
    estacion,
    canal
):

    assert estacion in inventarios, (
        f"Sin StationXML para {estacion}"
    )


    inv = inventarios[
        estacion
    ]


    sel = inv.select(
        network="C1",
        station=estacion,
        channel=canal,
        time=T_EVENTO
    )


    candidatos = []


    for net in sel:

        for sta in net:

            for cha in sta:

                candidatos.append(
                    cha
                )


    assert len(candidatos) >= 1, (
        f"No hay respuesta histórica para "
        f"{estacion} {canal}"
    )


    candidatos.sort(
        key=lambda c:
            (
                c.location_code != "",
                c.location_code
            )
    )


    cha = candidatos[0]


    resp = cha.response


    assert (
        resp.instrument_sensitivity
        is not None
    )


    sens = resp.instrument_sensitivity


    return {
        "location":
            cha.location_code,

        "sensitivity":
            float(
                sens.value
            ),

        "frequency":
            float(
                sens.frequency
            ),

        "input_units":
            str(
                sens.input_units
            ),

        "output_units":
            str(
                sens.output_units
            ),

        "sensor_description":
            (
                cha.sensor.description
                if cha.sensor
                else None
            ),

        "sample_rate":
            float(
                cha.sample_rate
            )
            if cha.sample_rate
            else np.nan
    }


# ======================================================================
# 9. COMPARAR RAW ↔ TEXTO ↔ STATIONXML
# ======================================================================

filas = []


for estacion, comp in COMPARABLES:

    canal = (
        "HNE"
        if comp == "E"
        else "HNN"
    )


    print(
        "\nProcesando:",
        estacion,
        canal
    )


    # RAW
    tr_sel = st.select(
        network="C1",
        station=estacion,
        channel=canal
    )


    assert len(tr_sel) == 1


    tr = tr_sel[0]

    raw = np.asarray(
        tr.data,
        dtype=float
    )


    # TXT CSN
    row = df[
        df[
            "Estacion"
        ].str.upper()
        ==
        estacion.upper()
    ]


    assert len(row) == 1


    row = row.iloc[0]


    path_txt = (
        row[
            "Archivo_E"
        ]
        if comp == "E"
        else
        row[
            "Archivo_N"
        ]
    )


    assert os.path.exists(
        path_txt
    )


    txt_info = leer_txt_csn(
        path_txt
    )


    txt = txt_info[
        "data"
    ]


    print(
        "  RAW N:",
        len(raw),
        "| TXT N:",
        len(txt)
    )

    print(
        "  RAW Fs:",
        tr.stats.sampling_rate,
        "| TXT Fs:",
        txt_info[
            "fs"
        ]
    )


    # Alineación
    lag, corr, xr, yt = alinear_lag(
        raw,
        txt,
        max_lag=20
    )


    fit = regresion_lineal(
        xr,
        yt
    )


    slope = fit[
        "slope"
    ]


    # StationXML
    try:

        meta = sensibilidad_historica(
            estacion,
            canal
        )


        sens = meta[
            "sensitivity"
        ]


        slope_teorico_abs = (
            1.0
            /
            abs(
                sens
            )
        )


        ratio_csn_vs_metadata = (
            abs(
                slope
            )
            /
            slope_teorico_abs
        )


        ratio_2csn_vs_metadata = (
            2.0
            *
            abs(
                slope
            )
            /
            slope_teorico_abs
        )


    except Exception as e:

        meta = {
            "location":
                None,

            "sensitivity":
                np.nan,

            "frequency":
                np.nan,

            "input_units":
                None,

            "output_units":
                None,

            "sensor_description":
                None,

            "sample_rate":
                np.nan
        }


        sens = np.nan
        slope_teorico_abs = np.nan
        ratio_csn_vs_metadata = np.nan
        ratio_2csn_vs_metadata = np.nan


        print(
            "  ERROR metadata:",
            repr(e)
        )


    filas.append(
        {
            "Estacion":
                estacion,

            "Canal":
                canal,

            "N_RAW":
                len(raw),

            "N_TXT":
                len(txt),

            "Fs_RAW_Hz":
                float(
                    tr.stats.sampling_rate
                ),

            "Fs_TXT_Hz":
                txt_info[
                    "fs"
                ],

            "Lag_muestras":
                lag,

            "Correlacion_RAW_TXT":
                corr,

            "Pendiente_TXT_m_s2_por_count":
                slope,

            "Intercepto_TXT_m_s2":
                fit[
                    "intercept"
                ],

            "R2_RAW_TXT":
                fit[
                    "r2"
                ],

            "NRMSE_RAW_TXT":
                fit[
                    "nrmse"
                ],

            "StationXML_sensor":
                meta[
                    "sensor_description"
                ],

            "StationXML_sensitivity":
                sens,

            "StationXML_input_units":
                meta[
                    "input_units"
                ],

            "StationXML_output_units":
                meta[
                    "output_units"
                ],

            "StationXML_sample_rate":
                meta[
                    "sample_rate"
                ],

            "Slope_teorico_abs_1_over_sens":
                slope_teorico_abs,

            "Ratio_CSN_vs_metadata":
                ratio_csn_vs_metadata,

            "Ratio_2xCSN_vs_metadata":
                ratio_2csn_vs_metadata
        }
    )


audit = pd.DataFrame(
    filas
)


# ======================================================================
# 10. RESULTADOS RAW ↔ TEXTO
# ======================================================================

print("\n" + "=" * 80)
print("RAW COUNTS ↔ TEXTO CSN")
print("=" * 80)


display(
    audit[
        [
            "Estacion",
            "Canal",
            "N_RAW",
            "N_TXT",
            "Fs_RAW_Hz",
            "Fs_TXT_Hz",
            "Lag_muestras",
            "Correlacion_RAW_TXT",
            "R2_RAW_TXT",
            "NRMSE_RAW_TXT",
            "Pendiente_TXT_m_s2_por_count"
        ]
    ]
)


# ======================================================================
# 11. RESPUESTA INSTRUMENTAL
# ======================================================================

print("\n" + "=" * 80)
print("STATIONXML HISTÓRICO — RESPUESTA INSTRUMENTAL")
print("=" * 80)


display(
    audit[
        [
            "Estacion",
            "Canal",
            "StationXML_sensor",
            "StationXML_sensitivity",
            "StationXML_input_units",
            "StationXML_output_units",
            "StationXML_sample_rate"
        ]
    ]
)


# ======================================================================
# 12. CLASIFICAR ESCALA
# ======================================================================

def clasificar_metadata(
    row
):

    r1 = row[
        "Ratio_CSN_vs_metadata"
    ]

    r2 = row[
        "Ratio_2xCSN_vs_metadata"
    ]


    if not (
        np.isfinite(r1)
        and
        np.isfinite(r2)
    ):

        return "SIN_METADATA"


    err1 = abs(
        r1 - 1.0
    )

    err2 = abs(
        r2 - 1.0
    )


    if (
        err1 <= 0.05
        and
        err1 < err2
    ):

        return "CSN_COHERENTE"


    elif (
        err2 <= 0.05
        and
        err2 < err1
    ):

        return "ESCALA_2X_COHERENTE"


    else:

        return "NO_CONCLUYENTE"


audit[
    "Diagnostico_metadata"
] = audit.apply(
    clasificar_metadata,
    axis=1
)


print("\n" + "=" * 80)
print("ESCALA OBSERVADA VS RESPUESTA HISTÓRICA")
print("=" * 80)


display(
    audit[
        [
            "Estacion",
            "Canal",
            "R2_RAW_TXT",
            "Ratio_CSN_vs_metadata",
            "Ratio_2xCSN_vs_metadata",
            "Diagnostico_metadata"
        ]
    ].round(
        6
    )
)


print(
    "\nConteo:"
)

print(
    audit[
        "Diagnostico_metadata"
    ].value_counts(
        dropna=False
    )
)


# ======================================================================
# 13. GATE
# ======================================================================

print("\n" + "=" * 80)
print("GATE RAW → CSN → METADATA")
print("=" * 80)


n_meta = (
    audit[
        "Diagnostico_metadata"
    ]
    !=
    "SIN_METADATA"
).sum()


n_csn = (
    audit[
        "Diagnostico_metadata"
    ]
    ==
    "CSN_COHERENTE"
).sum()


n_2x = (
    audit[
        "Diagnostico_metadata"
    ]
    ==
    "ESCALA_2X_COHERENTE"
).sum()


n_nc = (
    audit[
        "Diagnostico_metadata"
    ]
    ==
    "NO_CONCLUYENTE"
).sum()


print(
    "Componentes con metadata:",
    n_meta,
    "/ 9"
)

print(
    "CSN coherente:",
    n_csn
)

print(
    "Escala 2x coherente:",
    n_2x
)

print(
    "No concluyente:",
    n_nc
)


# ======================================================================
# 14. GUARDAR
# ======================================================================

RUTA_OUT = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_RAW_CSN_StationXML.csv"
)


audit.to_csv(
    RUTA_OUT,
    index=False
)


print("\nArchivo:")
print(RUTA_OUT)

print(
    "\nV5.1 NO FUE EJECUTADO."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — AUDITORÍA DE RESPONSE STAGES
# MT01 / MT02
#
# Objetivo:
# verificar la cadena completa de respuesta instrumental antes de
# interpretar la sensibilidad global de MT01 y MT02.
#
# NO modifica señales.
# NO calcula Sa.
# NO ejecuta V5.1.
# ======================================================================

from obspy import read_inventory
import os
import pandas as pd


BASE = "/content/drive/MyDrive/Tesis/Datos"

META_DIR = os.path.join(
    BASE,
    "Valparaiso2017",
    "StationXML_2017"
)


ESTACIONES = [
    "MT01",
    "MT02"
]


CANALES = [
    "HNE",
    "HNN"
]


filas = []


for estacion in ESTACIONES:

    ruta = os.path.join(
        META_DIR,
        f"C1_{estacion}_20170424_response.xml"
    )

    assert os.path.exists(ruta)

    inv = read_inventory(ruta)


    for canal in CANALES:

        sel = inv.select(
            network="C1",
            station=estacion,
            channel=canal
        )


        cha = None

        for net in sel:

            for sta in net:

                for c in sta:

                    cha = c


        assert cha is not None


        resp = cha.response


        print("\n" + "=" * 80)
        print(estacion, canal)
        print("=" * 80)


        sens = resp.instrument_sensitivity


        print(
            "Instrument sensitivity:",
            sens.value
        )

        print(
            "Input units:",
            sens.input_units
        )

        print(
            "Output units:",
            sens.output_units
        )


        for i, stage in enumerate(
            resp.response_stages,
            start=1
        ):

            print(
                f"\nStage {i}"
            )

            print(
                "  tipo:",
                type(stage).__name__
            )

            print(
                "  input_units:",
                getattr(
                    stage,
                    "input_units",
                    None
                )
            )

            print(
                "  output_units:",
                getattr(
                    stage,
                    "output_units",
                    None
                )
            )

            print(
                "  stage_gain:",
                getattr(
                    stage,
                    "stage_gain",
                    None
                )
            )

            print(
                "  stage_gain_frequency:",
                getattr(
                    stage,
                    "stage_gain_frequency",
                    None
                )
            )


            filas.append(
                {
                    "Estacion":
                        estacion,

                    "Canal":
                        canal,

                    "Stage":
                        i,

                    "Tipo":
                        type(stage).__name__,

                    "Input_units":
                        getattr(
                            stage,
                            "input_units",
                            None
                        ),

                    "Output_units":
                        getattr(
                            stage,
                            "output_units",
                            None
                        ),

                    "Stage_gain":
                        getattr(
                            stage,
                            "stage_gain",
                            None
                        ),

                    "Stage_gain_frequency":
                        getattr(
                            stage,
                            "stage_gain_frequency",
                            None
                        )
                }
            )


etapas = pd.DataFrame(
    filas
)


print("\n" + "=" * 80)
print("RESUMEN DE ETAPAS")
print("=" * 80)

display(etapas)


ruta_out = os.path.join(
    BASE,
    "Valparaiso2017",
    "Valparaiso2017_MT01_MT02_response_stages.csv"
)


etapas.to_csv(
    ruta_out,
    index=False
)


print("\nArchivo:")
print(ruta_out)

print("\nV5.1 NO FUE EJECUTADO.")

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — CIERRE DE AUDITORÍA DE ESCALA + FREEZE FORMAL
#
# Objetivos:
# 1. documentar el defecto de unidades StationXML de MT01/MT02;
# 2. verificar que el producto de stages reproduce instrument_sensitivity;
# 3. cerrar las 9 componentes de control como compatibles con escala CSN;
# 4. congelar EXACTAMENTE el CSV candidato de 42 estaciones;
# 5. generar SHA256 y manifiesto de auditoría.
#
# NO modifica valores Sa.
# NO recalcula RotD50.
# NO ejecuta V5.1.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import json
import shutil
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# ======================================================================
# 1. RUTAS
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Valparaiso2017"
)

RUTA_CANDIDATO = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_CANDIDATO.csv"
)

RUTA_RAW_META = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_RAW_CSN_StationXML.csv"
)

RUTA_STAGES = os.path.join(
    CARPETA,
    "Valparaiso2017_MT01_MT02_response_stages.csv"
)

RUTA_ESCALA = os.path.join(
    CARPETA,
    "Valparaiso2017_auditoria_escala_CSN_vs_CESMD.csv"
)

RUTA_FINAL = os.path.join(
    CARPETA,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_FINAL_FROZEN.csv"
)

RUTA_RESOLUCION = os.path.join(
    CARPETA,
    "Valparaiso2017_resolucion_escala_CSN_CESMD_FINAL.csv"
)

RUTA_MANIFIESTO = os.path.join(
    CARPETA,
    "Valparaiso2017_FREEZE_MANIFEST.json"
)


for ruta in [
    RUTA_CANDIDATO,
    RUTA_RAW_META,
    RUTA_STAGES,
    RUTA_ESCALA
]:
    assert os.path.exists(ruta), ruta


# ======================================================================
# 2. UTILIDAD SHA256
# ======================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for bloque in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(bloque)

    return h.hexdigest()


# ======================================================================
# 3. CARGAR AUDITORÍAS
# ======================================================================

candidate = pd.read_csv(
    RUTA_CANDIDATO
)

rawmeta = pd.read_csv(
    RUTA_RAW_META
)

stages = pd.read_csv(
    RUTA_STAGES
)

escala = pd.read_csv(
    RUTA_ESCALA
)


assert len(candidate) == 42
assert len(rawmeta) == 9


print("=" * 80)
print("VALPARAÍSO 2017 — CIERRE DE AUDITORÍA")
print("=" * 80)

print(
    "Estaciones candidato:",
    len(candidate)
)

print(
    "Componentes de control:",
    len(rawmeta)
)


# ======================================================================
# 4. PRODUCTO DE STAGES MT01 / MT02
# ======================================================================

productos = (
    stages
    .groupby(
        ["Estacion", "Canal"],
        as_index=False
    )
    .agg(
        Producto_stage_gain=(
            "Stage_gain",
            "prod"
        )
    )
)


mt = rawmeta[
    rawmeta[
        "Estacion"
    ].isin(
        ["MT01", "MT02"]
    )
].copy()


mt = mt.merge(
    productos,
    on=[
        "Estacion",
        "Canal"
    ],
    how="left"
)


mt[
    "Delta_producto_vs_global_pct"
] = (
    100.0
    *
    (
        mt[
            "Producto_stage_gain"
        ]
        /
        mt[
            "StationXML_sensitivity"
        ]
        -
        1.0
    )
)


print("\n" + "=" * 80)
print("MT01 / MT02 — PRODUCTO DE RESPONSE STAGES")
print("=" * 80)


display(
    mt[
        [
            "Estacion",
            "Canal",
            "Producto_stage_gain",
            "StationXML_sensitivity",
            "Delta_producto_vs_global_pct",
            "Ratio_CSN_vs_metadata",
            "Ratio_2xCSN_vs_metadata"
        ]
    ].round(8)
)


max_delta_stage = (
    mt[
        "Delta_producto_vs_global_pct"
    ]
    .abs()
    .max()
)


assert max_delta_stage < 0.01, (
    "STOP: producto de stages no reproduce "
    "instrument_sensitivity."
)


print(
    "✅ Producto de stages reproduce "
    "instrument_sensitivity."
)


# ======================================================================
# 5. CREAR TABLA DE RESOLUCIÓN FINAL
# ======================================================================

res = rawmeta.copy()


# Incorporar ratios CESMD
escala2 = escala.copy()

escala2[
    "Canal"
] = escala2[
    "Componente"
].map(
    {
        "E": "HNE",
        "N": "HNN"
    }
)


res = res.merge(
    escala2[
        [
            "Estacion",
            "Canal",
            "Ratio_PGA_CESMD_CSN",
            "Ratio_PSA_CESMD_CSN"
        ]
    ],
    on=[
        "Estacion",
        "Canal"
    ],
    how="left"
)


# Tipo de evidencia
def tipo_evidencia(row):

    if row["Estacion"] in [
        "MT01",
        "MT02"
    ]:

        return (
            "GAIN_CHAIN_OK_UNIT_LABEL_DEFECT"
        )

    else:

        return (
            "STATIONXML_UNITS_DIRECT"
        )


res[
    "Tipo_evidencia"
] = res.apply(
    tipo_evidencia,
    axis=1
)


# Criterio:
# escala CSN compatible si difiere <=5% de sensibilidad histórica.
res[
    "Error_CSN_metadata_pct"
] = (
    100.0
    *
    (
        res[
            "Ratio_CSN_vs_metadata"
        ]
        -
        1.0
    )
)


res[
    "Error_2x_metadata_pct"
] = (
    100.0
    *
    (
        res[
            "Ratio_2xCSN_vs_metadata"
        ]
        -
        1.0
    )
)


res[
    "Escala_CSN_PASS"
] = (
    res[
        "Error_CSN_metadata_pct"
    ].abs()
    <=
    5.0
)


res[
    "Escala_2x_PASS"
] = (
    res[
        "Error_2x_metadata_pct"
    ].abs()
    <=
    5.0
)


res[
    "Decision_final"
] = np.where(
    (
        res[
            "Escala_CSN_PASS"
        ]
        &
        ~res[
            "Escala_2x_PASS"
        ]
    ),
    "CSN_SCALE_SUPPORTED",
    "REVIEW"
)


print("\n" + "=" * 80)
print("RESOLUCIÓN FINAL DE ESCALA")
print("=" * 80)


display(
    res[
        [
            "Estacion",
            "Canal",
            "R2_RAW_TXT",
            "Ratio_CSN_vs_metadata",
            "Ratio_2xCSN_vs_metadata",
            "Ratio_PGA_CESMD_CSN",
            "Ratio_PSA_CESMD_CSN",
            "Tipo_evidencia",
            "Decision_final"
        ]
    ].round(6)
)


# ======================================================================
# 6. GATE FINAL
# ======================================================================

n_pass = (
    res[
        "Decision_final"
    ]
    ==
    "CSN_SCALE_SUPPORTED"
).sum()


print("\n" + "=" * 80)
print("GATE FINAL DE ESCALA")
print("=" * 80)

print(
    "CSN respaldado:",
    n_pass,
    "/ 9"
)


assert n_pass == 9, (
    "STOP: no todas las componentes "
    "respaldan la escala CSN."
)


print(
    "✅ 9/9 componentes respaldan "
    "la escala CSN."
)

print(
    "✅ No existe justificación para "
    "multiplicar los registros CSN por 2."
)


# ======================================================================
# 7. GUARDAR RESOLUCIÓN
# ======================================================================

res.to_csv(
    RUTA_RESOLUCION,
    index=False
)


# ======================================================================
# 8. FREEZE DEL CANDIDATO
# ======================================================================

sha_candidate = sha256_file(
    RUTA_CANDIDATO
)


# Copia byte-a-byte.
shutil.copy2(
    RUTA_CANDIDATO,
    RUTA_FINAL
)


sha_final = sha256_file(
    RUTA_FINAL
)


assert sha_candidate == sha_final, (
    "STOP: el archivo congelado no es "
    "idéntico al candidato."
)


final_df = pd.read_csv(
    RUTA_FINAL
)


assert len(final_df) == 42


# Verificación numérica completa
pd.testing.assert_frame_equal(
    candidate,
    final_df,
    check_exact=True
)


print("\n" + "=" * 80)
print("FREEZE FORMAL Sa")
print("=" * 80)

print(
    "Filas:",
    len(final_df)
)

print(
    "SHA256 candidato:",
    sha_candidate
)

print(
    "SHA256 frozen   :",
    sha_final
)

print(
    "Identidad byte-a-byte:",
    sha_candidate == sha_final
)

print(
    "Identidad dataframe:",
    True
)


# ======================================================================
# 9. MANIFIESTO
# ======================================================================

manifest = {
    "evento":
        "Valparaiso 2017",

    "usgs_event_id":
        "us10008kce",

    "csn_event_id":
        "6c5752b76db0f46280949a798662a224",

    "n_estaciones":
        42,

    "target":
        "RotD50 PSA T=1.0 s, damping=5%",

    "fuente_movimiento_primaria":
        "CSN text strong-motion records",

    "decision_escala":
        "CSN_SCALE_SUPPORTED",

    "componentes_control":
        9,

    "componentes_CSN_supported":
        int(n_pass),

    "nota_MT01_MT02":
        (
            "StationXML unit labels are internally inconsistent; "
            "however response-stage gain product reproduces global "
            "instrument sensitivity within <0.01%, and observed "
            "RAW-to-CSN scale agrees with that gain magnitude."
        ),

    "candidate_file":
        os.path.basename(
            RUTA_CANDIDATO
        ),

    "candidate_sha256":
        sha_candidate,

    "frozen_file":
        os.path.basename(
            RUTA_FINAL
        ),

    "frozen_sha256":
        sha_final,

    "resolution_file":
        os.path.basename(
            RUTA_RESOLUCION
        ),

    "resolution_sha256":
        sha256_file(
            RUTA_RESOLUCION
        ),

    "raw_metadata_audit_sha256":
        sha256_file(
            RUTA_RAW_META
        ),

    "response_stages_audit_sha256":
        sha256_file(
            RUTA_STAGES
        ),

    "cesmd_scale_audit_sha256":
        sha256_file(
            RUTA_ESCALA
        ),

    "freeze_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "v5_1_executed":
        False
}


with open(
    RUTA_MANIFIESTO,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\nManifiesto:")
print(
    RUTA_MANIFIESTO
)


print("\nArchivo frozen:")
print(
    RUTA_FINAL
)


print("\nResolución:")
print(
    RUTA_RESOLUCION
)


# ======================================================================
# 10. DECISIÓN
# ======================================================================

print("\n" + "=" * 80)
print("DECISIÓN FINAL PRE-MODELO")
print("=" * 80)

print(
    "✅ ESCALA CSN RESPALDADA."
)

print(
    "✅ 42 valores Sa_RotD50 quedan "
    "formalmente CONGELADOS."
)

print(
    "✅ El archivo frozen es idéntico "
    "byte-a-byte al candidato."
)

print(
    "✅ Ninguna estación fue excluida "
    "por desempeño del modelo."
)

print(
    "\nV5.1 TODAVÍA NO FUE EJECUTADO."
)

print(
    "Siguiente paso permitido:"
)

print(
    "ejecutar V5.1 UNA SOLA VEZ "
    "sobre este archivo frozen."
)

In [ ]:
# ======================================================================
# VALPARAÍSO 2017 — EJECUCIÓN EXTERNA FINAL V5.1
#
# MODELO TOTALMENTE CONGELADO:
#   Gradient Boosting V5.1
#   Predictores: Mw, Rrup, Vs30
#
# PROCEDIMIENTO:
# 1. Verificar archivo frozen de Valparaíso y su SHA256.
# 2. Reconstruir EXACTAMENTE el V5.1 histórico.
# 3. Reproducir las métricas oficiales del test histórico.
# 4. SOLO SI PASA: predecir Valparaíso 2017 una sola vez.
# 5. Guardar predicciones, métricas y manifiesto.
#
# Residuo:
#   ln(Sa_pred) - ln(Sa_obs)
#   positivo = sobrepredicción
#   negativo = subpredicción
#
# NO reoptimiza.
# NO calibra.
# NO cambia predictores.
# NO cambia hiperparámetros.
# NO elimina estaciones.
# ======================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import json
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


# ======================================================================
# 0. CONFIGURACIÓN
# ======================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA_VALPO = os.path.join(
    BASE,
    "Valparaiso2017"
)

RUTA_FROZEN = os.path.join(
    CARPETA_VALPO,
    "Valparaiso2017_42_estaciones_Rrup_Vs30_RotD50_FINAL_FROZEN.csv"
)

RUTA_FREEZE_MANIFEST = os.path.join(
    CARPETA_VALPO,
    "Valparaiso2017_FREEZE_MANIFEST.json"
)


# Archivos que se crearán SOLO después de ejecutar Valparaíso
RUTA_PRED = os.path.join(
    CARPETA_VALPO,
    "Valparaiso2017_V5_1_predicciones_FINAL.csv"
)

RUTA_METRICAS = os.path.join(
    CARPETA_VALPO,
    "Valparaiso2017_V5_1_metricas_FINAL.csv"
)

RUTA_RUN_MANIFEST = os.path.join(
    CARPETA_VALPO,
    "Valparaiso2017_V5_1_RUN_MANIFEST.json"
)


SHA_FROZEN_ESPERADO = (
    "936283884656a9821f36de5141139f5ae38705bc272210658bf1a6f3a6b97afe"
)


# ======================================================================
# 1. PROTECCIÓN CONTRA SEGUNDA EJECUCIÓN
# ======================================================================

existentes = [
    p
    for p in [
        RUTA_PRED,
        RUTA_METRICAS,
        RUTA_RUN_MANIFEST
    ]
    if os.path.exists(p)
]


assert len(existentes) == 0, (
    "STOP: ya existe una ejecución V5.1 de Valparaíso.\n"
    "No se volverá a ejecutar automáticamente.\n"
    +
    "\n".join(existentes)
)


# ======================================================================
# 2. SHA256
# ======================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for bloque in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(bloque)

    return h.hexdigest()


# ======================================================================
# 3. VERIFICAR FREEZE
# ======================================================================

assert os.path.exists(RUTA_FROZEN), RUTA_FROZEN
assert os.path.exists(RUTA_FREEZE_MANIFEST), RUTA_FREEZE_MANIFEST


sha_frozen = sha256_file(
    RUTA_FROZEN
)


print("=" * 80)
print("VALPARAÍSO 2017 — EJECUCIÓN FINAL V5.1")
print("=" * 80)

print("\nSHA256 frozen:")
print(sha_frozen)


assert sha_frozen == SHA_FROZEN_ESPERADO, (
    "STOP: el SHA256 del archivo frozen cambió."
)


with open(
    RUTA_FREEZE_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    freeze_manifest = json.load(f)


assert freeze_manifest[
    "frozen_sha256"
] == SHA_FROZEN_ESPERADO


assert freeze_manifest[
    "v5_1_executed"
] is False


print("✅ Archivo frozen verificado.")
print("✅ Manifiesto PRE-modelo verificado.")


# ======================================================================
# 4. LOCALIZAR DATASET OFICIAL V5.1
# ======================================================================

NOMBRE_DATASET = (
    "01_dataset_analitico_v5_1987.csv"
)


def localizar_archivo_oficial(nombre):

    candidatos_directos = [
        os.path.join(
            BASE,
            "Resultados_V5",
            nombre
        ),
        os.path.join(
            "/content/drive/MyDrive/Tesis",
            "Resultados_V5",
            nombre
        )
    ]


    directos = [
        p
        for p in candidatos_directos
        if os.path.exists(p)
    ]


    if len(directos) == 1:

        return directos[0]


    # Búsqueda restringida a la carpeta Tesis
    raiz_busqueda = (
        "/content/drive/MyDrive/Tesis"
    )

    encontrados = []


    for raiz, dirs, files in os.walk(
        raiz_busqueda
    ):

        if nombre in files:

            encontrados.append(
                os.path.join(
                    raiz,
                    nombre
                )
            )


    # Preferir inequívocamente Resultados_V5
    preferidos = [
        p
        for p in encontrados
        if "Resultados_V5" in p
    ]


    if len(preferidos) == 1:

        return preferidos[0]


    if len(encontrados) == 1:

        return encontrados[0]


    raise RuntimeError(
        "No se pudo identificar de forma inequívoca "
        f"el archivo oficial {nombre}.\n"
        f"Encontrados: {encontrados}"
    )


RUTA_DATASET = localizar_archivo_oficial(
    NOMBRE_DATASET
)


print("\nDataset oficial V5.1:")
print(RUTA_DATASET)

print(
    "SHA256 dataset:",
    sha256_file(
        RUTA_DATASET
    )
)


# ======================================================================
# 5. CARGAR DATASET HISTÓRICO OFICIAL
# ======================================================================

hist = pd.read_csv(
    RUTA_DATASET
)


COLUMNAS_REQUERIDAS = [
    "Earthquake_Name",
    "NGAsubEQID",
    "Earthquake_Magnitude",
    "ClstD_km",
    "Vs30_Selected_for_Analysis_m_s",
    "T1pt000S"
]


faltantes = [
    c
    for c in COLUMNAS_REQUERIDAS
    if c not in hist.columns
]


assert not faltantes, (
    f"Faltan columnas oficiales: {faltantes}"
)


assert len(hist) == 1987, (
    f"Dataset inesperado: {len(hist)} registros."
)


assert hist[
    "NGAsubEQID"
].nunique() == 108


print("\nDataset histórico:")
print("Registros:", len(hist))
print(
    "Eventos:",
    hist["NGAsubEQID"].nunique()
)


# ======================================================================
# 6. RECONSTRUIR PARTICIONES OFICIALES
# ======================================================================

mask_pisco = (
    hist[
        "Earthquake_Name"
    ]
    .astype(str)
    .str.contains(
        "Pisco",
        case=False,
        na=False
    )
)


pisco = hist[
    mask_pisco
].copy()


pool = hist[
    ~mask_pisco
].copy()


assert len(pisco) == 22
assert pisco["NGAsubEQID"].nunique() == 1


gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)


idx_dev, idx_test = next(
    gss.split(
        pool,
        groups=pool[
            "NGAsubEQID"
        ]
    )
)


dev = pool.iloc[
    idx_dev
].copy()


test = pool.iloc[
    idx_test
].copy()


assert len(dev) == 1583
assert len(test) == 382

assert (
    dev[
        "NGAsubEQID"
    ].nunique()
    ==
    85
)

assert (
    test[
        "NGAsubEQID"
    ].nunique()
    ==
    22
)


overlap = (
    set(
        dev[
            "NGAsubEQID"
        ].unique()
    )
    &
    set(
        test[
            "NGAsubEQID"
        ].unique()
    )
)


assert len(overlap) == 0


print("\n" + "=" * 80)
print("PARTICIONES OFICIALES RECONSTRUIDAS")
print("=" * 80)

print(
    "Development:",
    len(dev),
    "registros /",
    dev["NGAsubEQID"].nunique(),
    "eventos"
)

print(
    "Test:",
    len(test),
    "registros /",
    test["NGAsubEQID"].nunique(),
    "eventos"
)

print(
    "Pisco reservado:",
    len(pisco),
    "registros /",
    pisco["NGAsubEQID"].nunique(),
    "evento"
)

print(
    "Solapamiento eventos Dev-Test:",
    len(overlap)
)


# ======================================================================
# 7. FEATURES Y TARGET OFICIALES
# ======================================================================

FEATURES = [
    "Earthquake_Magnitude",
    "ClstD_km",
    "Vs30_Selected_for_Analysis_m_s"
]

TARGET = "T1pt000S"


for conjunto, nombre in [
    (dev, "development"),
    (test, "test")
]:

    assert np.isfinite(
        conjunto[
            FEATURES
        ].to_numpy(
            dtype=float
        )
    ).all(), nombre

    assert (
        conjunto[
            TARGET
        ].to_numpy(
            dtype=float
        )
        >
        0
    ).all(), nombre


X_dev = dev[
    FEATURES
].astype(float)


y_dev = np.log(
    dev[
        TARGET
    ].astype(float)
)


X_test = test[
    FEATURES
].astype(float)


y_test = np.log(
    test[
        TARGET
    ].astype(float)
)


# ======================================================================
# 8. MODELO V5.1 EXACTAMENTE CONGELADO
# ======================================================================

PARAMS_V5_1 = {
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 2,
    "min_samples_split": 15,
    "min_samples_leaf": 2,
    "subsample": 0.7,
    "random_state": 42
}


modelo = GradientBoostingRegressor(
    **PARAMS_V5_1
)


# ÚNICO FIT DEL MODELO EN ESTA CELDA
modelo.fit(
    X_dev,
    y_dev
)


# ======================================================================
# 9. GATE: REPRODUCIR TEST HISTÓRICO OFICIAL
# ======================================================================

pred_test = modelo.predict(
    X_test
)


res_test = (
    pred_test
    -
    y_test.to_numpy()
)


metricas_test = {
    "R2":
        r2_score(
            y_test,
            pred_test
        ),

    "RMSE":
        np.sqrt(
            mean_squared_error(
                y_test,
                pred_test
            )
        ),

    "MAE":
        mean_absolute_error(
            y_test,
            pred_test
        ),

    "BIAS":
        np.mean(
            res_test
        ),

    "SD_RESIDUAL":
        np.std(
            res_test,
            ddof=1
        )
}


OFICIAL = {
    "R2":
        0.8728150622180848,

    "RMSE":
        1.1516018048736865,

    "MAE":
        0.8817139866066795,

    "BIAS":
        -0.07367153944567756,

    "SD_RESIDUAL":
        1.1507500948945866
}


print("\n" + "=" * 80)
print("GATE DE REPRODUCCIÓN V5.1 — TEST HISTÓRICO")
print("=" * 80)


filas_gate = []


for k in OFICIAL:

    calculado = float(
        metricas_test[k]
    )

    esperado = float(
        OFICIAL[k]
    )

    delta = (
        calculado
        -
        esperado
    )


    filas_gate.append(
        {
            "Metrica":
                k,

            "Calculado":
                calculado,

            "Oficial":
                esperado,

            "Diferencia":
                delta
        }
    )


gate_df = pd.DataFrame(
    filas_gate
)


display(
    gate_df
)


TOL = 1e-8


gate_ok = all(
    abs(
        metricas_test[k]
        -
        OFICIAL[k]
    )
    <=
    TOL
    for k in OFICIAL
)


assert gate_ok, (
    "STOP: el modelo reconstruido NO reproduce "
    "las métricas oficiales V5.1. "
    "Valparaíso NO fue predicho."
)


print(
    "✅ V5.1 reproducido correctamente."
)

print(
    "✅ Se autoriza la predicción externa "
    "sobre el archivo frozen."
)


# ======================================================================
# 10. CARGAR VALPARAÍSO FROZEN
# ======================================================================

valpo = pd.read_csv(
    RUTA_FROZEN
)


COLS_VALPO = [
    "Estacion",
    "Mw",
    "Rrup_km",
    "Vs30_m_s",
    "RotD50_T1_g"
]


faltantes_valpo = [
    c
    for c in COLS_VALPO
    if c not in valpo.columns
]


assert not faltantes_valpo, (
    f"Faltan columnas frozen: {faltantes_valpo}\n"
    f"Columnas disponibles:\n{list(valpo.columns)}"
)


assert len(valpo) == 42
assert valpo["Estacion"].nunique() == 42


for c in [
    "Mw",
    "Rrup_km",
    "Vs30_m_s",
    "RotD50_T1_g"
]:

    valpo[c] = pd.to_numeric(
        valpo[c],
        errors="raise"
    )


assert np.isfinite(
    valpo[
        [
            "Mw",
            "Rrup_km",
            "Vs30_m_s",
            "RotD50_T1_g"
        ]
    ].to_numpy()
).all()


assert (
    valpo[
        [
            "Rrup_km",
            "Vs30_m_s",
            "RotD50_T1_g"
        ]
    ].to_numpy()
    >
    0
).all()


assert np.allclose(
    valpo[
        "Mw"
    ].to_numpy(),
    6.9,
    rtol=0,
    atol=1e-12
)


# Control específico de M13L
m13l = valpo[
    valpo[
        "Estacion"
    ]
    ==
    "M13L"
]


assert len(m13l) == 1


assert np.isclose(
    float(
        m13l.iloc[0][
            "Rrup_km"
        ]
    ),
    286.222325,
    atol=1e-5
)


assert np.isclose(
    float(
        m13l.iloc[0][
            "Vs30_m_s"
        ]
    ),
    271.460602,
    atol=1e-5
)


print("\n" + "=" * 80)
print("INPUT EXTERNO CONGELADO")
print("=" * 80)

print("N estaciones:", len(valpo))
print(
    "Mw:",
    valpo["Mw"].min(),
    "a",
    valpo["Mw"].max()
)

print(
    "Rrup:",
    valpo["Rrup_km"].min(),
    "a",
    valpo["Rrup_km"].max(),
    "km"
)

print(
    "Vs30:",
    valpo["Vs30_m_s"].min(),
    "a",
    valpo["Vs30_m_s"].max(),
    "m/s"
)

print(
    "Sa observado:",
    valpo["RotD50_T1_g"].min(),
    "a",
    valpo["RotD50_T1_g"].max(),
    "g"
)


# ======================================================================
# 11. CONSTRUIR INPUT V5.1
# ======================================================================

X_valpo = pd.DataFrame(
    {
        "Earthquake_Magnitude":
            valpo[
                "Mw"
            ].to_numpy(
                dtype=float
            ),

        "ClstD_km":
            valpo[
                "Rrup_km"
            ].to_numpy(
                dtype=float
            ),

        "Vs30_Selected_for_Analysis_m_s":
            valpo[
                "Vs30_m_s"
            ].to_numpy(
                dtype=float
            )
    },
    index=valpo.index
)


y_valpo = np.log(
    valpo[
        "RotD50_T1_g"
    ].to_numpy(
        dtype=float
    )
)


# ======================================================================
# 12. >>> ÚNICA PREDICCIÓN EXTERNA V5.1 <<<
# ======================================================================

print("\n" + "=" * 80)
print("EJECUTANDO V5.1 SOBRE VALPARAÍSO 2017")
print("=" * 80)

print(
    "Esta es la primera ejecución sobre "
    "el input congelado de 42 estaciones."
)


pred_valpo = modelo.predict(
    X_valpo
)


# ======================================================================
# 13. MÉTRICAS EXTERNAS
# ======================================================================

residual = (
    pred_valpo
    -
    y_valpo
)


R2 = r2_score(
    y_valpo,
    pred_valpo
)


RMSE = np.sqrt(
    mean_squared_error(
        y_valpo,
        pred_valpo
    )
)


MAE = mean_absolute_error(
    y_valpo,
    pred_valpo
)


BIAS = np.mean(
    residual
)


ABS_BIAS = abs(
    BIAS
)


SD_RESIDUAL = np.std(
    residual,
    ddof=1
)


EXP_BIAS = np.exp(
    BIAS
)


metricas_valpo = pd.DataFrame(
    [
        {
            "Evento":
                "Valparaiso 2017",

            "USGS_ID":
                "us10008kce",

            "N":
                len(valpo),

            "Mw":
                6.9,

            "R2_lnSa":
                R2,

            "RMSE_lnSa":
                RMSE,

            "MAE_lnSa":
                MAE,

            "BIAS_ln_pred_minus_obs":
                BIAS,

            "ABS_BIAS":
                ABS_BIAS,

            "SD_RESIDUAL":
                SD_RESIDUAL,

            "exp_BIAS":
                EXP_BIAS
        }
    ]
)


# ======================================================================
# 14. TABLA DE PREDICCIONES
# ======================================================================

predicciones = valpo[
    [
        "Estacion",
        "Mw",
        "Rrup_km",
        "Vs30_m_s",
        "RotD50_T1_g"
    ]
].copy()


predicciones[
    "ln_Sa_obs"
] = y_valpo


predicciones[
    "ln_Sa_pred"
] = pred_valpo


predicciones[
    "Sa_pred_g"
] = np.exp(
    pred_valpo
)


predicciones[
    "Residual_ln_pred_minus_obs"
] = residual


predicciones[
    "Ratio_Sa_pred_obs"
] = (
    predicciones[
        "Sa_pred_g"
    ]
    /
    predicciones[
        "RotD50_T1_g"
    ]
)


# ======================================================================
# 15. RESULTADOS
# ======================================================================

print("\n" + "=" * 80)
print("RESULTADO EXTERNO — VALPARAÍSO 2017")
print("=" * 80)


display(
    metricas_valpo.T
)


print(
    "\nConvención de residuo:"
)

print(
    "ln(Sa_pred) - ln(Sa_obs)"
)

print(
    "positivo = sobrepredicción"
)

print(
    "negativo = subpredicción"
)


print("\nPrimeras 10 estaciones:")


display(
    predicciones.head(
        10
    )
)


print("\nResumen residuos:")


print(
    pd.Series(
        residual
    ).describe()
)


print(
    "\nSobrepredicciones:",
    int(
        (
            residual
            >
            0
        ).sum()
    ),
    "/",
    len(residual)
)


print(
    "Subpredicciones:",
    int(
        (
            residual
            <
            0
        ).sum()
    ),
    "/",
    len(residual)
)


# ======================================================================
# 16. GUARDADO ATÓMICO
# ======================================================================

PRED_TMP = (
    RUTA_PRED
    +
    ".tmp"
)

MET_TMP = (
    RUTA_METRICAS
    +
    ".tmp"
)

MAN_TMP = (
    RUTA_RUN_MANIFEST
    +
    ".tmp"
)


predicciones.to_csv(
    PRED_TMP,
    index=False
)


metricas_valpo.to_csv(
    MET_TMP,
    index=False
)


run_manifest = {
    "evento":
        "Valparaiso 2017",

    "usgs_event_id":
        "us10008kce",

    "csn_event_id":
        "6c5752b76db0f46280949a798662a224",

    "modelo":
        "Gradient Boosting V5.1",

    "features":
        FEATURES,

    "target_modelo":
        "ln(T1pt000S)",

    "target_externo":
        "ln(RotD50_T1_g)",

    "residual_definition":
        "ln(Sa_pred)-ln(Sa_obs)",

    "hyperparameters":
        PARAMS_V5_1,

    "random_seed":
        42,

    "historical_dataset":
        os.path.basename(
            RUTA_DATASET
        ),

    "historical_dataset_sha256":
        sha256_file(
            RUTA_DATASET
        ),

    "historical_test_reproduced":
        True,

    "historical_test_metrics":
        {
            k:
                float(
                    metricas_test[k]
                )
            for k in metricas_test
        },

    "frozen_input":
        os.path.basename(
            RUTA_FROZEN
        ),

    "frozen_input_sha256":
        sha_frozen,

    "n_external":
        42,

    "external_metrics":
        {
            "R2_lnSa":
                float(R2),

            "RMSE_lnSa":
                float(RMSE),

            "MAE_lnSa":
                float(MAE),

            "BIAS_ln_pred_minus_obs":
                float(BIAS),

            "ABS_BIAS":
                float(ABS_BIAS),

            "SD_RESIDUAL":
                float(SD_RESIDUAL),

            "exp_BIAS":
                float(EXP_BIAS)
        },

    "sklearn_version":
        sklearn.__version__,

    "run_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "selection_after_results":
        False,

    "records_removed_after_prediction":
        0
}


with open(
    MAN_TMP,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        run_manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# Mover a nombres definitivos solo al final
os.replace(
    PRED_TMP,
    RUTA_PRED
)

os.replace(
    MET_TMP,
    RUTA_METRICAS
)

os.replace(
    MAN_TMP,
    RUTA_RUN_MANIFEST
)


# ======================================================================
# 17. HASHES DE SALIDA
# ======================================================================

print("\n" + "=" * 80)
print("ARCHIVOS FINALES")
print("=" * 80)


print(
    "Predicciones:"
)

print(
    RUTA_PRED
)

print(
    "SHA256:",
    sha256_file(
        RUTA_PRED
    )
)


print(
    "\nMétricas:"
)

print(
    RUTA_METRICAS
)

print(
    "SHA256:",
    sha256_file(
        RUTA_METRICAS
    )
)


print(
    "\nManifiesto:"
)

print(
    RUTA_RUN_MANIFEST
)

print(
    "SHA256:",
    sha256_file(
        RUTA_RUN_MANIFEST
    )
)


# ======================================================================
# 18. CIERRE
# ======================================================================

print("\n" + "=" * 80)
print("VALIDACIÓN EXTERNA VALPARAÍSO 2017 — EJECUCIÓN COMPLETADA")
print("=" * 80)

print(
    f"R²   = {R2:.10f}"
)

print(
    f"RMSE = {RMSE:.10f}"
)

print(
    f"MAE  = {MAE:.10f}"
)

print(
    f"BIAS = {BIAS:+.10f}"
)

print(
    f"exp(BIAS) = {EXP_BIAS:.10f}"
)

print(
    "\n✅ V5.1 fue ejecutado sobre las "
    "42 estaciones congeladas."
)

print(
    "✅ Ninguna estación fue retirada "
    "después de observar las predicciones."
)

print(
    "✅ Los resultados quedan congelados "
    "para interpretación."
)